# AttackAware PolyIoM v1.1.4: the whole experiment, run freshly on Kaggle

This notebook reruns the study from the public datasets: it downloads LFW,
LibriSpeech and VCTK, extracts face and voice embeddings, splits the people,
searches for the polynomial key, selects the operating point, evaluates, runs
the ablation, the attack and the revocation test, draws every figure, and
compares every result with the paper.

**Before you start (Kaggle settings, right-hand panel)**
1. **Internet: On.** This needs a phone-verified Kaggle account.
2. **Accelerator: GPU T4 x2 or P100.** Used only for the key search.
3. **Add Input → Datasets → `chinafax/cfpw-dataset`.** This is the CFP name
   list the study used to exclude overlapping people.

**How to run.** Click **Save Version → Save & Run All (Commit)**. One Kaggle
session lasts at most 12 hours, and the whole experiment needs about three:

| Session | What it does |
|---|---|
| 1 | downloads, face and voice embeddings, the split, the tensors; starts the key search |
| 2 | finishes the key search |
| 3 | selection, evaluation, the external corpus, the analyses, the figures and the report |

**To continue after a session:** open the notebook, choose **Add Input →
Your Work → this notebook's output** (the latest version), and Save & Run All
again. The first cell restores the earlier work, and every finished stage is
skipped. When nothing is left to do, `REPLICATION_REPORT.md`, the figures and
a zip appear in the output.

**What "reproduce" means here.** The notebook runs the study's own code
(ported from its Colab notebook): the same downloads, face and voice models,
seeds, split, key search, tensors and selection rule. After every stage it
compares its output with the study's record and prints **identical** or
**DIFFERENT**. If every stage is identical, the paper is reproduced exactly.
If a stage differs (for example, a newer library moves a float), the run
continues, and the final report sets each result beside the paper's with
the fresh 95% interval.


In [ ]:
#@title 1. Settings, earlier work and checklist
import hashlib, importlib.util, json, math, os, shutil, sys
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic GPU maths
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path(os.environ.get("POLYIOM_PROJECT", "/kaggle/working/AttackAware_PolyIoM_fresh"))
INPUT_ROOT = Path(os.environ.get("POLYIOM_INPUT", "/kaggle/input"))
RAW = Path(os.environ.get("POLYIOM_RAW", "/kaggle/tmp/attackaware_raw"))
RUNTIME = Path(os.environ.get("POLYIOM_RUNTIME", "/kaggle/tmp/polyiom_runtime"))

# Earlier sessions: copy their saved work back before anything is written.
MARKER = ".attackaware_fresh"
if not (PROJECT / MARKER).is_file() and INPUT_ROOT.exists():
    for found in sorted(INPUT_ROOT.glob(f"**/{MARKER}")):
        shutil.copytree(found.parent, PROJECT, dirs_exist_ok=True)
        print(f"Restored earlier work from {found.parent}")
        break
PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / MARKER).write_text("AttackAware PolyIoM fresh Kaggle run\n")

DIR = {"protocol": PROJECT / "protocol", "embeddings": PROJECT / "embeddings",
       "runs": PROJECT / "runs", "seal": PROJECT / "seal",
       "datasets": RAW / "datasets", "preprocessed": PROJECT / "preprocessed"}
for path in list(DIR.values()) + [RAW, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)
RUNS_IN = DIR["runs"]
S0, G, PROTOCOL_VERSION = 2026, 5, "1.1.1"
DEVICE = torch.device("cpu")                 # hashing on CPU, as the study
DIMENSION = {"face": 512, "voice": 192}
N_BOOTSTRAP, CONFIDENCE = 2000, 0.95
LFW_FACE_TRIALS = DIR["protocol"] / "lfw_face_trials.tsv"
LFW_EMB = DIR["embeddings"] / "lfw_all_valid_embeddings.npz"
LIBRI_MANIFEST = DIR["protocol"] / "librispeech_internal.tsv"
LIBRI_EMB = DIR["embeddings"] / "librispeech_internal_embeddings.npz"
HELDOUT_RESULTS = {m: RUNS_IN / "heldout" / m / "heldout_result.json"
                   for m in ("face", "voice")}
ABLATION_OUTPUT = RUNS_IN / "ablation" / "ablation_result.json"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

free = shutil.disk_usage(RAW).free / 1e9
print(f"GPU      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none - the key search will be slow'}")
print(f"disk     {free:.0f} GB free for downloads (about 25 GB needed)")
print(f"project  {PROJECT}")


In [ ]:
%%writefile {RUNTIME}/ablation_only.py
# 2. The study's pipeline primitives and the ablation
"""Self-contained ablation runtime for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_ABLATION.ipynb after its preflight cell
has defined PROJECT, DIR, S0, G, DEVICE, the embedding paths, HELDOUT_RESULTS
and ABLATION_OUTPUT.

Rebuilds the frozen score histograms from the stored embeddings and keys, so
it depends on no other notebook. Writes one file, runs/ablation/
ablation_result.json, and no seal.
"""

def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str))
    os.replace(temporary, path)


def H(*fields):
    payload = "|".join(str(field) for field in fields).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") & ((1 << 63) - 1)


def seed_unlink(modality, key_index):
    return H(S0, "UNLINK", modality, key_index)


def l2_np(vector, eps=1e-12):
    vector = np.asarray(vector, dtype=np.float32)
    norm = np.linalg.norm(vector)
    if norm <= eps:
        raise RuntimeError("Zero-norm embedding")
    return (vector / norm).astype(np.float32)


def selected_poly_key(modality):
    path = DIR["runs"] / "key_search" / modality / "selected_key.json"
    obj = json.loads(path.read_text())
    return np.asarray(obj["C"], np.float32), np.asarray(obj["E"], np.int64)


def face_heldout_data():
    trials = pd.read_csv(LFW_FACE_TRIALS, sep="\t")
    stored = np.load(LFW_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    held = trials[trials.partition == "evaluation"]
    subjects, enroll, probes = [], {}, {}
    for uid, group in held.groupby("subject_id"):
        erow = group[group.role == "enroll"]
        prows = group[group.role == "probe"]
        if len(erow) != 1 or len(prows) < 1:
            continue
        uid = str(uid)
        subjects.append(uid)
        enroll[uid] = embeddings[int(erow.iloc[0].embedding_index)]
        probes[uid] = [
            embeddings[int(row.embedding_index)] for row in prows.itertuples()
        ]
    return sorted(subjects), enroll, probes


def voice_heldout_data():
    manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
    stored = np.load(LIBRI_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": stored["speaker_id"].astype(str),
        "rank": stored["rank"].astype(int),
        "embedding_index": np.arange(len(embeddings), dtype=np.int64),
    })
    held = manifest.merge(
        table[["speaker_id", "rank", "embedding_index"]],
        on=["speaker_id", "rank"], how="inner",
    )
    held = held[held.partition == "evaluation"]
    subjects, enroll, probes = [], {}, {}
    for uid, group in held.groupby("speaker_id"):
        group = group.sort_values("rank")
        if len(group) != 15:
            continue
        uid = str(uid)
        members = np.stack([
            embeddings[int(row.embedding_index)]
            for row in group[group["rank"] < 5].itertuples()
        ])
        subjects.append(uid)
        enroll[uid] = l2_np(members.mean(axis=0))
        probes[uid] = [
            embeddings[int(row.embedding_index)]
            for row in group[group["rank"] >= 5].itertuples()
        ]
    return sorted(subjects), enroll, probes


def poly_indices(d, overlap, device):
    stride = G - overlap
    k = 1 + math.ceil((d - G) / stride)
    idx = torch.arange(k, device=device)[:, None] * stride + torch.arange(G, device=device)
    mask = idx < d
    return idx.clamp(max=d - 1).long(), mask


def poly_transform(vectors, coefficients, exponents, overlap):
    idx, mask = poly_indices(vectors.shape[-1], overlap, vectors.device)
    values = vectors[..., idx] * mask.to(vectors.dtype)
    coefficients = torch.as_tensor(coefficients, device=vectors.device, dtype=vectors.dtype)
    exponents = torch.as_tensor(exponents, device=vectors.device, dtype=torch.int64)
    return (torch.pow(values, exponents) * coefficients).sum(dim=-1)


def load_iom_tensor(modality, overlap, d):
    k = 1 + math.ceil((d - G) / (G - overlap))
    path = DIR["runs"] / "iom_tensors" / f"{modality}_o{overlap}_k{k}.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Frozen IoM tensor is missing: {path}")
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        tensor = torch.load(path, map_location="cpu")
    if (tensor.ndim != 3 or tensor.shape[0] < 256 or tensor.shape[1] < 32
            or tensor.shape[-1] != k):
        raise RuntimeError(f"Frozen IoM tensor has an invalid shape: {path}")
    return tensor


print("Frozen-pipeline primitives ready.")


ARMS = ("polyiom", "iom_only", "randproj_iom")
ARM_NOTE = {
    "polyiom": "z -> P_K*(z; o*) -> IoM-GRP   (the proposed method)",
    "iom_only": "z ----------------> IoM-GRP   (the practical baseline)",
    "randproj_iom": "z -> A z ----------> IoM-GRP   (isolates the polynomial)",
}


def abl_seed(modality, label):
    """Deterministic seed, distinct from every seed stream in the study."""
    return H(S0, "ABLATION", modality, label)


def abl_randn(shape, modality, label):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(abl_seed(modality, label))
    return torch.randn(shape, generator=generator, dtype=torch.float32)


def hash_arm(vectors, arm, key, overlap, tensor, M, q, A, batch_size=64):
    """One arm's protected codes. Only the pre-projection step differs."""
    coefficients, exponents = key
    tensor = tensor[:M, :q].to(DEVICE, torch.float32)
    outputs = []
    for start in range(0, len(vectors), batch_size):
        batch = torch.as_tensor(
            np.stack(vectors[start:start + batch_size]),
            dtype=torch.float32, device=DEVICE,
        )
        if arm == "polyiom":
            projected = poly_transform(batch, coefficients, exponents, overlap)
        elif arm == "randproj_iom":
            projected = batch @ A
        elif arm == "iom_only":
            projected = batch
        else:
            raise ValueError(arm)
        scores = torch.einsum("bk,mqk->bmq", projected, tensor)
        outputs.append(torch.argmax(scores, dim=-1).to(torch.int16).cpu().numpy())
    return np.concatenate(outputs, axis=0)


def check_gaussian(tensor, label, tolerance=0.05):
    """The generated tensors are standard normal. If the frozen recognition
    tensor is not, the arms are not drawn from the same family and the
    comparison is confounded, so say so loudly rather than quietly."""
    mean, std = float(tensor.mean()), float(tensor.std())
    ok = abs(mean) <= tolerance and abs(std - 1.0) <= tolerance
    print(f"    {label}: mean={mean:+.4f} std={std:.4f} "
          f"{'~N(0,1)' if ok else '*** NOT ~N(0,1) - comparison confounded'}")
    return ok


def build_arm_histograms(modality):
    """Per-identity histograms for all three arms, sharing identity order."""
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    threshold = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    if len(subjects) != sealed["n_subjects"]:
        raise RuntimeError(
            f"{modality}: reconstructed {len(subjects)} identities; "
            f"sealed result has {sealed['n_subjects']}"
        )

    key = selected_poly_key(modality)
    n = len(subjects)
    d = len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n}  d={d}  k={k}  M={M} q={q} o={overlap}")

    frozen = load_iom_tensor(modality, overlap, d)
    print("  recognition tensors:")
    gaussian_ok = check_gaussian(frozen[:M, :q], "polyiom (frozen, from disk)")
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only"),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj"),
    }
    A = abl_randn((d, k), modality, "projmap") / math.sqrt(d)

    unlink = {"polyiom": [], "iom_only": [], "randproj_iom": []}
    for key_index in (1, 2):
        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed_unlink(modality, key_index))
        unlink["polyiom"].append(
            torch.randn((M, q, k), generator=generator, dtype=torch.float32)
        )
        unlink["iom_only"].append(
            abl_randn((M, q, d), modality, f"unlink|raw|{key_index}")
        )
        unlink["randproj_iom"].append(
            abl_randn((M, q, k), modality, f"unlink|prj|{key_index}")
        )

    enrolment_vectors = [enroll[uid] for uid in subjects]
    probe_vectors, probe_owners = [], []
    for owner, uid in enumerate(subjects):
        probe_vectors.extend(probes[uid])
        probe_owners.extend([owner] * len(probes[uid]))

    bundles = {}
    for arm in ARMS:
        enrolment_hashes = hash_arm(
            enrolment_vectors, arm, key, overlap, recognition[arm], M, q, A)
        probe_hashes = hash_arm(
            probe_vectors, arm, key, overlap, recognition[arm], M, q, A)
        collision = (probe_hashes[:, None, :]
                     == enrolment_hashes[None, :, :]).sum(axis=-1)

        genuine_by_subject = np.zeros((n, M + 1), dtype=np.float64)
        impostor_by_pair = np.zeros((n, n, M + 1), dtype=np.float64)
        for probe_index, owner in enumerate(probe_owners):
            genuine_by_subject[owner, collision[probe_index, owner]] += 1
            for claimant in range(n):
                if claimant != owner:
                    impostor_by_pair[owner, claimant,
                                     collision[probe_index, claimant]] += 1

        first = hash_arm(enrolment_vectors, arm, key, overlap,
                         unlink[arm][0], M, q, A)
        second = hash_arm(enrolment_vectors, arm, key, overlap,
                          unlink[arm][1], M, q, A)
        unlink_collision = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
        mated_by_subject = np.zeros((n, M + 1), dtype=np.float64)
        nonmated_by_pair = np.zeros((n, n, M + 1), dtype=np.float64)
        for left in range(n):
            mated_by_subject[left, unlink_collision[left, left]] = 1
            for right in range(n):
                if left != right:
                    nonmated_by_pair[left, right,
                                     unlink_collision[left, right]] = 1

        bundles[arm] = {
            "genuine_by_subject": genuine_by_subject,
            "impostor_by_pair": impostor_by_pair,
            "mated_by_subject": mated_by_subject,
            "nonmated_by_pair": nonmated_by_pair,
        }
        print(f"    {arm:<13} built   {ARM_NOTE[arm]}")

    return {
        "modality": modality, "sealed": sealed, "subjects": subjects,
        "M": M, "q": q, "o": overlap, "d": d, "k": k,
        "threshold": threshold, "arms": bundles,
        "frozen_tensor_is_gaussian": bool(gaussian_ok),
    }


print("Arm construction ready.")


def eer_from_histograms(genuine, impostor):
    genuine_total, impostor_total = genuine.sum(), impostor.sum()
    if genuine_total <= 0 or impostor_total <= 0:
        raise RuntimeError("EER requires non-empty distributions")
    far = np.r_[np.cumsum(impostor[::-1])[::-1], 0.0] / impostor_total
    frr = np.r_[0.0, np.cumsum(genuine)] / genuine_total
    difference = far - frr
    c_eer = int(np.flatnonzero(np.abs(difference) == np.abs(difference).min())[0])
    exact = np.flatnonzero(difference == 0.0)
    if len(exact):
        return float(far[exact[0]]), c_eer
    crossings = np.flatnonzero((difference[:-1] > 0.0) & (difference[1:] < 0.0))
    if not len(crossings):
        raise AssertionError("EER crossing not found")
    index = int(crossings[0])
    weight = difference[index] / (difference[index] - difference[index + 1])
    return float(far[index] + weight * (far[index + 1] - far[index])), c_eer


def dsys_from_histograms(mated, nonmated):
    if mated.sum() <= 0 or nonmated.sum() <= 0:
        raise RuntimeError("Dsys requires non-empty distributions")
    pm, pn = mated / mated.sum(), nonmated / nonmated.sum()
    local = np.zeros_like(pm)
    for score in range(len(pm)):
        if pm[score] == 0:
            local[score] = 0
        elif pn[score] == 0:
            local[score] = 1
        else:
            ratio = pm[score] / pn[score]
            local[score] = 0 if ratio <= 1 else 2 * ratio / (1 + ratio) - 1
    return float(np.sum(pm * local))


def collapse(bundle, identity_weights, n):
    pair_weights = np.outer(identity_weights, identity_weights)
    np.fill_diagonal(pair_weights, 0.0)
    flat = pair_weights.reshape(-1)
    return (
        identity_weights @ bundle["genuine_by_subject"],
        flat @ bundle["impostor_by_pair"].reshape(n * n, -1),
        identity_weights @ bundle["mated_by_subject"],
        flat @ bundle["nonmated_by_pair"].reshape(n * n, -1),
    )


def arm_metrics(bundle, identity_weights, n):
    genuine, impostor, mated, nonmated = collapse(bundle, identity_weights, n)
    eer, c_eer = eer_from_histograms(genuine, impostor)
    return {"EER": eer, "c_EER": c_eer,
            "Dsys": dsys_from_histograms(mated, nonmated)}, genuine, impostor


def tar_at_matched_fmr(genuine, impostor, target):
    """Lowest threshold whose FMR still meets the target; descriptive only."""
    total = impostor.sum()
    best = None
    for c in range(len(impostor) - 1, -1, -1):
        if float(impostor[c:].sum() / total) <= target + 1e-12:
            best = c
        else:
            break
    if best is None:
        return float("nan"), -1
    return float(genuine[best:].sum() / genuine.sum()), best


def verify_against_seal(record):
    """The polyiom arm rebuilds the sealed pipeline exactly, so its EER must
    reproduce the sealed held-out figure. Anything else means this notebook
    has drifted from the study and its numbers must not be reported."""
    sealed = record["sealed"]
    reproduced = record["point"]["polyiom"]["EER"]
    if not np.isclose(reproduced, sealed["EER_HOLDOUT"], rtol=0.0, atol=1e-12):
        raise RuntimeError(
            f"{record['modality']}: the polyiom arm reproduces "
            f"EER={reproduced}, but the sealed held-out result is "
            f"{sealed['EER_HOLDOUT']}. Refusing to report - the arm is not "
            f"the study's pipeline."
        )
    print(f"  self-check: polyiom EER {reproduced * 100:.4f}% reproduces the "
          f"sealed held-out result")


def paired_bootstrap(record, n_bootstrap, confidence, seed):
    """One identity weight vector per replicate, applied to all three arms,
    so an interval on a DIFFERENCE accounts for the arms sharing identities.
    An unpaired interval would be too wide."""
    n = len(record["subjects"])
    rng = np.random.default_rng(seed)
    parts, remaining = [], n_bootstrap
    while remaining:
        candidates = rng.multinomial(n, np.full(n, 1 / n), size=remaining)
        candidates = candidates[(candidates > 0).sum(axis=1) >= 2]
        parts.append(candidates[:remaining])
        remaining -= len(parts[-1])
    weights = np.concatenate(parts, axis=0).astype(np.float64)

    per_arm = {arm: {"EER": [], "Dsys": []} for arm in ARMS}
    contrasts = {arm: {"delta_EER": [], "delta_Dsys": []}
                 for arm in ARMS if arm != "polyiom"}
    for replicate in range(n_bootstrap):
        w = weights[replicate]
        values = {}
        for arm in ARMS:
            measured, _, _ = arm_metrics(record["arms"][arm], w, n)
            values[arm] = measured
            per_arm[arm]["EER"].append(measured["EER"])
            per_arm[arm]["Dsys"].append(measured["Dsys"])
        for arm in contrasts:
            contrasts[arm]["delta_EER"].append(
                values[arm]["EER"] - values["polyiom"]["EER"])
            contrasts[arm]["delta_Dsys"].append(
                values[arm]["Dsys"] - values["polyiom"]["Dsys"])
        if (replicate + 1) % 500 == 0:
            print(f"    {record['modality']}: {replicate + 1}/{n_bootstrap}")

    alpha = 1.0 - confidence

    def interval(values):
        lower, upper = np.quantile(values, [alpha / 2, 1 - alpha / 2])
        return [float(lower), float(upper)]

    return per_arm, contrasts, interval


def run_ablation(n_bootstrap=2000, confidence=0.95):
    if ABLATION_OUTPUT.is_file():
        stored = json.loads(ABLATION_OUTPUT.read_text())
        print("Resume: stored ablation result found; returning it unchanged.")
        return stored

    source_hashes = {m: sha256_file(p) for m, p in HELDOUT_RESULTS.items()}
    result = {
        "schema": 1,
        "analysis": "hardening_ablation",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap),
        "confidence": float(confidence),
        "master_seed": S0,
        "changes_any_seal": False,
        "threshold_policy": "none fitted; EER and Dsys are threshold-free",
        "arms": {arm: ARM_NOTE[arm] for arm in ARMS},
        "source_heldout_sha256": source_hashes,
        "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = build_arm_histograms(modality)
        n = len(record["subjects"])
        flat = np.ones(n, dtype=np.float64)

        record["point"] = {}
        matched = {}
        for arm in ARMS:
            measured, genuine, impostor = arm_metrics(record["arms"][arm], flat, n)
            record["point"][arm] = measured
            matched[arm] = (genuine, impostor)
        verify_against_seal(record)

        reference_fmr = float(
            matched["polyiom"][1][record["threshold"]:].sum()
            / matched["polyiom"][1].sum()
        )
        for arm in ARMS:
            tar, c = tar_at_matched_fmr(*matched[arm], reference_fmr)
            record["point"][arm]["TAR_at_matched_FMR_descriptive"] = tar
            record["point"][arm]["c_matched"] = int(c)

        print(f"  paired bootstrap over {n} identities")
        per_arm, contrasts, interval = paired_bootstrap(
            record, n_bootstrap, confidence,
            seed=abl_seed(modality, "bootstrap"),
        )

        result["modalities"][modality] = {
            "n_subjects": n, "M": record["M"], "q": record["q"],
            "o": record["o"], "d": record["d"], "k": record["k"],
            "c_tau_carried_from_DEV": record["threshold"],
            "frozen_tensor_is_gaussian": record["frozen_tensor_is_gaussian"],
            "polyiom_reproduces_sealed_heldout": True,
            "reference_FMR_for_matched_TAR": reference_fmr,
            "arms": {arm: {
                "EER": record["point"][arm]["EER"],
                "EER_CI": interval(per_arm[arm]["EER"]),
                "Dsys": record["point"][arm]["Dsys"],
                "Dsys_CI": interval(per_arm[arm]["Dsys"]),
                "TAR_at_matched_FMR_descriptive":
                    record["point"][arm]["TAR_at_matched_FMR_descriptive"],
            } for arm in ARMS},
            "contrasts_vs_polyiom": {arm: {
                "delta_EER": (record["point"][arm]["EER"]
                              - record["point"]["polyiom"]["EER"]),
                "delta_EER_CI": interval(contrasts[arm]["delta_EER"]),
                "delta_Dsys": (record["point"][arm]["Dsys"]
                               - record["point"]["polyiom"]["Dsys"]),
                "delta_Dsys_CI": interval(contrasts[arm]["delta_Dsys"]),
            } for arm in contrasts},
        }

    if {m: sha256_file(p) for m, p in HELDOUT_RESULTS.items()} != source_hashes:
        raise RuntimeError("A sealed held-out result changed during the ablation")
    atomic_json(ABLATION_OUTPUT, result)
    print("\nWrote", ABLATION_OUTPUT)
    return result


def ablation_table(result):
    rows = []
    for modality, data in result["modalities"].items():
        for arm, values in data["arms"].items():
            rows.append({
                "modality": modality, "arm": arm,
                "EER": values["EER"],
                "EER_lo": values["EER_CI"][0], "EER_hi": values["EER_CI"][1],
                "Dsys": values["Dsys"],
                "Dsys_lo": values["Dsys_CI"][0], "Dsys_hi": values["Dsys_CI"][1],
                "TAR_matched_FMR": values["TAR_at_matched_FMR_descriptive"],
            })
    return pd.DataFrame(rows)


def contrast_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  d={data['d']} -> k={data['k']}  "
              f"n={data['n_subjects']} identities")
        print("  arm minus proposed; an interval excluding 0 is firm")
        for arm, c in data["contrasts_vs_polyiom"].items():
            for name, scale, unit in (("EER", 100, " %"), ("Dsys", 1, "")):
                value = c[f"delta_{name}"]
                lower, upper = c[f"delta_{name}_CI"]
                firm = "FIRM" if (lower > 0 or upper < 0) else "not distinguishable"
                print(f"    {arm:<14} d{name:<5}{value * scale:+8.3f}{unit}"
                      f"  [{lower * scale:+7.3f},{upper * scale:+7.3f}]  {firm}")
    print("\n  A POSITIVE delta means that arm is WORSE than the proposed")
    print("  method (higher EER, higher Dsys). randproj_iom is the arm that")
    print("  isolates the polynomial: it matches the proposed method's")
    print("  projection dimension k, so a null contrast there means the gain")
    print("  came from the dimension change and not from P_K*.")


print("Metrics and paired bootstrap ready.")


In [ ]:
%%writefile {RUNTIME}/scoredump_only.py
# 3. The study's score histograms
"""Score-histogram dump for AttackAware PolyIoM v1.1.4.

Loaded after ablation_only.py, which supplies the frozen-pipeline
primitives. Read-only: writes one file and no seal.

Why histograms rather than raw score vectors
--------------------------------------------
A collision score is an integer in 0..M. A count per score value is
therefore a LOSSLESS record of the score distribution, and every results
figure a biometric paper needs can be derived from it exactly:

  * FMR and FNMR at every threshold, hence the full DET curve
  * EER, and the threshold at which it occurs
  * genuine and impostor distributions, with tau* marked
  * mated and non-mated distributions, which is what D_sys summarises

The file is a few kilobytes rather than a few megabytes, and nothing is
approximated. The evaluation already builds these histograms internally
and discards them; this cell keeps them.

Covers two systems, because the first criterion a cancelable-biometric
paper has to address is performance PRESERVATION, which is reported as the
protected system against the UNPROTECTED one:

  protected     the sealed pipeline, integer collision scores in 0..M
  unprotected   plain cosine similarity between the same embeddings

Cosine is continuous, so the unprotected side is binned finely over
[-1, 1] and the bin edges are stored with it. At 2000 bins the resolution
is 0.001, which is far finer than any DET curve needs.

Also stores the LOCAL unlinkability curve D_link(s) alongside the global
value. The global value is what the evaluation already reports; the local
curve is the part that is plottable, and it is the standard unlinkability
figure in this literature.

Note that the mated distribution serves two figures at once. Under the
unlinkability framework it is "same subject, two different keys"; under
revocability it is the pseudo-impostor distribution. They are the same
measurement, and a paper should not present them as independent evidence.

Partitions: development and evaluation for both modalities, plus external
VCTK for voice when its embeddings are present.
"""

COS_BINS = 2000

SCORE_OUTPUT = None          # set in run_score_dump


def partition_data(modality, partition):
    """Enrolment and probe embeddings for one partition of one modality.

    Same construction as the held-out loaders in ablation_only.py, with the
    partition as a parameter so development can be read the same way.
    """
    if modality == "face":
        trials = pd.read_csv(LFW_FACE_TRIALS, sep="\t")
        stored = np.load(LFW_EMB, allow_pickle=True)
        embeddings = stored["embedding"].astype(np.float32)
        rows = trials[trials.partition == partition]
        subjects, enroll, probes = [], {}, {}
        for uid, group in rows.groupby("subject_id"):
            erow = group[group.role == "enroll"]
            prows = group[group.role == "probe"]
            if len(erow) != 1 or len(prows) < 1:
                continue
            uid = str(uid)
            subjects.append(uid)
            enroll[uid] = embeddings[int(erow.iloc[0].embedding_index)]
            probes[uid] = [embeddings[int(r.embedding_index)]
                           for r in prows.itertuples()]
        return sorted(subjects), enroll, probes

    manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
    stored = np.load(LIBRI_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": stored["speaker_id"].astype(str),
        "rank": stored["rank"].astype(int),
        "embedding_index": np.arange(len(embeddings), dtype=np.int64),
    })
    rows = manifest.merge(table, on=["speaker_id", "rank"], how="inner")
    rows = rows[rows.partition == partition]
    subjects, enroll, probes = [], {}, {}
    for uid, group in rows.groupby("speaker_id"):
        group = group.sort_values("rank")
        if len(group) != 15:
            continue
        uid = str(uid)
        members = np.stack([embeddings[int(r.embedding_index)]
                            for r in group[group["rank"] < 5].itertuples()])
        subjects.append(uid)
        enroll[uid] = l2_np(members.mean(axis=0))
        probes[uid] = [embeddings[int(r.embedding_index)]
                       for r in group[group["rank"] >= 5].itertuples()]
    return sorted(subjects), enroll, probes


def histograms_for(modality, subjects, enroll, probes, M, q, overlap, key,
                   recognition, unlink_pair):
    """Flat genuine / impostor / mated / non-mated histograms over 0..M."""
    enrolment = [enroll[uid] for uid in subjects]
    enrolled = hash_arm(enrolment, "polyiom", key, overlap, recognition,
                        M, q, None)
    probe_vectors, owners = [], []
    for index, uid in enumerate(subjects):
        probe_vectors.extend(probes[uid])
        owners.extend([index] * len(probes[uid]))
    hashed = hash_arm(probe_vectors, "polyiom", key, overlap, recognition,
                      M, q, None)
    collision = (hashed[:, None, :] == enrolled[None, :, :]).sum(axis=-1)

    genuine = np.zeros(M + 1, dtype=np.int64)
    impostor = np.zeros(M + 1, dtype=np.int64)
    for row, owner in enumerate(owners):
        for claimant in range(len(subjects)):
            c = int(collision[row, claimant])
            if claimant == owner:
                genuine[c] += 1
            else:
                impostor[c] += 1

    first = hash_arm(enrolment, "polyiom", key, overlap, unlink_pair[0],
                     M, q, None)
    second = hash_arm(enrolment, "polyiom", key, overlap, unlink_pair[1],
                      M, q, None)
    cross = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
    mated = np.zeros(M + 1, dtype=np.int64)
    nonmated = np.zeros(M + 1, dtype=np.int64)
    for a in range(len(subjects)):
        mated[int(cross[a, a])] += 1
        for b in range(len(subjects)):
            if a != b:
                nonmated[int(cross[a, b])] += 1

    return {"genuine": genuine.tolist(), "impostor": impostor.tolist(),
            "mated": mated.tolist(), "nonmated": nonmated.tolist(),
            "n_subjects": len(subjects),
            "n_genuine": int(genuine.sum()),
            "n_impostor": int(impostor.sum())}


def cosine_histograms(subjects, enroll, probes, nbins=COS_BINS):
    """Unprotected baseline: plain cosine between the same embeddings.

    This is the reference the protected system has to be compared against.
    Binned over [-1, 1] rather than stored raw: 2000 bins is 0.001
    resolution and a few kilobytes instead of megabytes.
    """
    def unit(a):
        a = np.asarray(a, dtype=np.float64)
        return a / max(np.linalg.norm(a), 1e-12)

    E = np.stack([unit(enroll[uid]) for uid in subjects])
    rows, owners = [], []
    for index, uid in enumerate(subjects):
        for v in probes[uid]:
            rows.append(unit(v))
            owners.append(index)
    P = np.stack(rows)
    scores = P @ E.T                                  # (probes, subjects)

    edges = np.linspace(-1.0, 1.0, nbins + 1)
    own = np.asarray(owners)
    mask = np.zeros_like(scores, dtype=bool)
    mask[np.arange(len(own)), own] = True
    gen, _ = np.histogram(scores[mask], bins=edges)
    imp, _ = np.histogram(scores[~mask], bins=edges)
    return {"genuine": gen.tolist(), "impostor": imp.tolist(),
            "bin_edges": [float(edges[0]), float(edges[-1])],
            "n_bins": int(nbins),
            "n_genuine": int(gen.sum()), "n_impostor": int(imp.sum())}


def unlinkability_curve(mated, nonmated):
    """Local D_link(s) and the global value, from the same histograms.

    The global value is what the evaluation already reports. The local
    curve is what a figure needs.
    """
    m = np.asarray(mated, float)
    n = np.asarray(nonmated, float)
    if m.sum() <= 0 or n.sum() <= 0:
        return [], float("nan")
    pm, pn = m / m.sum(), n / n.sum()
    local = np.zeros_like(pm)
    for s_ in range(len(pm)):
        if pm[s_] == 0:
            local[s_] = 0.0
        elif pn[s_] == 0:
            local[s_] = 1.0
        else:
            ratio = pm[s_] / pn[s_]
            local[s_] = 0.0 if ratio <= 1 else 2 * ratio / (1 + ratio) - 1
    return local.tolist(), float(np.sum(pm * local))


def eer_cosine(hist_gen, hist_imp):
    """EER from a binned continuous score, for the unprotected baseline.

    The exact-zero branch is not optional here. When the two distributions
    are well separated there is a run of thresholds at which FAR and FRR
    are both zero, so FAR - FRR never changes sign between adjacent bins
    and a crossing-only search returns nothing. The unprotected baseline
    is the case most likely to be well separated, which is exactly the
    number this function exists to produce.
    """
    g = np.asarray(hist_gen, float)
    i = np.asarray(hist_imp, float)
    if g.sum() <= 0 or i.sum() <= 0:
        return float("nan")
    far = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / i.sum()
    frr = np.r_[0.0, np.cumsum(g)] / g.sum()
    d = far - frr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(far[exact[0]])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    if len(cross):
        j = int(cross[0])
        w = d[j] / (d[j] - d[j + 1])
        return float(far[j] + w * (far[j + 1] - far[j]))
    j = int(np.argmin(np.maximum(far, frr)))      # never crosses
    return float((far[j] + frr[j]) / 2.0)


def eer_check(genuine, impostor, M):
    """Recompute EER from the histogram, to prove the dump is faithful."""
    g = np.asarray(genuine, float)
    i = np.asarray(impostor, float)
    far = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / i.sum()
    frr = np.r_[0.0, np.cumsum(g)] / g.sum()
    d = far - frr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(far[exact[0]])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    j = int(cross[0])
    w = d[j] / (d[j] - d[j + 1])
    return float(far[j] + w * (far[j + 1] - far[j]))


def run_score_dump():
    output = DIR["runs"] / "scores" / "score_histograms.json"
    if output.is_file():
        print("Resume: stored score histograms found; returning unchanged.")
        return json.loads(output.read_text())

    result = {"schema": 1, "analysis": "score_histograms",
              "systems": {
                  "protected": "polyiom, the sealed pipeline; integer "
                               "collision scores in 0..M",
                  "unprotected": "plain cosine between the same embeddings; "
                                 "binned over [-1,1], the performance-"
                                 "preservation reference"},
              "mated_note": "the mated distribution is both the "
                            "unlinkability mated set and the revocability "
                            "pseudo-impostor set; one measurement, not two",
              "lossless": "scores are integers in 0..M, so a count per "
                          "score value loses nothing",
              "master_seed": S0, "changes_any_seal": False, "modalities": {}}

    for modality in ("voice", "face"):
        sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
        M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
        tau = int(sealed["c_tau_carried_from_DEV"])
        key = selected_poly_key(modality)
        print(f"\n{modality.upper()}  M={M} q={q} o={overlap} tau={tau}")

        entry = {"M": M, "q": q, "o": overlap, "tau": tau,
                 "partitions": {}}
        for partition in ("development", "evaluation"):
            subjects, enroll, probes = partition_data(modality, partition)
            if len(subjects) < 2:
                print(f"  {partition}: too few identities, skipped")
                continue
            d = len(enroll[subjects[0]])
            k = 1 + math.ceil((d - G) / (G - overlap))
            recognition = load_iom_tensor(modality, overlap, d)[:M, :q] \
                .to(DEVICE, torch.float32)
            pair = []
            for index in (1, 2):
                gen = torch.Generator(device="cpu")
                gen.manual_seed(seed_unlink(modality, index))
                pair.append(torch.randn((M, q, k), generator=gen,
                                        dtype=torch.float32).to(DEVICE))
            h = histograms_for(modality, subjects, enroll, probes, M, q,
                               overlap, key, recognition, pair)
            h["dlink_curve"], h["dlink_global"] = unlinkability_curve(
                h["mated"], h["nonmated"])
            h["unprotected"] = cosine_histograms(subjects, enroll, probes)
            entry["partitions"][partition] = h
            recomputed = eer_check(h["genuine"], h["impostor"], M)
            raw = eer_cosine(h["unprotected"]["genuine"],
                             h["unprotected"]["impostor"])
            h["unprotected"]["EER"] = raw
            note = ""
            if partition == "evaluation":
                delta = abs(recomputed - sealed["EER_HOLDOUT"])
                note = (f"  reproduces sealed held-out EER "
                        f"({sealed['EER_HOLDOUT'] * 100:.4f}%)"
                        if delta <= 1e-9 else
                        f"  *** MISMATCH vs sealed "
                        f"{sealed['EER_HOLDOUT'] * 100:.4f}%")
            print(f"  {partition:<12} {h['n_subjects']} identities, "
                  f"{h['n_genuine']} genuine / {h['n_impostor']} impostor")
            print(f"    protected   EER {recomputed * 100:7.4f}%{note}")
            print(f"    unprotected EER {raw * 100:7.4f}%   "
                  f"-> protection costs "
                  f"{(recomputed - raw) * 100:+.4f} pp")
            print(f"    D_link global {h['dlink_global']:.6f}")
        result["modalities"][modality] = entry

    # External voice, if its embeddings are present
    ext = PROJECT / "external_voice_only.py"
    vctk = DIR["embeddings"] / "vctk_external_embeddings.npz"
    if ext.is_file() and vctk.is_file():
        try:
            scope = {"PROJECT": PROJECT, "DIR": DIR, "np": np, "pd": pd,
                     "torch": torch, "json": json, "math": math,
                     "hashlib": hashlib, "os": os, "Path": Path}
            exec(compile(ext.read_text(), str(ext), "exec"), scope)
            subjects, enroll, probes = scope["external_voice_data"]()
            sealed = json.loads(HELDOUT_RESULTS["voice"].read_text())
            M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
            key = selected_poly_key("voice")
            d = len(enroll[subjects[0]])
            k = 1 + math.ceil((d - G) / (G - overlap))
            recognition = load_iom_tensor("voice", overlap, d)[:M, :q] \
                .to(DEVICE, torch.float32)
            pair = []
            for index in (1, 2):
                gen = torch.Generator(device="cpu")
                gen.manual_seed(seed_unlink("voice", index))
                pair.append(torch.randn((M, q, k), generator=gen,
                                        dtype=torch.float32).to(DEVICE))
            h = histograms_for("voice", subjects, enroll, probes, M, q,
                               overlap, key, recognition, pair)
            h["dlink_curve"], h["dlink_global"] = unlinkability_curve(
                h["mated"], h["nonmated"])
            h["unprotected"] = cosine_histograms(subjects, enroll, probes)
            h["unprotected"]["EER"] = eer_cosine(
                h["unprotected"]["genuine"], h["unprotected"]["impostor"])
            result["modalities"]["voice"]["partitions"]["external"] = h
            print(f"\n  external     {h['n_subjects']} speakers, "
                  f"{h['n_genuine']} genuine / {h['n_impostor']} impostor")
            print(f"    protected   EER "
                  f"{eer_check(h['genuine'], h['impostor'], M) * 100:7.4f}%")
            print(f"    unprotected EER "
                  f"{h['unprotected']['EER'] * 100:7.4f}%")
        except Exception as exc:
            print(f"\n  external: skipped - {type(exc).__name__}: {exc}")
    else:
        print("\n  external: skipped - VCTK embeddings or runtime not found")

    atomic_json(output, result)
    print("\nWrote", output)
    return result


print("Score dump ready. Run run_score_dump().")


In [ ]:
%%writefile {RUNTIME}/inversion_only.py
# 4. The study's full-knowledge inversion attack
"""Full-knowledge inversion evaluation for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_INVERSION.ipynb after its preflight
cell, and after ablation_only.py has supplied the frozen-pipeline
primitives. Pre-specification is in THREAT_MODEL.md; budget and success
criteria were fixed before this was run.

Writes one file, runs/inversion/inversion_result.json, and no seal.
"""

INV_RESTARTS = 5
INV_STEPS = 800
INV_LR = 0.05
INV_TEMP0, INV_TEMP1 = 1.0, 0.05
INV_N_BOOTSTRAP = 2000
INV_CONFIDENCE = 0.95


def poly_transform_diff(vectors, coefficients, exponents, overlap):
    """Differentiable polynomial, numerically identical to the study's.

    The study calls torch.pow(values, int64_tensor). Casting that exponent
    to float to get gradients would produce nan on negative bases, so the
    powers are built with Python integer exponents instead, which keeps
    integer-power semantics and a defined gradient.
    """
    idx, mask = poly_indices(vectors.shape[-1], overlap, vectors.device)
    values = vectors[..., idx] * mask.to(vectors.dtype)
    coefficients = torch.as_tensor(
        coefficients, device=vectors.device, dtype=vectors.dtype)
    powers = torch.stack(
        [values[..., i] ** int(exponents[i]) for i in range(values.shape[-1])],
        dim=-1,
    )
    return (powers * coefficients).sum(dim=-1)


def assert_diff_matches(modality, key, overlap, d, tolerance=1e-4):
    """Assertion 1 - the differentiable path is the study's path.

    Checked on RELATIVE error, and on the bucket indices it produces, not
    on absolute error. torch.pow(x, int64_tensor) and x ** int dispatch to
    different kernels, so they round differently; on a polynomial whose
    intermediate values can be large that shows as an absolute gap of a few
    1e-4 while the relative gap is ~1e-7. An earlier absolute threshold of
    1e-4 here was simply the wrong test.

    What actually matters is that no bucket index moves. This function
    checks that directly on the probes, and assertion 2 then confirms it
    end to end against the real stored templates.
    """
    generator = torch.Generator(device="cpu")
    generator.manual_seed(H(S0, "INVERSION-CHECK", modality))
    probe = torch.randn((64, d), generator=generator, dtype=torch.float32)
    probe = probe / probe.norm(dim=-1, keepdim=True)   # as real embeddings are
    reference = poly_transform(probe, key[0], key[1], overlap)
    candidate = poly_transform_diff(probe, key[0], key[1], overlap)

    absolute = float((reference - candidate).abs().max())
    scale = float(reference.abs().max())
    relative = absolute / max(scale, 1e-30)
    if not relative < tolerance:
        raise RuntimeError(
            f"{modality}: differentiable polynomial differs from the "
            f"study's by a relative {relative:.3e} (absolute {absolute:.3e} "
            f"at scale {scale:.3e}); the attack would not be attacking the "
            f"real scheme."
        )

    # The decisive check: same argmax, on a tensor of the right width.
    k = 1 + math.ceil((d - G) / (G - overlap))
    probe_generator = torch.Generator(device="cpu")
    probe_generator.manual_seed(H(S0, "INVERSION-CHECK-TENSOR", modality))
    probe_tensor = torch.randn((64, 32, k), generator=probe_generator,
                               dtype=torch.float32)
    left = torch.argmax(torch.einsum("bk,mqk->bmq", reference, probe_tensor), -1)
    right = torch.argmax(torch.einsum("bk,mqk->bmq", candidate, probe_tensor), -1)
    moved = int((left != right).sum())
    if moved:
        raise RuntimeError(
            f"{modality}: {moved} bucket indices move between the study's "
            f"polynomial and the differentiable one; the attack would not "
            f"be attacking the real scheme."
        )
    print(f"    assertion 1 OK: differentiable polynomial matches "
          f"(relative {relative:.2e}, absolute {absolute:.2e} at scale "
          f"{scale:.2e}) and moves no bucket index")


def project(arm, z, key, overlap, A):
    """Arm-specific pre-projection, rescaled to unit norm.

    argmax within a group is invariant to a positive per-sample rescale, so
    this changes no bucket index (assertion 2 checks that empirically). It
    exists so the annealed temperature means the same thing for every arm:
    the polynomial's output norm is not 1, and without this the three arms
    would be attacked at different effective temperatures, which would make
    the comparison unfair and could overflow the softmax.
    """
    if arm == "polyiom":
        x = poly_transform_diff(z, key[0], key[1], overlap)
    elif arm == "randproj_iom":
        x = z @ A
    elif arm == "iom_only":
        x = z
    else:
        raise ValueError(arm)
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def codes_of(arm, z, key, overlap, tensor, A):
    """Hard bucket indices, no gradient."""
    with torch.no_grad():
        scores = torch.einsum(
            "bk,mqk->bmq", project(arm, z, key, overlap, A), tensor)
        return torch.argmax(scores, dim=-1)


def invert_arm(arm, targets, key, overlap, tensor, A, d, seed,
               restarts=INV_RESTARTS, steps=INV_STEPS, lr=INV_LR):
    """Attack every template at once. Identical budget for every arm."""
    n = targets.shape[0]
    best_z = torch.zeros((n, d), dtype=torch.float32, device=DEVICE)
    best_hits = torch.full((n,), -1, dtype=torch.int64, device=DEVICE)

    for restart in range(restarts):
        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed + restart)
        z = torch.randn((n, d), generator=generator, dtype=torch.float32)
        z = (z / z.norm(dim=-1, keepdim=True)).to(DEVICE).requires_grad_(True)
        optimiser = torch.optim.Adam([z], lr=lr)

        for step in range(steps):
            temperature = INV_TEMP0 * (INV_TEMP1 / INV_TEMP0) ** (step / max(steps - 1, 1))
            scores = torch.einsum(
                "bk,mqk->bmq", project(arm, z, key, overlap, A), tensor)
            loss = torch.nn.functional.cross_entropy(
                (scores / temperature).reshape(-1, scores.shape[-1]),
                targets.reshape(-1),
            )
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([z], 5.0)
            optimiser.step()
            with torch.no_grad():
                z /= z.norm(dim=-1, keepdim=True).clamp_min(1e-12)

        hits = (codes_of(arm, z.detach(), key, overlap, tensor, A)
                == targets).sum(dim=-1)
        improved = hits > best_hits
        best_hits = torch.where(improved, hits, best_hits)
        best_z[improved] = z.detach()[improved]
        print(f"      {arm:<13} restart {restart + 1}/{restarts}  "
              f"loss {float(loss.detach()):7.4f}  best mean hits "
              f"{float(best_hits.float().mean()):7.2f}")

    return best_z, best_hits


def cosine_rows(a, b):
    a = a / a.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    b = b / b.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return (a * b).sum(dim=-1)


def inversion_for(modality):
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n} d={d} k={k} M={M} q={q} tau={tau}")

    assert_diff_matches(modality, key, overlap, d)

    frozen = load_iom_tensor(modality, overlap, d)[:M, :q].to(DEVICE, torch.float32)
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only")[:M, :q].to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj")[:M, :q].to(DEVICE),
    }
    A = abl_randn((d, k), modality, "projmap").to(DEVICE) / math.sqrt(d)

    cross_generator = torch.Generator(device="cpu")
    cross_generator.manual_seed(seed_unlink(modality, 1))
    cross = {
        "polyiom": torch.randn((M, q, k), generator=cross_generator,
                               dtype=torch.float32).to(DEVICE),
        "iom_only": abl_randn((M, q, d), modality, "unlink|raw|1").to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "unlink|prj|1").to(DEVICE),
    }

    truth = torch.as_tensor(
        np.stack([enroll[uid] for uid in subjects]),
        dtype=torch.float32, device=DEVICE)

    out = {}
    for arm in ARMS:
        print(f"    --- {arm}")
        targets = codes_of(arm, truth, key, overlap, recognition[arm], A)

        if arm == "polyiom":                       # assertion 2
            stored = torch.as_tensor(
                hash_arm([enroll[uid] for uid in subjects], arm, key, overlap,
                         recognition[arm], M, q, A).astype(np.int64),
                device=DEVICE)
            if not bool((stored == targets).all()):
                raise RuntimeError(
                    f"{modality}: attack targets do not equal the stored "
                    f"template; the attack would be aiming at the wrong thing."
                )
            print("    assertion 2 OK: attack targets equal the stored "
                  "templates, so a perfect solution exists in the search space")

        guess_generator = torch.Generator(device="cpu")
        guess_generator.manual_seed(H(S0, "INVERSION-GUESS", modality, arm))
        guess = torch.randn((n, d), generator=guess_generator, dtype=torch.float32)
        guess = (guess / guess.norm(dim=-1, keepdim=True)).to(DEVICE)
        guess_hits = (codes_of(arm, guess, key, overlap, recognition[arm], A)
                      == targets).sum(dim=-1)

        recovered, hits = invert_arm(
            arm, targets, key, overlap, recognition[arm], A, d,
            seed=H(S0, "INVERSION", modality, arm) % (2 ** 31))

        cross_true = codes_of(arm, truth, key, overlap, cross[arm], A)
        cross_rec = codes_of(arm, recovered, key, overlap, cross[arm], A)
        cross_hits = (cross_true == cross_rec).sum(dim=-1)
        cross_impostor = float(
            (cross_true[:, None, :] == cross_true[None, :, :]).sum(-1)
            [~torch.eye(n, dtype=torch.bool, device=DEVICE)].float().mean())

        cos_true = cosine_rows(recovered, truth)
        chance = cosine_rows(truth[:, None, :].expand(n, n, d).reshape(-1, d),
                             truth[None, :, :].expand(n, n, d).reshape(-1, d))
        chance = chance.reshape(n, n)[~torch.eye(n, dtype=torch.bool,
                                                 device=DEVICE)]

        out[arm] = {
            "hits": hits.cpu().numpy(),
            "guess_hits": guess_hits.cpu().numpy(),
            "cos": cos_true.cpu().numpy(),
            "cross_hits": cross_hits.cpu().numpy(),
            "cross_impostor_mean": cross_impostor,
            "cos_chance_mean": float(chance.mean()),
        }
        accepted = float((hits >= tau).float().mean())
        print(f"    {arm:<13} SAR {accepted * 100:6.2f}%   "
              f"mean hits {float(hits.float().mean()):6.2f}/{M}   "
              f"random guess {float(guess_hits.float().mean()):5.2f}/{M} "
              f"(chance {M / q:.1f})")

    return {"modality": modality, "subjects": subjects, "M": M, "tau": tau,
            "d": d, "k": k, "n": n, "arms": out, "chance": M / q}


def run_inversion(n_bootstrap=INV_N_BOOTSTRAP, confidence=INV_CONFIDENCE):
    output = DIR["runs"] / "inversion" / "inversion_result.json"
    if output.is_file():
        print("Resume: stored inversion result found; returning it unchanged.")
        return json.loads(output.read_text())

    result = {
        "schema": 1, "analysis": "full_knowledge_inversion",
        "adversary": "level 3: holds K*, R, the algorithm and the template",
        "attack": "annealed-softmax gradient descent on the unit sphere, "
                  "identical budget for every arm",
        "budget": {"restarts": INV_RESTARTS, "steps": INV_STEPS,
                   "lr": INV_LR, "temperature": [INV_TEMP0, INV_TEMP1]},
        "measures": "practical inversion resistance under this budget; "
                    "NOT information-theoretic irreversibility",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap), "confidence": float(confidence),
        "master_seed": S0, "changes_any_seal": False, "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = inversion_for(modality)
        n, tau, M = record["n"], record["tau"], record["M"]
        rng = np.random.default_rng(H(S0, "INVERSION-BOOTSTRAP", modality))

        accepted = {a: (record["arms"][a]["hits"] >= tau).astype(float)
                    for a in ARMS}
        draws = {a: [] for a in ARMS}
        contrasts = {a: [] for a in ARMS if a != "polyiom"}
        for _ in range(n_bootstrap):
            w = rng.multinomial(n, np.full(n, 1 / n)) / float(n)
            values = {a: float(w @ accepted[a]) for a in ARMS}
            for a in ARMS:
                draws[a].append(values[a])
            for a in contrasts:
                contrasts[a].append(values[a] - values["polyiom"])

        alpha = 1.0 - confidence

        def interval(v):
            lo, hi = np.quantile(v, [alpha / 2, 1 - alpha / 2])
            return [float(lo), float(hi)]

        result["modalities"][modality] = {
            "n_subjects": n, "M": M, "tau": tau, "d": record["d"],
            "k": record["k"],
            "arms": {a: {
                "SAR": float(accepted[a].mean()),
                "SAR_CI": interval(draws[a]),
                "mean_collisions": float(record["arms"][a]["hits"].mean()),
                "random_guess_collisions":
                    float(record["arms"][a]["guess_hits"].mean()),
                "chance_collisions": record["chance"],
                "cos_to_true": float(record["arms"][a]["cos"].mean()),
                "cos_chance": record["arms"][a]["cos_chance_mean"],
                "cross_key_collisions":
                    float(record["arms"][a]["cross_hits"].mean()),
                "cross_key_impostor_mean":
                    record["arms"][a]["cross_impostor_mean"],
            } for a in ARMS},
            "contrasts_vs_polyiom": {a: {
                "delta_SAR": float(accepted[a].mean()
                                   - accepted["polyiom"].mean()),
                "delta_SAR_CI": interval(contrasts[a]),
            } for a in contrasts},
        }

    atomic_json(output, result)
    print("\nWrote", output)
    return result


def inversion_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  n={data['n_subjects']}  M={data['M']}  "
              f"tau={data['tau']}")
        print(f"  {'arm':<14}{'SAR %':>22}{'hits/M':>10}{'guess':>8}"
              f"{'cos':>8}{'chance':>8}{'xkey':>7}{'xkey imp':>10}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}{v['SAR'] * 100:7.2f} "
                  f"[{v['SAR_CI'][0] * 100:5.2f},{v['SAR_CI'][1] * 100:6.2f}]"
                  f"{v['mean_collisions']:10.1f}"
                  f"{v['random_guess_collisions']:8.1f}"
                  f"{v['cos_to_true']:8.3f}{v['cos_chance']:8.3f}"
                  f"{v['cross_key_collisions']:7.1f}"
                  f"{v['cross_key_impostor_mean']:10.1f}")
        chance = list(data["arms"].values())[0].get("chance_collisions")
        if chance:
            print(f"  a random unit vector lands on {chance:.1f} of "
                  f"{data['M']} buckets by chance; 'guess' should sit near it")
        print("  contrasts (arm minus proposed; interval excluding 0 is firm)")
        for arm, c in data["contrasts_vs_polyiom"].items():
            lo, hi = c["delta_SAR_CI"]
            firm = "FIRM" if (lo > 0 or hi < 0) else "not distinguishable"
            print(f"    {arm:<14} dSAR {c['delta_SAR'] * 100:+7.2f} %  "
                  f"[{lo * 100:+7.2f},{hi * 100:+7.2f}]  {firm}")
    print("\n  A POSITIVE dSAR means that arm is EASIER to invert than the")
    print("  proposed method, i.e. the polynomial helped. 'cos' against")
    print("  'chance' says whether the biometric itself was recovered;")
    print("  'xkey' against 'xkey imp' says whether the attack found the")
    print("  subject or merely a preimage of this one hash.")


print("Inversion evaluation ready.")


In [ ]:
%%writefile {RUNTIME}/revocation_only.py
# 5. The study's revocation test
"""Post-revocation acceptance for AttackAware PolyIoM v1.1.4.

Loaded after ablation_only.py and inversion_only.py. Answers the one
question the inversion run left open: does the attacker's reconstruction,
built from an OLD template, still get in after the subject re-enrols under
a FULLY re-keyed system?

The inversion run re-keyed the IoM projection R only. This re-keys
everything each arm has:

  polyiom       fresh polynomial key K** AND fresh projection R'
  randproj_iom  fresh random map A'      AND fresh projection R'
  iom_only      fresh projection R'      (it has nothing else)

The attacker never re-attacks. They hold one stolen template, reconstruct
once, and the subject then re-enrols. That is the real revocation
scenario.

Headline metric: **PRAR**, post-revocation acceptance rate - the fraction
of subjects whose old-template reconstruction is accepted at the sealed
threshold after full re-keying. Revocation works only if PRAR falls to the
impostor floor.

The first version of this used 3 fresh keys and bootstrapped over
identities. That was the wrong axis: with one catastrophic key out of
three, an identity bootstrap concentrates tightly around 1/3 and reports a
confidence interval for a question nobody asked. The dominant uncertainty
is over KEYS. This version draws 40 and bootstraps over both, and reports
the per-key distribution as well as the mean, because "fails for 1 key in
3" and "fails 34% of the time" are different claims.

K** is drawn to match K*'s structure by resampling its own coefficient and
exponent values, NOT by re-running Stage A's inversion gate. For this
question that is the right construction: we are asking whether z'
resembles z in the ways any polynomial of that family sees, not whether
K** is a good key. Three independent draws guard against a fluke.

Writes one file, runs/revocation/revocation_result.json, and no seal.
"""

REV_N_KEYS = 40
REV_N_BOOTSTRAP = 2000
REV_CONFIDENCE = 0.95


def fresh_poly_key(modality, key, index):
    """A new polynomial key matched to K*'s own value distribution."""
    C, E = key
    rng = np.random.default_rng(H(S0, "REVOCATION-KEY", modality, index))
    return (rng.choice(np.asarray(C).reshape(-1), size=np.asarray(C).shape,
                       replace=True).astype(np.float32),
            rng.choice(np.asarray(E).reshape(-1), size=np.asarray(E).shape,
                       replace=True).astype(np.int64))


def fresh_params(arm, modality, index, key, M, q, d, k):
    """Everything this arm would re-issue on revocation."""
    label = f"revocation|{index}"
    if arm == "polyiom":
        return (fresh_poly_key(modality, key, index),
                None,
                abl_randn((M, q, k), modality, f"{label}|R|poly").to(DEVICE))
    if arm == "randproj_iom":
        return (key,
                abl_randn((d, k), modality, f"{label}|A").to(DEVICE) / math.sqrt(d),
                abl_randn((M, q, k), modality, f"{label}|R|prj").to(DEVICE))
    if arm == "iom_only":
        return (key, None,
                abl_randn((M, q, d), modality, f"{label}|R|raw").to(DEVICE))
    raise ValueError(arm)


def revocation_for(modality):
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n} d={d} k={k} M={M} q={q} tau={tau}")
    assert_diff_matches(modality, key, overlap, d)

    frozen = load_iom_tensor(modality, overlap, d)[:M, :q].to(DEVICE, torch.float32)
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only").to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj").to(DEVICE),
    }
    A = abl_randn((d, k), modality, "projmap").to(DEVICE) / math.sqrt(d)
    truth = torch.as_tensor(np.stack([enroll[uid] for uid in subjects]),
                            dtype=torch.float32, device=DEVICE)

    out = {}
    for arm in ARMS:
        print(f"    --- {arm}")
        targets = codes_of(arm, truth, key, overlap, recognition[arm], A)
        # Same seeds as the inversion run, so z' is bit-identical to it.
        recovered, hits = invert_arm(
            arm, targets, key, overlap, recognition[arm], A, d,
            seed=H(S0, "INVERSION", modality, arm) % (2 ** 31))

        accepted_per_key, impostor_per_key, degenerate = [], [], []
        for index in range(REV_N_KEYS):
            new_key, new_A, new_R = fresh_params(
                arm, modality, index, key, M, q, d, k)
            eff_A = new_A if new_A is not None else A
            true_codes = codes_of(arm, truth, new_key, overlap, new_R, eff_A)
            off = ~torch.eye(n, dtype=torch.bool, device=DEVICE)
            floor = float((true_codes[:, None, :] == true_codes[None, :, :])
                          .sum(-1)[off].float().mean())
            # A resampled key can pair a large exponent with a small
            # coefficient so hard that every power underflows, the projection
            # collapses to zero, and argmax returns bucket 0 for everyone.
            # That would read as PRAR 100% for entirely the wrong reason.
            # Such keys are recorded and excluded, not silently averaged in:
            # how many there are is itself a finding about the key family.
            if floor > M / 4:
                degenerate.append(index)
                continue
            rec_codes = codes_of(arm, recovered, new_key, overlap, new_R, eff_A)
            agree = (true_codes == rec_codes).sum(dim=-1)
            accepted_per_key.append((agree >= tau).float().cpu().numpy())
            impostor_per_key.append(floor)

        if len(accepted_per_key) < 2:
            raise RuntimeError(
                f"{modality}/{arm}: only {len(accepted_per_key)} of "
                f"{REV_N_KEYS} fresh keys were usable; the key family is too "
                f"fragile for this test to say anything."
            )
        matrix = np.stack(accepted_per_key)          # (usable_keys, n)
        per_key = matrix.mean(axis=1)                # PRAR of each key
        print(f"      {len(matrix)}/{REV_N_KEYS} keys usable"
              f"{f' ({len(degenerate)} degenerate, excluded)' if degenerate else ''}"
              f"   per-key PRAR: median {np.median(per_key) * 100:6.2f}%"
              f"  min {per_key.min() * 100:6.2f}%  max {per_key.max() * 100:6.2f}%"
              f"   keys failing outright (>50%): "
              f"{float((per_key >= 0.5).mean()) * 100:5.1f}%")

        out[arm] = {
            "matrix": matrix,                          # (keys, identities)
            "per_key": per_key,
            "cos": cosine_rows(recovered, truth).cpu().numpy(),
            "hits": hits.cpu().numpy(),
            "impostor_floor": float(np.mean(impostor_per_key)),
            "n_degenerate": len(degenerate),
        }
        print(f"    {arm:<13} PRAR {matrix.mean() * 100:6.2f}%   "
              f"cos {out[arm]['cos'].mean():.3f}")

    return {"modality": modality, "M": M, "tau": tau, "d": d, "k": k,
            "n": n, "arms": out}


def run_revocation(n_bootstrap=REV_N_BOOTSTRAP, confidence=REV_CONFIDENCE):
    output = DIR["runs"] / "revocation" / "revocation_result.json"
    if output.is_file():
        print("Resume: stored revocation result found; returning it unchanged.")
        return json.loads(output.read_text())

    result = {
        "schema": 1, "analysis": "post_revocation_acceptance",
        "question": "does an old template's reconstruction still get in "
                    "after the subject re-enrols under a fully re-keyed system",
        "re_keys": {"polyiom": "fresh K** and fresh R",
                    "randproj_iom": "fresh A and fresh R",
                    "iom_only": "fresh R"},
        "n_fresh_keys": REV_N_KEYS,
        "fresh_key_construction": "resampled from K*'s own coefficient and "
                                  "exponent values; NOT passed through "
                                  "Stage A's inversion gate",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap), "confidence": float(confidence),
        "master_seed": S0, "changes_any_seal": False, "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = revocation_for(modality)
        n = record["n"]
        rng = np.random.default_rng(H(S0, "REVOCATION-BOOTSTRAP", modality))
        stats = {a: {"PRAR": [], "cos": []} for a in ARMS}
        contrasts = {a: {"delta_PRAR": [], "delta_cos": []}
                     for a in ARMS if a != "polyiom"}
        n_keys = min(len(record["arms"][a]["matrix"]) for a in ARMS)

        # Two-level: resample KEYS and IDENTITIES in every replicate, because
        # both vary and an identity-only bootstrap understates the spread.
        # Key index and identity weights are shared across arms, so a contrast
        # is paired on both.
        for _ in range(n_bootstrap):
            picks = rng.integers(0, n_keys, size=n_keys)
            w = rng.multinomial(n, np.full(n, 1 / n)) / float(n)
            values = {}
            for a in ARMS:
                arm_matrix = record["arms"][a]["matrix"][:n_keys]
                values[a] = (float(np.mean(arm_matrix[picks] @ w)),
                             float(w @ record["arms"][a]["cos"]))
                stats[a]["PRAR"].append(values[a][0])
                stats[a]["cos"].append(values[a][1])
            for a in contrasts:
                contrasts[a]["delta_PRAR"].append(values[a][0] - values["polyiom"][0])
                contrasts[a]["delta_cos"].append(values[a][1] - values["polyiom"][1])

        alpha = 1.0 - confidence

        def interval(v):
            lo, hi = np.quantile(v, [alpha / 2, 1 - alpha / 2])
            return [float(lo), float(hi)]

        result["modalities"][modality] = {
            "n_subjects": n, "M": record["M"], "tau": record["tau"],
            "d": record["d"], "k": record["k"],
            "bootstrap_axes": "keys and identities",
            "arms": {a: {
                "PRAR": float(record["arms"][a]["matrix"].mean()),
                "PRAR_CI": interval(stats[a]["PRAR"]),
                "per_key_PRAR": [float(x) for x in record["arms"][a]["per_key"]],
                "per_key_median": float(np.median(record["arms"][a]["per_key"])),
                "per_key_min": float(record["arms"][a]["per_key"].min()),
                "per_key_max": float(record["arms"][a]["per_key"].max()),
                "keys_failing_outright":
                    float((record["arms"][a]["per_key"] >= 0.5).mean()),
                "n_keys_usable": int(len(record["arms"][a]["matrix"])),
                "n_keys_degenerate": int(record["arms"][a]["n_degenerate"]),
                "cos_to_true": float(record["arms"][a]["cos"].mean()),
                "cos_CI": interval(stats[a]["cos"]),
                "impostor_floor_collisions": record["arms"][a]["impostor_floor"],
                "attack_collisions": float(record["arms"][a]["hits"].mean()),
            } for a in ARMS},
            "contrasts_vs_polyiom": {a: {
                "delta_PRAR": float(record["arms"][a]["matrix"].mean()
                                    - record["arms"]["polyiom"]["matrix"].mean()),
                "delta_PRAR_CI": interval(contrasts[a]["delta_PRAR"]),
                "delta_cos": float(record["arms"][a]["cos"].mean()
                                   - record["arms"]["polyiom"]["cos"].mean()),
                "delta_cos_CI": interval(contrasts[a]["delta_cos"]),
            } for a in contrasts},
        }

    atomic_json(output, result)
    print("\nWrote", output)
    return result


def revocation_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  n={data['n_subjects']}  M={data['M']}  "
              f"tau={data['tau']}")
        print(f"  {'arm':<14}{'PRAR %':>22}{'per-key med/min/max %':>26}"
              f"{'keys fail':>11}{'keys ok':>9}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}"
                  f"{v['PRAR'] * 100:7.2f} [{v['PRAR_CI'][0] * 100:5.2f},"
                  f"{v['PRAR_CI'][1] * 100:6.2f}]"
                  f"{v['per_key_median'] * 100:9.2f}"
                  f"{v['per_key_min'] * 100:7.2f}"
                  f"{v['per_key_max'] * 100:7.2f}"
                  f"{v['keys_failing_outright'] * 100:10.1f}%"
                  f"{v['n_keys_usable']:6d}/{v['n_keys_usable'] + v['n_keys_degenerate']}")
        print(f"  {'':14}{'cosine to true embedding':>22}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}{v['cos_to_true']:11.3f} "
                  f"[{v['cos_CI'][0]:6.3f},{v['cos_CI'][1]:6.3f}]")
        print("  contrasts (arm minus proposed; interval excluding 0 is firm)")
        for arm, c in data["contrasts_vs_polyiom"].items():
            for name, scale, unit in (("PRAR", 100, " %"), ("cos", 1, "")):
                value, (lo, hi) = c[f"delta_{name}"], c[f"delta_{name}_CI"]
                firm = "FIRM" if (lo > 0 or hi < 0) else "not distinguishable"
                print(f"    {arm:<14} d{name:<5}{value * scale:+8.3f}{unit}  "
                      f"[{lo * scale:+7.3f},{hi * scale:+7.3f}]  {firm}")
    print("\n  PRAR is the fraction of subjects whose OLD template's")
    print("  reconstruction is still accepted after full re-keying.")
    print("  Intervals resample BOTH keys and identities.")
    print("  'keys fail' is the share of fresh keys for which revocation")
    print("  fails outright (PRAR >= 50%) - the number to quote when the")
    print("  per-key spread is wide, because a mean over keys hides it.")
    print("  'keys ok' counts keys that stayed discriminating; degenerate")
    print("  ones are excluded, and a large exclusion count is itself a")
    print("  finding about how fragile the key family is.")


print("Revocation evaluation ready.")


In [ ]:
%%writefile {RUNTIME}/figures_only.py
# 6. The study's code for Figures A to D
"""Results figures for AttackAware PolyIoM v1.1.4.

Reads runs/scores/score_histograms.json and writes four figures, one per
ISO/IEC 24745 criterion the paper has to address. Read-only: touches no
seal, fits no threshold, computes no new score.

  A  det_preservation     the protected system's DET curves, and what
                          protection cost against the unprotected baseline
  B  score_separability   genuine and impostor, with tau* marked
  C  unlinkability        mated and non-mated, with D_link(s) overlaid
  D  revocability         genuine, impostor and pseudo-impostor together

Colour carries one meaning across all four figures
--------------------------------------------------
  blue    genuine comparisons, or the protected system
  grey    impostor comparisons, or non-mated pairs
  green   pseudo-impostor comparisons (a revoked key), or face
  orange  the unprotected baseline, and only ever that

Why figure A does not draw the baseline as a DET curve on voice
---------------------------------------------------------------
The unprotected voice baseline produces fewer than one genuine error: the
resolution of the evaluation split is 0.17% and of the external split
0.09%, and the measured EERs are 0.085% and 0.003%. A DET curve through
that region would sit off the bottom-left of any readable axis and would
imply a precision the trial count cannot support. The ratio it suggests
(22.7x held-out, 848x external) is an artifact of dividing by an
unresolved number.

The right panel therefore reports EER directly, with the baseline drawn as
a one-sided 95% upper bound wherever it is unresolved and as a point only
where it is genuinely measured. A reader can then see which contrasts the
data settle and which it does not.

Why figure C marks part of its own curve unreliable
---------------------------------------------------
The local unlinkability score D_link(s) is a ratio of two densities, and
the mated density is estimated from only one comparison per identity. In
score bins where the non-mated histogram is nearly empty the ratio is
driven by single observations and the curve runs to 1 for that reason
alone. Those bins are shaded rather than silently plotted.

Why the mated distribution appears twice
----------------------------------------
Figure C uses it as the unlinkability mated set and figure D uses it as
the revocability pseudo-impostor set. It is one measurement. Both captions
say so, and the paper must not count it as two independent results.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator

GEN = "#1c5cab"         # genuine, and the protected system
IMP = "#8a919c"         # impostor, and non-mated
PSE = "#158a5f"         # pseudo-impostor, and face
BASE = "#c94e1f"        # the unprotected baseline, and nothing else
INK = "#1a1a1a"
MUTE = "#6b7280"
GRID = "#d9dde3"
WARN = "#9aa1ab"
DERIV = INK          # a derived measure, never a data series

COL_W = 7.16            # double-column width, inches
DPI = 400

# Bootstrap intervals carried from the sealed held-out and external
# analyses. Not recomputed here: the histograms record distributions, not
# the identity-cluster resampling those intervals came from. Development
# is the selection split and was never given an interval.
CARRIED_CI = {
    ("voice", "evaluation"): (1.928, 1.092, 2.919),
    ("voice", "external"): (2.798, 1.992, 3.807),
    ("face", "evaluation"): (4.799, 1.945, 10.345),
}

ROWS = [("voice", "development"), ("voice", "evaluation"),
        ("voice", "external"), ("face", "development"),
        ("face", "evaluation")]
PANELS = [("voice", "evaluation"), ("face", "evaluation")]
LABEL = {("voice", "development"): "Voice, development (42)",
         ("voice", "evaluation"): "Voice, held-out (58)",
         ("voice", "external"): "Voice, external VCTK (110)",
         ("face", "development"): "Face, development (42)",
         ("face", "evaluation"): "Face, held-out (58)"}
TITLE = {("voice", "evaluation"): "Voice, 58 held-out speakers",
         ("face", "evaluation"): "Face, 58 held-out subjects",
         ("voice", "external"): "Voice, 110 external speakers",
         ("voice", "development"): "Voice, 42 development speakers",
         ("face", "development"): "Face, 42 development subjects"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "axes.edgecolor": MUTE, "axes.linewidth": 0.6,
    "xtick.color": MUTE, "ytick.color": MUTE, "text.color": INK,
    "axes.labelcolor": INK, "axes.spines.top": False,
    "axes.spines.right": False, "figure.facecolor": "white",
    "savefig.facecolor": "white",
})


def _ndtri(p):
    from scipy.special import ndtri as f
    return f(np.clip(np.asarray(p, float), 1e-9, 1 - 1e-9))


def rates(genuine, impostor):
    """FMR and FNMR at every threshold, from two histograms.

    A comparison is accepted at a score of at least the threshold, so FMR
    is the upper tail of the impostor histogram and FNMR the lower tail of
    the genuine one.
    """
    g = np.asarray(genuine, float)
    i = np.asarray(impostor, float)
    fmr = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / max(i.sum(), 1.0)
    fnmr = np.r_[0.0, np.cumsum(g)] / max(g.sum(), 1.0)
    return fmr, fnmr


def eer_from(genuine, impostor):
    fmr, fnmr = rates(genuine, impostor)
    d = fmr - fnmr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(fmr[exact[0]]), int(exact[0])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    if len(cross):
        j = int(cross[0])
        w = d[j] / (d[j] - d[j + 1])
        return float(fmr[j] + w * (fmr[j + 1] - fmr[j])), j
    j = int(np.argmin(np.maximum(fmr, fnmr)))
    return float((fmr[j] + fnmr[j]) / 2.0), j


def baseline_bound(eer, n_genuine):
    """One-sided 95% upper bound on an unresolved baseline error rate.

    Returns (bound, resolved). An EER below one genuine error means the
    crossing sits where the genuine side has at most one mistake, so the
    point estimate carries no information and only a bound is honest.
    """
    from scipy.stats import beta
    floor = 1.0 / n_genuine
    k = int(np.floor(eer * n_genuine + 1e-9))
    bound = float(beta.ppf(0.95, k + 1, n_genuine - k))
    return bound, bool(eer >= floor)


def support(arrays, M, pad=0.10):
    """Score range actually occupied, so panels do not waste width.

    The distributions occupy a third of 0..M or less. Plotting the whole
    range leaves most of every panel blank and shrinks the part a reader
    needs to see. The axis label states that the axis is truncated.
    """
    lo, hi = M, 0
    for a in arrays:
        nz = np.flatnonzero(np.asarray(a, float) > 0)
        if len(nz):
            lo, hi = min(lo, int(nz[0])), max(hi, int(nz[-1]))
    if hi <= lo:
        return 0, M
    span = hi - lo
    return max(0, int(lo - pad * span)), min(M, int(hi + pad * span) + 1)


def _det_axes(ax, ticks=(0.1, 1, 5, 20, 50)):
    p = [t / 100 for t in ticks]
    pos = _ndtri(p)
    ax.set_xlim(pos[0], pos[-1])
    ax.set_ylim(pos[0], pos[-1])
    ax.xaxis.set_major_locator(FixedLocator(pos))
    ax.yaxis.set_major_locator(FixedLocator(pos))
    ax.set_xticklabels([f"{t:g}" for t in ticks])
    ax.set_yticklabels([f"{t:g}" for t in ticks])
    ax.grid(True, color=GRID, linewidth=0.5)
    ax.set_axisbelow(True)


def figure_a(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 3.55),
                             gridspec_kw={"width_ratios": [1, 1.28]})

    # Left: the protected system's DET curves, which are measurable.
    ax = axes[0]
    curves = [(("voice", "evaluation"), GEN, "-", "Voice, held-out"),
              (("voice", "external"), GEN, (0, (4, 2)), "Voice, external"),
              (("face", "evaluation"), PSE, "-", "Face, held-out")]
    for (mod, part), colour, style, label in curves:
        parts = data["modalities"][mod]["partitions"]
        if part not in parts:
            continue
        h = parts[part]
        fmr, fnmr = rates(h["genuine"], h["impostor"])
        keep = (fmr > 0) & (fnmr > 0)
        eer, _ = eer_from(h["genuine"], h["impostor"])
        ax.plot(_ndtri(fmr[keep]), _ndtri(fnmr[keep]), color=colour,
                linewidth=1.5, linestyle=style, zorder=4,
                label=f"{label}  {eer * 100:.2f}%")
        p = _ndtri(eer)
        ax.plot([p], [p], "o", color=colour, markersize=3.6,
                markeredgecolor="white", markeredgewidth=0.6, zorder=5)
    # Stops short of the corner so the legend sits on clean ground.
    lim = _ndtri((0.001, 0.22))
    ax.plot(lim, lim, color=MUTE, linewidth=0.5, linestyle=(0, (4, 3)))
    _det_axes(ax)
    ax.set_xlabel("False match rate (%)")
    ax.set_ylabel("False non-match rate (%)")
    ax.set_title("Protected system, equal-error point marked", color=INK,
                 loc="left")
    ax.legend(loc="upper right", frameon=False, handlelength=1.8,
              borderpad=0.2, labelspacing=0.3)

    # Right: what protection cost, and what the data can resolve.
    ax = axes[1]
    ypos = np.arange(len(ROWS))[::-1]
    for y, (mod, part) in zip(ypos, ROWS):
        parts = data["modalities"][mod]["partitions"]
        if part not in parts:
            continue
        h = parts[part]
        prot, _ = eer_from(h["genuine"], h["impostor"])
        prot *= 100
        raw, _ = eer_from(h["unprotected"]["genuine"],
                          h["unprotected"]["impostor"])
        raw *= 100
        bound, resolved = baseline_bound(raw / 100, h["n_genuine"])
        bound *= 100

        # Two rows within a row: on face the protected and unprotected
        # points nearly coincide, and one would hide the other.
        yp, yb = y + 0.16, y - 0.16
        ci = CARRIED_CI.get((mod, part))
        if ci:
            ax.plot([ci[1], ci[2]], [yp, yp], color=GEN, linewidth=1.1,
                    solid_capstyle="butt", zorder=3)
            for e in (ci[1], ci[2]):
                ax.plot([e, e], [yp - 0.12, yp + 0.12], color=GEN,
                        linewidth=1.1, zorder=3)
        ax.plot([prot], [yp], "o", color=GEN, markersize=4.6,
                markeredgecolor="white", markeredgewidth=0.7, zorder=5)

        if resolved:
            ax.plot([raw], [yb], "s", color=BASE, markersize=4.2,
                    markeredgecolor="white", markeredgewidth=0.7, zorder=5)
            ax.plot([raw, bound], [yb, yb], color=BASE, linewidth=1.0,
                    alpha=0.55, zorder=3)
            ax.plot([bound, bound], [yb - 0.12, yb + 0.12], color=BASE,
                    linewidth=1.0, alpha=0.55, zorder=3)
            verdict = "not resolvable" if ci else "no interval"
        else:
            ax.annotate("", xy=(0.02, yb), xytext=(bound, yb),
                        arrowprops=dict(arrowstyle="-|>", color=BASE,
                                        linewidth=1.0, shrinkA=0, shrinkB=0),
                        zorder=4)
            ax.plot([bound], [yb], "|", color=BASE, markersize=7,
                    markeredgewidth=1.2, zorder=5)
            verdict = ("no interval" if not ci else
                       "firm" if ci[1] > bound else "not resolvable")
        ax.text(15.8, y, verdict, fontsize=6.4, va="center", ha="right",
                color=INK if verdict == "firm" else MUTE,
                style="normal" if verdict == "firm" else "italic")

    ax.set_yticks(ypos)
    ax.set_yticklabels([LABEL[r] for r in ROWS], fontsize=7)
    ax.set_xlim(0, 16)
    ax.set_ylim(-0.6, len(ROWS) - 0.4)
    ax.set_xlabel("Equal error rate (%)")
    ax.grid(True, axis="x", color=GRID, linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_title("Protection cost, and what the data settle", color=INK,
                 loc="left")
    handles = [
        plt.Line2D([], [], color=GEN, marker="o", linestyle="-",
                   markersize=4.6, linewidth=1.1,
                   label="Protected, 95% interval"),
        plt.Line2D([], [], color=BASE, marker="s", linestyle="-",
                   markersize=4.2, linewidth=1.0, alpha=0.7,
                   label="Unprotected, measured"),
        plt.Line2D([], [], color=BASE, marker="|", linestyle="none",
                   markersize=7, markeredgewidth=1.2,
                   label="Unprotected, 95% upper bound only"),
    ]
    # Below the axis: every in-panel corner is occupied by a row.
    ax.legend(handles=handles, loc="upper center",
              bbox_to_anchor=(0.5, -0.20), ncol=3, frameon=False,
              handlelength=1.5, borderpad=0.2, columnspacing=1.1,
              fontsize=6.0)
    fig.suptitle("Performance preservation against the unprotected "
                 "baseline", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0.03, 1, 0.93))
    _save(fig, outdir, "figA_det_preservation")


def _dist_panel(ax, x, series, lo, hi, headroom=1.30):
    top = 0.0
    for label, raw, colour, style in series:
        v = np.asarray(raw, float)
        v = v / v.sum()
        top = max(top, v[lo:hi].max() if hi > lo else v.max())
        ax.fill_between(x, v, color=colour, alpha=0.42, linewidth=0, zorder=3)
        ax.plot(x, v, color=colour, linewidth=1.15, linestyle=style,
                zorder=4, label=label)
    ax.set_xlim(lo, hi)
    ax.set_ylim(0, top * headroom)
    return top


def figure_b(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 2.9))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M, tau = entry["M"], entry["tau"]
        lo, hi = support([h["genuine"], h["impostor"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [("Impostor", h["impostor"], IMP, "-"),
                                  ("Genuine", h["genuine"], GEN, "-")],
                          lo, hi)
        eer, _ = eer_from(h["genuine"], h["impostor"])
        ax.plot([tau, tau], [0, top * 0.95], color=INK, linewidth=0.9,
                zorder=5)
        ax.annotate(rf"$\tau^{{*}}={tau}$", xy=(tau, top * 0.055),
                    xytext=(4, 0), textcoords="offset points", fontsize=7,
                    color=INK, ha="left", va="bottom", zorder=6)
        ax.set_title(f"{TITLE[(modality, partition)]}   EER {eer * 100:.2f}%",
                     color=INK, loc="left")
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")
        ax.legend(loc="upper right", frameon=False, handlelength=1.4,
                  borderpad=0.2)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Where the sealed threshold sits in the score "
                 "distributions", fontsize=9.5, x=0.005, ha="left",
                 color=INK, y=0.995)
    fig.tight_layout(rect=(0, 0.02, 1, 0.92))
    _save(fig, outdir, "figB_score_separability")


def figure_c(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 2.95))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M = entry["M"]
        lo, hi = support([h["mated"], h["nonmated"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [("Non-mated", h["nonmated"], IMP, "-"),
                                  ("Mated", h["mated"], PSE, "-")], lo, hi,
                          headroom=1.52)
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")

        # Bins where the non-mated side is nearly empty drive D_link(s) to
        # 1 on single observations. Shade them rather than imply a measure.
        non = np.asarray(h["nonmated"], float)
        thin = non < 5
        edges = np.flatnonzero(thin[:-1] != thin[1:])
        start = 0 if thin[0] else None
        for e in edges:
            if thin[e]:
                if start is not None:
                    ax.axvspan(start, e + 1, color=WARN, alpha=0.16,
                               linewidth=0, zorder=1)
                    start = None
            else:
                start = e + 1
        if start is not None:
            ax.axvspan(start, M, color=WARN, alpha=0.16, linewidth=0,
                       zorder=1)

        twin = ax.twinx()
        twin.plot(x, np.asarray(h["dlink_curve"], float), color=DERIV,
                  linewidth=1.25, zorder=5, label=r"$D_{\mathrm{link}}(s)$")
        twin.set_ylim(0, 1.32)
        twin.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        twin.spines["top"].set_visible(False)
        twin.spines["right"].set_visible(True)
        twin.spines["right"].set_color(MUTE)
        twin.set_ylabel(r"$D_{\mathrm{link}}(s)$", color=DERIV)
        twin.tick_params(axis="y", colors=DERIV)

        ax.set_title(f"{TITLE[(modality, partition)]}   "
                     rf"$D^{{\mathrm{{sys}}}}_{{\leftrightarrow}}=$"
                     f"{h['dlink_global']:.3f}", color=INK, loc="left")
        hl, ll = ax.get_legend_handles_labels()
        h2, l2 = twin.get_legend_handles_labels()
        shade = plt.Rectangle((0, 0), 1, 1, color=WARN, alpha=0.16,
                              label="Fewer than 5 non-mated pairs")
        ax.legend(hl + h2 + [shade], ll + l2 + [shade.get_label()],
                  loc="upper right", frameon=False, handlelength=1.3,
                  borderpad=0.2, labelspacing=0.3, fontsize=6.4)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Unlinkability: can two protected templates be tied to "
                 "one person?", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    _save(fig, outdir, "figC_unlinkability")


def figure_d(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 3.05))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M, tau = entry["M"], entry["tau"]
        lo, hi = support([h["genuine"], h["impostor"], h["mated"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [
            ("Impostor (other person)", h["impostor"], IMP, "-"),
            ("Pseudo-impostor (revoked key)", h["mated"], PSE, (0, (3, 1.6))),
            ("Genuine (current key)", h["genuine"], GEN, "-")], lo, hi,
            headroom=1.58)

        mated = np.asarray(h["mated"], float)
        prar = float(mated[tau:].sum() / mated.sum())
        ax.plot([tau, tau], [0, top * 0.95], color=INK, linewidth=0.9,
                zorder=5)
        ax.annotate(rf"$\tau^{{*}}={tau}$", xy=(tau, top * 0.055),
                    xytext=(4, 0), textcoords="offset points", fontsize=7,
                    color=INK, ha="left", va="bottom", zorder=6)
        ax.annotate(f"{prar * 100:.1f}% of revoked templates still\n"
                    f"accepted at $\\tau^{{*}}$ (PRAR, this key pair)",
                    xy=(0.98, 0.74), xycoords="axes fraction", fontsize=6.2,
                    color=INK, ha="right", va="top", zorder=6)
        ax.set_title(TITLE[(modality, partition)], color=INK, loc="left")
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")
        ax.legend(loc="upper right", frameon=False, handlelength=1.5,
                  borderpad=0.2, labelspacing=0.3, fontsize=6.4)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Revocability: does a template survive its own key being "
                 "revoked?", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0.02, 1, 0.92))
    _save(fig, outdir, "figD_revocability")


def _save(fig, outdir, stem):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(outdir / f"{stem}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {stem}.pdf and {stem}.png")


def make_figures(score_json=None, outdir=None):
    score_json = Path(score_json) if score_json else \
        DIR["runs"] / "scores" / "score_histograms.json"
    outdir = Path(outdir) if outdir else \
        score_json.parent.parent.parent / "figures" / "results"
    data = json.loads(score_json.read_text())
    if data.get("SYNTHETIC"):
        print("*** SYNTHETIC input: layout test only, not for the paper")
    print(f"Read {score_json}")
    for fn in (figure_a, figure_b, figure_c, figure_d):
        fn(data, outdir)
    print(f"\nFour figures in {outdir}")
    return outdir


print("Figure runtime ready. Run make_figures().")


In [ ]:
%%writefile {RUNTIME}/results_plots.py
# 7. Drawing code for Figures 3 to 8, S2 and P
"""Drawing code for the results figures of AttackAware PolyIoM v1.1.4.

One source for two callers:

  reproducibility/figures.py       draws from the stored result files
  ALL_FIGURES notebook (Colab)     draws from results it has just computed

Every function takes the data it draws as an argument and reads nothing
from disk, so the figures are only ever as good as their inputs. The data
layout is the one the study's own runtime writes (ablation, inversion and
revocation results), plus four small pieces described in draw_all().

Needs numpy and matplotlib only.
"""

from pathlib import Path

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ------------------------------------------------------------------ style
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, DEEMPH, CRITICAL = "#e1e0d9", "#c3c2b7", "#c9c8c2", "#d03b3b"
VOICE, FACE = "#2a78d6", "#eb6834"
BLUES = LinearSegmentedColormap.from_list("blues", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95",
    "#0d366b"])

ARMS = ["polyiom", "iom_only", "randproj_iom"]
LABEL = {"polyiom": "PolyIoM (ours)", "iom_only": "IoM only",
         "randproj_iom": "Linear map + IoM"}
HUE = {"polyiom": VOICE, "iom_only": FACE, "randproj_iom": "#1baf7a"}
SHAPE = {"polyiom": "o", "iom_only": "s", "randproj_iom": "^"}

STYLE = {
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "font.size": 8.5, "axes.titlesize": 8.6, "axes.labelsize": 8.2,
    "xtick.labelsize": 7.6, "ytick.labelsize": 7.6, "legend.fontsize": 7.6,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "text.color": INK,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.6, "grid.color": GRID,
    "grid.linewidth": 0.6, "grid.linestyle": "-", "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}

def _check(cond, what):
    if not cond:
        raise ValueError(f"figure input check failed: {what}")


def close(a, b, tol):
    return abs(a - b) <= tol


def eligible(rows, mod, floor):
    e, t = floor[mod]
    return [r for r in rows if r["EER"] <= e and r["TAR_DEV"] >= t]


def save(fig, out, stem):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    # No timestamps in the files, so a rerun reproduces them byte for byte.
    fig.savefig(out / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(out / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {stem}.png and .pdf")
    return stem

def style(ax, grid_axis="both"):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis)
    ax.tick_params(length=3, pad=3)




def figure_selection(D, out):
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(7.4, 3.2))
    style(axa)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        rows = D["sweep"][mod]
        axa.scatter([r["EER"] * 100 for r in rows], [r["TAR_DEV"] * 100 for r in rows],
                    s=26, c=col, alpha=0.75, linewidths=0.7, edgecolors=SURFACE,
                    zorder=3, label=f"{mod} (80 settings)")
        e, t = D["floor"][mod]
        axa.add_patch(Rectangle((0, t * 100), e * 100, 100 - t * 100, facecolor=col,
                                alpha=0.07, edgecolor=col, linewidth=0.7, zorder=1))
    axa.annotate("voice floor", xy=(1.18, 97.8), fontsize=7.2, color=INK2, va="center")
    axa.annotate("face floor", xy=(3.18, 94.8), fontsize=7.2, color=INK2, va="center")
    for mod, col, xytext in (("voice", VOICE, (0.18, 86.5)), ("face", FACE, (4.30, 92.5))):
        s = D["selected"][mod]
        axa.scatter([s["EER"] * 100], [s["TAR_DEV"] * 100], s=124, c=col,
                    linewidths=1.8, edgecolors=SURFACE, zorder=5)
        axa.annotate("selected", xy=(s["EER"] * 100, s["TAR_DEV"] * 100), xytext=xytext,
                     fontsize=7.4, va="center",
                     arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.7,
                                     shrinkA=3, shrinkB=7))
    axa.set_xlabel("EER (%), development")
    axa.set_ylabel("TAR (%), development")
    axa.set_xlim(0, 7.2)
    axa.set_ylim(35, 103)
    axa.set_title("(a)  No face setting meets the voice floor", loc="left", pad=7)
    axa.legend(loc="lower right", handletextpad=0.4, borderpad=0.2)

    style(axb)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        el = eligible(D["sweep"][mod], mod, D["floor"])
        ine = [r for r in D["sweep"][mod] if r not in el]
        axb.scatter([r["EER"] * 100 for r in ine], [r["Dsys_DEV"] for r in ine], s=20,
                    c=DEEMPH, alpha=0.85, linewidths=0.6, edgecolors=SURFACE, zorder=2)
        axb.scatter([r["EER"] * 100 for r in el], [r["Dsys_DEV"] for r in el], s=26,
                    c=col, alpha=0.85, linewidths=0.7, edgecolors=SURFACE, zorder=3,
                    label=f"{mod}, meets floor ({len(el)})")
        s = D["selected"][mod]
        axb.scatter([s["EER"] * 100], [s["Dsys_DEV"]], s=124, c=col, linewidths=1.8,
                    edgecolors=SURFACE, zorder=5)
    axb.scatter([], [], s=20, c=DEEMPH, linewidths=0.6, edgecolors=SURFACE,
                label="below floor")
    axb.annotate("large mark = selected", xy=(7.05, 0.012), fontsize=7.0, color=MUTED,
                 ha="right", va="bottom")
    axb.set_xlabel("EER (%), development")
    axb.set_ylabel("$D_{sys}$, development  (lower is better)")
    axb.set_xlim(0, 7.2)
    axb.set_ylim(0, 0.41)
    axb.set_title("(b)  Lowest $D_{sys}$ that clears each floor", loc="left", pad=7)
    axb.legend(loc="upper right", handletextpad=0.4, borderpad=0.2)
    fig.tight_layout(pad=1.1)
    return save(fig, out, "FigureS2_OperatingPointSelection")


# ============================================== Figure 3: the design space
def figure_design_space(D, out, MS=(32, 64, 128, 256), QS=(4, 8, 16, 32), OS=(0, 1, 2, 3, 4)):
    vals = [r["Dsys_DEV"] for m in D["sweep"] for r in D["sweep"][m]]
    norm = Normalize(vmin=min(vals), vmax=max(vals))
    fig, axes = plt.subplots(2, 5, figsize=(7.5, 3.5),
                             gridspec_kw={"wspace": 0.16, "hspace": 0.13})
    for row, mod in enumerate(("face", "voice")):
        for col, o in enumerate(OS):
            ax = axes[row, col]
            g = np.full((len(MS), len(QS)), np.nan)
            for r in D["sweep"][mod]:
                if r["o"] == o:
                    g[MS.index(r["M"]), QS.index(r["q"])] = r["Dsys_DEV"]
            ax.imshow(g, cmap=BLUES, norm=norm, origin="upper", aspect="auto")
            for i in range(len(MS)):
                for j in range(len(QS)):
                    v = g[i, j]
                    ax.text(j, i, f"{v:.2f}".lstrip("0"), ha="center", va="center",
                            fontsize=5.9, color=SURFACE if norm(v) > 0.55 else INK2)
            s = D["selected"][mod]
            if s["o"] == o:
                ax.add_patch(plt.Rectangle((QS.index(s["q"]) - 0.5, MS.index(s["M"]) - 0.5),
                                           1, 1, fill=False, linewidth=1.9, zorder=5,
                                           edgecolor=FACE if mod == "face" else "#0d366b"))
            ax.set_xticks(range(len(QS)))
            ax.set_yticks(range(len(MS)))
            ax.set_xticklabels(QS if row == 1 else [])
            ax.set_yticklabels(MS if col == 0 else [])
            ax.tick_params(length=0, pad=2)
            for sp in ax.spines.values():
                sp.set_visible(False)
            if row == 0:
                ax.set_title(f"$o$ = {o}", pad=4, color=INK2)
            if col == 0:
                ax.set_ylabel(f"{mod}\n\n$M$", color=INK, labelpad=2)
            if row == 1:
                ax.set_xlabel("$q$")
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=BLUES), ax=axes,
                      fraction=0.022, pad=0.015)
    cb.set_label("$D_{sys}$, development  (lower is better)", color=INK2, fontsize=8)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=2, labelsize=7.4, color=AXIS)
    fig.suptitle("Unlinkability across the 80-setting design space; the ringed cell "
                 "is the sealed setting", fontsize=8.8, x=0.008, ha="left", y=0.985)
    return save(fig, out, "Figure3_DesignSpace")


# ================================================== Figure 4: the frontier
def pareto(rows):
    out, best = [], float("inf")
    for r in sorted(rows, key=lambda r: (r["EER"], r["Dsys_DEV"])):
        if r["Dsys_DEV"] < best:
            out.append(r)
            best = r["Dsys_DEV"]
    return out


def figure_frontier(D, out):
    fig, ax = plt.subplots(figsize=(4.9, 3.6))
    style(ax)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        ok = eligible(D["sweep"][mod], mod, D["floor"])
        no = [r for r in D["sweep"][mod] if r not in ok]
        ax.scatter([r["EER"] * 100 for r in no], [r["Dsys_DEV"] for r in no], s=17,
                   c=DEEMPH, linewidths=0.6, edgecolors=SURFACE, zorder=2)
        ax.scatter([r["EER"] * 100 for r in ok], [r["Dsys_DEV"] for r in ok], s=24,
                   c=col, alpha=0.9, linewidths=0.7, edgecolors=SURFACE, zorder=3,
                   label=f"{mod}, meets floor ({len(ok)})")
        pf = pareto(D["sweep"][mod])
        ax.step([r["EER"] * 100 for r in pf], [r["Dsys_DEV"] for r in pf], where="post",
                color=col, linewidth=1.3, alpha=0.9, zorder=4)
        s = D["selected"][mod]
        ax.scatter([s["EER"] * 100], [s["Dsys_DEV"]], s=120, c=col, linewidths=1.8,
                   edgecolors=SURFACE, zorder=6)
    ax.scatter([], [], s=17, c=DEEMPH, linewidths=0.6, edgecolors=SURFACE,
               label="below floor")
    ax.plot([], [], color=MUTED, linewidth=1.3, label="frontier")
    ax.annotate("large mark = sealed setting", xy=(7.05, 0.015), fontsize=7.0,
                color=MUTED, ha="right", va="bottom")
    ax.set_xlabel("EER (%), development")
    ax.set_ylabel("$D_{sys}$, development  (lower is better)")
    ax.set_xlim(0, 7.2)
    ax.set_ylim(0, 0.40)
    ax.set_title("Recognition and unlinkability trade off across the design space",
                 loc="left", pad=7, fontsize=8.6)
    ax.legend(loc="upper right", handletextpad=0.5, borderpad=0.2)
    fig.tight_layout(pad=0.9)
    return save(fig, out, "Figure4_Frontier")


# ============================================ Figure 5: generalisation
def figure_generalisation(D, out):
    rows = [("face", "development (42)", None), ("face", "held-out (58)", "face/evaluation"),
            ("voice", "development (42)", None), ("voice", "held-out (58)", "voice/evaluation"),
            ("voice", "external, VCTK (110)", "voice/external")]
    dev = {m: {"EER": D["selected"][m]["EER"] * 100, "TAR": D["selected"][m]["TAR_DEV"] * 100,
               "FMR": D["selected"][m]["FMR_DEV"] * 100, "Dsys": D["selected"][m]["Dsys_DEV"]}
           for m in ("face", "voice")}
    panels = [("EER", "EER (%)", [(1.0, "1%"), (3.0, "3%")], (0, 11)),
              ("TAR", "TAR (%)", [], (65, 100)),
              ("FMR", "FMR (%)", [(0.1, "0.1%")], (0, 0.68)),
              ("Dsys", "$D_{sys}$", [], (0, 0.30))]
    fig, axes = plt.subplots(1, 4, figsize=(7.6, 3.05))
    ypos = list(range(len(rows)))[::-1]
    for ax, (key, xlabel, refs, xlim) in zip(axes, panels):
        style(ax, "x")
        for ref, lab in refs:
            ax.axvline(ref, color=MUTED, linewidth=0.7, zorder=1)
            ax.annotate(lab, xy=(ref, len(rows) - 0.38), fontsize=6.8, color=MUTED,
                        ha="center", va="bottom")
        for y, (mod, _, part) in zip(ypos, rows):
            col = VOICE if mod == "voice" else FACE
            if part:
                est, lo, hi = D["intervals"][part][key]
                ax.plot([lo, hi], [y, y], color=col, linewidth=1.8,
                        solid_capstyle="butt", zorder=3)
                for cap in (lo, hi):
                    ax.plot([cap, cap], [y - 0.16, y + 0.16], color=col, linewidth=1.8,
                            zorder=3)
                ax.scatter([est], [y], s=46, c=col, linewidths=1.5, edgecolors=SURFACE,
                           zorder=5)
            else:
                ax.scatter([dev[mod][key]], [y], s=34, facecolors=SURFACE,
                           linewidths=1.3, edgecolors=col, zorder=5)
        ax.set_yticks(ypos)
        ax.set_ylim(-0.6, len(rows) - 0.15)
        ax.set_xlim(*xlim)
        ax.set_xlabel(xlabel)
        ax.set_yticklabels([f"{m} · {lab}" for m, lab, _ in rows] if ax is axes[0] else [],
                           fontsize=7.4, color=INK2)
    handles = [Line2D([], [], color=FACE, linewidth=1.8, marker="o", markersize=5,
                      markeredgecolor=SURFACE, label="face"),
               Line2D([], [], color=VOICE, linewidth=1.8, marker="o", markersize=5,
                      markeredgecolor=SURFACE, label="voice"),
               Line2D([], [], color=MUTED, linewidth=0, marker="o", markersize=5,
                      markerfacecolor=SURFACE, markeredgecolor=MUTED,
                      label="development (no interval)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.005),
               handletextpad=0.4, columnspacing=1.8)
    fig.suptitle("Point estimates with 95% identity-cluster bootstrap intervals "
                 "(2,000 replicates)", fontsize=8.8, x=0.008, ha="left", y=0.99)
    fig.tight_layout(pad=1.0, rect=(0, 0.075, 1, 0.935))
    return save(fig, out, "Figure5_Generalisation")


# ============================================ Figure P: protection cost
def figure_protection_cost(D, out):
    order = [("voice", "development"), ("voice", "evaluation"), ("voice", "external"),
             ("face", "development"), ("face", "evaluation")]
    name = {"development": "development (42)", "evaluation": "held-out (58)",
            "external": "external, VCTK (110)"}
    parts = {(p["modality"], p["partition"]): p for p in D["baseline"]}
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    style(ax, "x")
    ypos = list(range(len(order)))[::-1]
    for y, key in zip(ypos, order):
        p = parts[key]
        ci = D["intervals"].get(f"{key[0]}/{key[1]}", {}).get("EER")
        yp, yb = y + 0.16, y - 0.16
        if ci:
            ax.plot([ci[1], ci[2]], [yp, yp], color=VOICE, linewidth=1.3, zorder=3)
            for e in ci[1:]:
                ax.plot([e, e], [yp - 0.1, yp + 0.1], color=VOICE, linewidth=1.3, zorder=3)
        ax.plot([p["protected_eer"]], [yp], "o", color=VOICE, markersize=5,
                markeredgecolor=SURFACE, zorder=5)
        bound = p["unprotected_upper_bound_95"]
        if p["baseline_resolved"]:
            ax.plot([p["unprotected_eer"]], [yb], "s", color=FACE, markersize=4.6,
                    markeredgecolor=SURFACE, zorder=5)
            ax.plot([p["unprotected_eer"], bound], [yb, yb], color=FACE, linewidth=1.0,
                    alpha=0.55, zorder=3)
            ax.plot([bound, bound], [yb - 0.1, yb + 0.1], color=FACE, linewidth=1.0,
                    alpha=0.55, zorder=3)
        else:
            ax.annotate("", xy=(0.02, yb), xytext=(bound, yb), zorder=4,
                        arrowprops=dict(arrowstyle="-|>", color=FACE, linewidth=1.0,
                                        shrinkA=0, shrinkB=0))
            ax.plot([bound], [yb], "|", color=FACE, markersize=7, markeredgewidth=1.2,
                    zorder=5)
        if not ci:
            verdict = "development: no interval"
        elif ci[1] > bound:
            verdict = f"+{p['cost_pp']:.2f} pp, firm"
        else:
            verdict = f"+{p['cost_pp']:.2f} pp, not resolvable"
        ax.text(16.4, y, verdict, fontsize=6.8, va="center", ha="right",
                color=INK if verdict.endswith("firm") else MUTED)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{m} · {name[p]}" for m, p in order], fontsize=7.4, color=INK2)
    ax.set_xlim(0, 16.5)
    ax.set_xticks([0, 2, 4, 6, 8, 10, 12])
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_xlabel("Equal error rate (%)")
    handles = [Line2D([], [], color=VOICE, marker="o", markersize=5, linewidth=1.3,
                      label="protected, 95% interval"),
               Line2D([], [], color=FACE, marker="s", markersize=4.6, linewidth=1.0,
                      alpha=0.7, label="unprotected, measured"),
               Line2D([], [], color=FACE, marker="|", linestyle="none", markersize=7,
                      markeredgewidth=1.2, label="unprotected, 95% upper bound only")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=3,
              handlelength=1.5, columnspacing=1.2, fontsize=6.8)
    ax.set_title("What protection costs against the unprotected system",
                 loc="left", pad=7)
    fig.tight_layout(pad=0.9)
    return save(fig, out, "FigureP_ProtectionCost")


# =============================================== Figure 6: the ablation
def dot_interval(ax, values, xmax, pct=False):
    for i, arm in enumerate(ARMS):
        point, lo, hi = values[arm]
        y = len(ARMS) - 1 - i
        ax.plot([lo, hi], [y, y], color=HUE[arm], lw=2.0, solid_capstyle="round", zorder=3)
        ax.plot([point], [y], marker=SHAPE[arm], ms=7.5, color=HUE[arm], mec=SURFACE,
                mew=1.4, zorder=4)
        ax.annotate(f"{point:.2f}%" if pct else f"{point:.3f}", (hi, y), xytext=(5, 0),
                    textcoords="offset points", fontsize=7.2, color=INK2, va="center")
    ax.set_yticks(range(len(ARMS)))
    ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)])
    ax.set_ylim(-0.6, len(ARMS) - 0.4)
    ax.grid(True, axis="x")
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0, pad=3)
    ax.set_xlim(0, xmax)


def figure_ablation(D, out):
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 3.6),
                             gridspec_kw={"wspace": 0.78, "hspace": 0.70})
    for r, mod in enumerate(("voice", "face")):
        arms = D["ablation"][mod]["arms"]
        eer = {a: (arms[a]["EER"] * 100, arms[a]["EER_CI"][0] * 100,
                   arms[a]["EER_CI"][1] * 100) for a in ARMS}
        dsys = {a: (arms[a]["Dsys"], *arms[a]["Dsys_CI"]) for a in ARMS}
        dot_interval(axes[r, 0], eer, xmax=11.5 if mod == "face" else 3.6, pct=True)
        dot_interval(axes[r, 1], dsys, xmax=0.33)
        axes[r, 0].set_title(f"{mod} — equal error rate (%)", loc="left", pad=6)
        axes[r, 1].set_title(f"{mod} — $D_{{sys}}$", loc="left", pad=6)
        for c in (0, 1):
            axes[r, c].set_xlabel("lower is better")
    v = {a: D["ablation"]["voice"]["arms"][a]["EER"] * 100 for a in ARMS}
    ax = axes[0, 0]
    for x0, x1, yy, col, txt in (
            (v["iom_only"], v["randproj_iom"], -0.50, MUTED,
             f"+{v['randproj_iom'] - v['iom_only']:.2f} pp  from the size reduction"),
            (v["randproj_iom"], v["polyiom"], -1.05, HUE["polyiom"],
             f"+{v['polyiom'] - v['randproj_iom']:.2f} pp  from the polynomial itself")):
        ax.annotate("", xy=(x0, yy), xytext=(x1, yy),
                    arrowprops=dict(arrowstyle="|-|,widthA=0.22,widthB=0.22", color=col,
                                    lw=0.9))
        ax.annotate(txt, (x0, yy - 0.13), ha="left", va="top", fontsize=6.4, color=col)
    ax.set_ylim(-1.80, len(ARMS) - 0.4)
    fig.suptitle("What each stage costs: error rate and linkability of the three arms",
                 fontsize=9, x=0.008, ha="left", y=0.995)
    return save(fig, out, "Figure6_Ablation")


# ============================================ Figure 7: revocation per key
def figure_revocation(D, out):
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.45), gridspec_kw={"wspace": 0.30})
    rng = np.random.default_rng(D["revocation"]["master_seed"])
    n_keys = D["revocation"]["n_fresh_keys"]
    for c, mod in enumerate(("voice", "face")):
        m = D["revocation"]["modalities"][mod]
        ax = axes[c]
        counts = []
        for i, arm in enumerate(ARMS):
            vals = np.asarray(m["arms"][arm]["per_key_PRAR"], float) * 100
            _check(vals.size == n_keys, f"{mod}/{arm}: {vals.size} keys, expected {n_keys}")
            y = len(ARMS) - 1 - i
            ax.scatter(vals, y + rng.uniform(-0.17, 0.17, vals.size), s=15,
                       marker=SHAPE[arm], facecolor=HUE[arm], edgecolor=SURFACE,
                       linewidth=0.5, alpha=0.85, zorder=3)
            counts.append(int((vals >= 50).sum()))
        ax.axvline(50, color=CRITICAL, lw=0.9, ls=(0, (3, 2)), zorder=2)
        ax.annotate("half or more exposed", (50, len(ARMS) - 1.5), fontsize=6.4,
                    color=CRITICAL, ha="center", va="center", rotation=90,
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
        ax.set_yticks(range(len(ARMS)))
        ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)] if c == 0 else [])
        ax.set_ylim(-0.55, len(ARMS) - 0.45)
        ax.set_xlim(-6, 106)
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.grid(True, axis="x")
        ax.set_axisbelow(True)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.tick_params(length=0, pad=3)
        ax.set_title(rf"{mod}  (M = {m['M']}, $\tau^*$ = {m['tau']})", loc="left", pad=6)
        ax.set_xlabel("acceptance rate after re-keying (%)")
        right = ax.twinx()
        right.set_ylim(ax.get_ylim())
        right.set_yticks(range(len(ARMS)))
        right.set_yticklabels([f"{n} of {n_keys}" for n in reversed(counts)])
        right.tick_params(length=0, pad=4, labelsize=6.8)
        for s in ("top", "right", "left", "bottom"):
            right.spines[s].set_visible(False)
        for tick, n in zip(right.get_yticklabels(), reversed(counts)):
            tick.set_color(CRITICAL if n else MUTED)
    fig.suptitle("Each dot is one fresh key. Most PolyIoM keys revoke completely; a "
                 "minority do not", fontsize=9, x=0.008, ha="left", y=1.02)
    return save(fig, out, "Figure7_RevocationPerKey")


# ================================================ Figure 8: the inversion
def figure_inversion(D, out):
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 3.6),
                             gridspec_kw={"wspace": 0.78, "hspace": 0.70})
    for r, mod in enumerate(("voice", "face")):
        inv = D["inversion"][mod]
        M, tau = inv["M"], inv["tau"]
        chance = inv["arms"]["polyiom"]["chance_collisions"]
        cos_chance = inv["arms"]["polyiom"]["cos_chance"]
        ax = axes[r, 0]
        for i, arm in enumerate(ARMS):
            v = inv["arms"][arm]["mean_collisions"]
            y = len(ARMS) - 1 - i
            ax.plot([0, v], [y, y], color=HUE[arm], lw=2.0, solid_capstyle="butt", zorder=3)
            ax.plot([v], [y], marker=SHAPE[arm], ms=7.5, color=HUE[arm], mec=SURFACE,
                    mew=1.4, zorder=4)
            ax.annotate(f"{v:.1f}", (v, y), xytext=(5, 0), textcoords="offset points",
                        fontsize=7.2, color=INK2, va="center")
        ax.axvline(tau, color=CRITICAL, lw=0.9, ls=(0, (3, 2)), zorder=2)
        ax.annotate(rf"$\tau^*$ = {tau}", (tau + 2, -0.50), fontsize=6.6, color=CRITICAL,
                    ha="left", va="center",
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.2))
        ax.axvline(chance, color=MUTED, lw=0.8, zorder=2)
        ax.set_yticks(range(len(ARMS)))
        ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)])
        ax.set_ylim(-0.75, len(ARMS) - 0.35)
        ax.set_xlim(0, M * 1.14)
        ax.grid(True, axis="x")
        ax.set_axisbelow(True)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.tick_params(length=0, pad=3)
        ax.set_title(f"{mod} — matching positions reached (of {M})", loc="left", pad=6)
        ax.set_xlabel(f"further right = more complete attack (chance {chance:.0f})")

        # The cosine intervals are stored with the revocation run, which
        # re-ran the same attack; the point estimates must agree.
        cos = {}
        for arm in ARMS:
            ra = D["revocation"]["modalities"][mod]["arms"][arm]
            _check(close(ra["cos_to_true"], inv["arms"][arm]["cos_to_true"], 1e-9),
                  f"{mod}/{arm}: inversion and revocation cosines differ")
            cos[arm] = (ra["cos_to_true"], *ra["cos_CI"])
        ax = axes[r, 1]
        dot_interval(ax, cos, xmax=1.06)
        ax.axvline(cos_chance, color=MUTED, lw=0.8, zorder=2)
        ax.annotate(f"chance {cos_chance:.3f}", (cos_chance, -0.50), fontsize=6.6,
                    color=MUTED, ha="left", va="center",
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.2))
        ax.set_ylim(-0.75, len(ARMS) - 0.35)
        ax.set_title(f"{mod} — cosine to the true embedding", loc="left", pad=6)
        ax.set_xlabel("higher = more of the biometric recovered")
    fig.suptitle("The attack clears the threshold in every arm; only the recovered "
                 "embedding separates them", fontsize=9, x=0.008, ha="left", y=0.995)
    return save(fig, out, "Figure8_Inversion")



FIGURES = (figure_selection, figure_design_space, figure_frontier,
           figure_generalisation, figure_protection_cost, figure_ablation,
           figure_revocation, figure_inversion)


def draw_all(D, out):
    """Draw the eight results figures into `out`; return their file stems.

    D["sweep"]       {modality: [row, ...]}, 80 rows each, with keys
                     M, q, o, EER, TAR_DEV, FMR_DEV, Dsys_DEV (fractions)
    D["floor"]       {modality: (max EER, min TAR)} of the selection rule
    D["selected"]    {modality: the sweep row that was sealed}
    D["intervals"]   {"voice/evaluation" | "face/evaluation" |
                      "voice/external": {"EER" | "TAR" | "FMR": (point,
                      low, high) in percent, "Dsys": (point, low, high)}}
    D["baseline"]    [{modality, partition, protected_eer, unprotected_eer,
                      cost_pp, baseline_resolved,
                      unprotected_upper_bound_95}, ...] in percent
    D["ablation"]    ablation_result["modalities"]
    D["revocation"]  revocation_result (the whole record)
    D["inversion"]   inversion_result["modalities"]
    """
    stems = []
    for draw in FIGURES:
        # Start from matplotlib's defaults every time, so a style set by
        # other plotting code in the same session cannot leak in.
        with mpl.rc_context():
            mpl.rcdefaults()
            mpl.rcParams.update(STYLE)
            stems.append(draw(D, out))
    return stems


In [ ]:
%%writefile {RUNTIME}/allfigures_only.py
# 8. Sweep, intervals, baseline and figure driver
"""Regenerate every results figure from code, for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_ALL_FIGURES.ipynb after the study's
own runtime (ablation_only.py, scoredump_only.py, inversion_only.py and
revocation_only.py). Those files compute the ablation, the score
histograms, the inversion attack and the revocation test exactly as the
study did. This file adds the three computations they do not cover, and
the checks:

  sweep        the 80-setting development sweep behind Figures 3, 4, S2
  intervals    the identity-cluster bootstrap intervals behind Figure 5
  baseline     protected against unprotected EER, behind Figure P

Inputs are read from RUNS_IN (the study's runs/ folder) and every output
is written to DIR["runs"], which the notebook points at a separate folder.
Nothing in runs/ or seal/ is ever written.

After each stage the regenerated result is compared with the study's
original output, where that output is on Drive, and the differences are
printed. A figure is therefore only drawn from numbers this notebook has
computed and then checked.
"""

# ------------------------------------------------------------ input paths
# The study's runtime reads its inputs through DIR["runs"]. The notebook
# redirects DIR["runs"] to a fresh output folder, so the two readers below
# are redefined to keep reading the sealed inputs from RUNS_IN.

def selected_poly_key(modality):
    path = RUNS_IN / "key_search" / modality / "selected_key.json"
    obj = json.loads(path.read_text())
    return np.asarray(obj["C"], np.float32), np.asarray(obj["E"], np.int64)


def load_iom_tensor(modality, overlap, d):
    k = 1 + math.ceil((d - G) / (G - overlap))
    path = RUNS_IN / "iom_tensors" / f"{modality}_o{overlap}_k{k}.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Frozen IoM tensor is missing: {path}")
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        tensor = torch.load(path, map_location="cpu")
    if (tensor.ndim != 3 or tensor.shape[0] < 256 or tensor.shape[1] < 32
            or tensor.shape[-1] != k):
        raise RuntimeError(f"Frozen IoM tensor has an invalid shape: {path}")
    return tensor


VCTK_EMB = DIR["embeddings"] / "vctk_external_embeddings.npz"


def vctk_external_data():
    """External VCTK identities, built as the sealed external evaluation
    built them: ranks 0-4 averaged for enrolment, ranks 5-14 as probes."""
    z = np.load(VCTK_EMB, allow_pickle=False)
    E = z["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": z["speaker_id"].astype(str),
        "rank": z["rank"].astype(int),
        "embedding_index": z["embedding_index"].astype(int),
    })
    subjects, enroll, probes = [], {}, {}
    for uid, g in table.groupby("speaker_id"):
        g = g.sort_values("rank")
        if len(g) != 15:
            continue
        uid = str(uid)
        members = np.stack([E[int(r.embedding_index)]
                            for r in g[g["rank"] < 5].itertuples(index=False)])
        subjects.append(uid)
        enroll[uid] = l2_np(np.mean(members, axis=0))
        probes[uid] = [E[int(r.embedding_index)]
                       for r in g[g["rank"] >= 5].itertuples(index=False)]
    return sorted(subjects), enroll, probes


# ------------------------------------------- weighted metrics (unchanged)
# From external_confidence_cells.py, the study's identity_cluster_percentile_v1
# method. A genuine trial of identity i carries weight w_i; an ordered
# impostor pair (a, b) carries w_a * w_b; same-identity pairs never form.

def _tar_w(gen_hist, tau):
    total = gen_hist.sum()
    return float(gen_hist[tau:].sum() / total) if total > 0 else float("nan")


def _fmr_w(imp_hist, tau):
    total = imp_hist.sum()
    return float(imp_hist[tau:].sum() / total) if total > 0 else float("nan")


def _eer_w(gen_hist, imp_hist, M):
    g_total, i_total = gen_hist.sum(), imp_hist.sum()
    if g_total <= 0 or i_total <= 0:
        return float("nan"), -1
    imp_tail = np.concatenate([imp_hist[::-1].cumsum()[::-1], [0.0]])
    gen_head = np.concatenate([[0.0], gen_hist.cumsum()])
    fmr = imp_tail / i_total
    fnmr = gen_head / g_total
    diff = fmr - fnmr
    cross = int(np.argmax(diff <= 0))
    if diff[cross] > 0:
        return float(fmr[-1]), M
    if cross == 0:
        return float((fmr[0] + fnmr[0]) / 2.0), 0
    d1, d2 = diff[cross - 1], diff[cross]
    t = 0.0 if d1 == d2 else float(d1 / (d1 - d2))
    return float(fmr[cross - 1] + t * (fmr[cross] - fmr[cross - 1])), cross


def _dsys_w(mated_hist, nonmated_hist, M):
    m_total, n_total = mated_hist.sum(), nonmated_hist.sum()
    if m_total <= 0 or n_total <= 0:
        return float("nan")
    p_m = mated_hist / m_total
    p_n = nonmated_hist / n_total
    denom = p_m + p_n
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.where(denom > 0, 2.0 * p_m / denom - 1.0, 0.0)
    return float(np.sum(p_m * np.clip(d, 0.0, 1.0)))


def _apply_weights(H, w):
    g = w @ H["Ghist"]
    i = w @ np.tensordot(w, H["Ihist"], axes=(0, 0))
    m = w @ H["Mhist"]
    nm = w @ np.tensordot(w, H["Nhist"], axes=(0, 0))
    return g, i, m, nm


def _metrics(H, w):
    g, i, m, nm = _apply_weights(H, w)
    eer, c_eer = _eer_w(g, i, H["M"])
    return {"EER": eer, "c_EER": c_eer,
            "TAR": _tar_w(g, H["tau"]), "FMR": _fmr_w(i, H["tau"]),
            "Dsys": _dsys_w(m, nm, H["M"])}


# ------------------------------------- per-identity score histograms
def unlink_tensor(modality, index, M, q, k, variant="shape"):
    """One of the two independently seeded unlinkability keys.

    "shape" draws exactly (M, q, k), as the held-out evaluation and the
    ablation do. "slice" draws the largest grid (256, 32, k) and slices it;
    the sweep stage tries it only if "shape" fails to reproduce a row.
    """
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed_unlink(modality, index))
    if variant == "shape":
        return torch.randn((M, q, k), generator=generator, dtype=torch.float32)
    full = torch.randn((max(GRID["M"]), max(GRID["q"]), k),
                       generator=generator, dtype=torch.float32)
    return full[:M, :q].contiguous()


def identity_histograms(modality, subjects, enroll, probes, M, q, o, tau,
                        frozen=None, variant="shape"):
    """Genuine, impostor, mated and non-mated score histograms per identity,
    from the sealed pipeline (polynomial K*, frozen IoM tensor R)."""
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - o))
    if frozen is None:
        frozen = load_iom_tensor(modality, o, d)
    recognition = frozen[:M, :q].to(DEVICE, torch.float32)

    enrolment = [enroll[u] for u in subjects]
    probe_vectors, owners = [], []
    for index, u in enumerate(subjects):
        probe_vectors.extend(probes[u])
        owners.extend([index] * len(probes[u]))
    owners = np.asarray(owners)
    enrolled = hash_arm(enrolment, "polyiom", key, o, recognition, M, q, None)
    hashed = hash_arm(probe_vectors, "polyiom", key, o, recognition, M, q, None)
    collision = (hashed[:, None, :] == enrolled[None, :, :]).sum(axis=-1)

    bins = M + 1
    Ghist = np.zeros((n, bins))
    Ihist = np.zeros((n, n, bins))           # [claimed, true, score]
    rows = np.arange(len(owners))
    np.add.at(Ghist, (owners, collision[rows, owners]), 1.0)
    for claimed in range(n):
        other = owners != claimed
        np.add.at(Ihist, (claimed, owners[other], collision[other, claimed]), 1.0)

    first = hash_arm(enrolment, "polyiom", key, o,
                     unlink_tensor(modality, 1, M, q, k, variant).to(DEVICE),
                     M, q, None)
    second = hash_arm(enrolment, "polyiom", key, o,
                      unlink_tensor(modality, 2, M, q, k, variant).to(DEVICE),
                      M, q, None)
    cross = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
    Mhist = np.zeros((n, bins))
    Nhist = np.zeros((n, n, bins))
    Mhist[np.arange(n), np.diag(cross)] = 1.0
    for a in range(n):
        for b in range(n):
            if a != b:
                Nhist[a, b, cross[a, b]] = 1.0
    return {"subjects": subjects, "n": n, "M": M, "q": q, "o": o, "tau": tau,
            "Ghist": Ghist, "Ihist": Ihist, "Mhist": Mhist, "Nhist": Nhist}


# ----------------------------------------------------------------- sweep
def _config():
    path = DIR["seal"] / "config_v1_1_4.json"
    if path.is_file():
        return json.loads(path.read_text())
    print("  seal/config_v1_1_4.json not found; using the documented grid")
    return {"M_grid": [32, 64, 128, 256], "q_grid": [4, 8, 16, 32],
            "overlaps": [0, 1, 2, 3, 4], "target_fmr": 0.001}


CONFIG = _config()
GRID = {"M": CONFIG["M_grid"], "q": CONFIG["q_grid"], "o": CONFIG["overlaps"]}
SWEEP_FIELDS = ("EER", "c_EER", "c_tau_DEV", "TAR_DEV", "FMR_DEV",
                "FNMR_DEV", "Dsys_DEV")


def dev_threshold(imp_hist, target):
    """The lowest score at which the development FMR meets the target."""
    total = imp_hist.sum()
    for c in range(len(imp_hist) + 1):
        if imp_hist[c:].sum() / total <= target + 1e-12:
            return c
    return len(imp_hist)


def sweep_row(modality, H):
    ones = np.ones(H["n"])
    g, i, m, nm = _apply_weights(H, ones)
    eer, c_eer = _eer_w(g, i, H["M"])
    tau = dev_threshold(i, CONFIG["target_fmr"])
    tar = _tar_w(g, tau)
    return {"modality": modality, "M": H["M"], "q": H["q"], "o": H["o"],
            "n_subjects": H["n"], "n_genuine": int(g.sum()),
            "n_impostor": int(i.sum()), "c_tau_DEV": tau, "EER": eer,
            "c_EER": c_eer, "TAR_DEV": tar, "FMR_DEV": _fmr_w(i, tau),
            "FNMR_DEV": 1.0 - tar, "Dsys_DEV": _dsys_w(m, nm, H["M"])}


def _row_matches(new, old, tol=1e-9):
    return all(abs(float(new[f]) - float(old[f])) <= tol for f in SWEEP_FIELDS)


def run_sweep(modality):
    """Recompute the 80-setting development sweep and compare it, row by
    row, with the sealed runs/sweep/<modality>/sweep_80.csv."""
    out = DIR["runs"] / "sweep" / modality / "sweep_80.csv"
    if out.is_file():
        print(f"  {modality}: resume - regenerated sweep found")
        rows = pd.read_csv(out).to_dict("records")
    else:
        subjects, enroll, probes = partition_data(modality, "development")
        d = len(enroll[subjects[0]])
        sealed_path = RUNS_IN / "sweep" / modality / "sweep_80.csv"
        sealed = ({(r["M"], r["q"], r["o"]): r for r in
                   pd.read_csv(sealed_path).to_dict("records")}
                  if sealed_path.is_file() else {})
        rows = []
        for o in GRID["o"]:
            frozen = load_iom_tensor(modality, o, d)
            for M in GRID["M"]:
                for q in GRID["q"]:
                    H = identity_histograms(modality, subjects, enroll, probes,
                                            M, q, o, None, frozen=frozen)
                    row = sweep_row(modality, H)
                    old = sealed.get((M, q, o))
                    if old is not None and not _row_matches(row, old):
                        H2 = identity_histograms(modality, subjects, enroll,
                                                 probes, M, q, o, None,
                                                 frozen=frozen, variant="slice")
                        row2 = sweep_row(modality, H2)
                        if _row_matches(row2, old):
                            row = row2
                    rows.append(row)
            print(f"    {modality} o={o}: {len(rows)} of "
                  f"{len(GRID['o']) * len(GRID['M']) * len(GRID['q'])} settings")
        out.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out, index=False)
    compare_sweep(modality, rows)
    return rows


def compare_sweep(modality, rows):
    path = RUNS_IN / "sweep" / modality / "sweep_80.csv"
    if not path.is_file():
        print(f"  {modality}: no sealed sweep on Drive to compare against")
        return None
    sealed = {(r["M"], r["q"], r["o"]): r
              for r in pd.read_csv(path).to_dict("records")}
    same = [r for r in rows if (r["M"], r["q"], r["o"]) in sealed
            and _row_matches(r, sealed[(r["M"], r["q"], r["o"])])]
    print(f"  {modality}: {len(same)} of {len(rows)} regenerated settings equal "
          f"the sealed sweep on {', '.join(SWEEP_FIELDS)}")
    for r in rows:
        old = sealed.get((r["M"], r["q"], r["o"]))
        if old is not None and not _row_matches(r, old):
            diffs = {f: (round(float(r[f]), 6), round(float(old[f]), 6))
                     for f in SWEEP_FIELDS
                     if abs(float(r[f]) - float(old[f])) > 1e-9}
            print(f"    differs M={r['M']} q={r['q']} o={r['o']}: {diffs}")
    return len(same) == len(rows)


def floor_of(modality):
    path = DIR["seal"] / f"operating_point_{modality}.json"
    if path.is_file():
        rule = json.loads(path.read_text())["rule"]
        return float(rule["eer_max"]), float(rule["tar_min"])
    raise FileNotFoundError(f"sealed operating point not found: {path}")


def select(modality, rows, floor):
    """The study's rule: lowest Dsys among settings meeting the floor; ties
    by lower EER, then smaller M, q, o."""
    eer_max, tar_min = floor
    ok = [r for r in rows if r["EER"] <= eer_max and r["TAR_DEV"] >= tar_min]
    if not ok:
        raise RuntimeError(f"{modality}: no setting meets the floor {floor}")
    best = min(ok, key=lambda r: (r["Dsys_DEV"], r["EER"], r["M"], r["q"], r["o"]))
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    chosen = (best["M"], best["q"], best["o"])
    expected = (sealed["M"], sealed["q"], sealed["o"])
    tau_ok = int(best["c_tau_DEV"]) == int(sealed["c_tau_carried_from_DEV"])
    print(f"  {modality}: {len(ok)} settings meet the floor; the rule selects "
          f"M={chosen[0]} q={chosen[1]} o={chosen[2]} with tau={best['c_tau_DEV']}"
          f" - {'as sealed' if chosen == expected and tau_ok else '*** NOT the sealed setting'}")
    if chosen != expected or not tau_ok:
        raise RuntimeError(f"{modality}: regenerated selection {chosen} "
                           f"differs from the sealed {expected}")
    return best


# ------------------------------------------------------------- intervals
def heldout_seed(modality):
    """The study's held-out interval seed. Reproduces the seeds recorded in
    RESULTS_SUMMARY.md: face 6353163838588607505, voice 4722292304065993655."""
    return H(S0, "HELDOUT-CONFIDENCE", modality)


def external_seed(modality):
    label = f"{S0}|external_confidence|{modality}".encode()
    return int.from_bytes(hashlib.sha256(label).digest()[:8], "big") >> 1


def _original_interval_file(partition):
    path = (RUNS_IN / "heldout" / "heldout_confidence.json"
            if partition == "evaluation"
            else RUNS_IN / "external" / "voice" / "external_confidence.json")
    return json.loads(path.read_text()) if path.is_file() else None


def _seed_for(modality, partition):
    original = _original_interval_file(partition)
    if original:
        recorded = original.get("modalities", {}).get(modality, {}) \
            .get("bootstrap_seed")
        if recorded is not None:
            return int(recorded)
    return (heldout_seed(modality) if partition == "evaluation"
            else external_seed(modality))


def interval_for(modality, partition, n_bootstrap, confidence):
    if partition == "evaluation":
        sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
        tag = "HOLDOUT"
        subjects, enroll, probes = (face_heldout_data() if modality == "face"
                                    else voice_heldout_data())
    else:
        sealed = json.loads((RUNS_IN / "external" / modality /
                             "external_result.json").read_text())
        tag = "EXT"
        subjects, enroll, probes = vctk_external_data()
    M, q, o = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    H = identity_histograms(modality, subjects, enroll, probes, M, q, o, tau)

    point = _metrics(H, np.ones(H["n"]))
    sealed_point = {"EER": sealed[f"EER_{tag}"],
                    "TAR": sealed[f"TAR_{tag}_at_dev_threshold"],
                    "FMR": sealed[f"FMR_{tag}_at_dev_threshold"],
                    "Dsys": sealed[f"Dsys_{tag}"]}
    drift = {k: (point[k], v) for k, v in sealed_point.items()
             if abs(point[k] - v) > 1e-6}
    if drift:
        raise RuntimeError(f"{modality}/{partition}: rebuilt scores do not "
                           f"reproduce the sealed result: {drift}")

    seed = _seed_for(modality, partition)
    rng = np.random.default_rng(seed)
    keys = ("EER", "TAR", "FMR", "Dsys")
    draws = {k: np.empty(n_bootstrap) for k in keys}
    for b in range(n_bootstrap):
        w = rng.multinomial(H["n"], np.full(H["n"], 1.0 / H["n"])).astype(float)
        mb = _metrics(H, w)
        for k in keys:
            draws[k][b] = mb[k]
    alpha = (1.0 - confidence) / 2.0
    out = {}
    for k in keys:
        d = draws[k][np.isfinite(draws[k])]
        out[k] = {"estimate": float(point[k]),
                  "lower": float(np.percentile(d, 100 * alpha)),
                  "upper": float(np.percentile(d, 100 * (1 - alpha)))}
    print(f"  {modality}/{partition}: {H['n']} identities, seed {seed}; "
          f"point estimates reproduce the sealed result")
    return {"metrics": out, "bootstrap_seed": seed, "n_subjects": H["n"]}


def run_intervals(n_bootstrap=2000, confidence=0.95):
    path = DIR["runs"] / "intervals" / "confidence_intervals.json"
    if path.is_file():
        print("Resume: regenerated intervals found; returning them unchanged.")
        return json.loads(path.read_text())
    result = {"schema": 1, "method_id": "identity_cluster_percentile_v1",
              "n_bootstrap": n_bootstrap, "confidence": confidence,
              "partitions": {}}
    parts = [("voice", "evaluation"), ("face", "evaluation")]
    if VCTK_EMB.is_file() and (RUNS_IN / "external" / "voice" /
                               "external_result.json").is_file():
        parts.append(("voice", "external"))
    else:
        print("  voice/external: VCTK embeddings or sealed external result "
              "not found; skipped")
    for modality, partition in parts:
        result["partitions"][f"{modality}/{partition}"] = interval_for(
            modality, partition, n_bootstrap, confidence)
    atomic_json(path, result)
    compare_intervals(result)
    return result


def compare_intervals(result):
    names = {"EER": "EER_{}", "TAR": "TAR_{}_at_dev_threshold",
             "FMR": "FMR_{}_at_dev_threshold", "Dsys": "Dsys_{}"}
    for part, rec in result["partitions"].items():
        modality, partition = part.split("/")
        original = _original_interval_file(partition)
        if not original:
            print(f"  {part}: no original interval file on Drive to compare")
            continue
        tag = "HOLDOUT" if partition == "evaluation" else "EXT"
        old = original["modalities"][modality]["metrics"]
        worst = max(abs(rec["metrics"][k][side] - old[names[k].format(tag)][side])
                    for k in names for side in ("estimate", "lower", "upper"))
        print(f"  {part}: largest difference from the original intervals "
              f"{worst:.2e}" + ("  (identical)" if worst <= 1e-12 else ""))


def intervals_for_plots(result):
    """Percent for EER, TAR, FMR; fraction for Dsys; as (point, low, high)."""
    out = {}
    for part, rec in result["partitions"].items():
        m = rec["metrics"]
        out[part] = {k: tuple(m[k][s] * (1 if k == "Dsys" else 100)
                              for s in ("estimate", "lower", "upper"))
                     for k in ("EER", "TAR", "FMR", "Dsys")}
    return out


# -------------------------------------------------------------- baseline
def clopper_pearson_upper(k, n, confidence=0.95):
    from scipy.stats import beta
    return float(beta.ppf(confidence, k + 1, n - k))


def baseline_from_scores(scores):
    """Protected against unprotected EER, from the score histograms.

    Where the unprotected system makes fewer than one genuine error, its
    EER is not resolvable and only a one-sided 95% upper bound is given.
    """
    rows = []
    for modality in ("voice", "face"):
        entry = scores["modalities"][modality]
        for partition in ("development", "evaluation", "external"):
            h = entry["partitions"].get(partition)
            if h is None:
                continue
            protected = eer_check(h["genuine"], h["impostor"], entry["M"])
            raw = h["unprotected"]["EER"]
            n = h["n_genuine"]
            k = int(math.floor(raw * n + 1e-9))
            rows.append({
                "modality": modality, "partition": partition,
                "n_genuine": n,
                "protected_eer": protected * 100,
                "unprotected_eer": raw * 100,
                "cost_pp": (protected - raw) * 100,
                "one_genuine_error_pct": 100.0 / n,
                "baseline_resolved": bool(raw >= 1.0 / n),
                "unprotected_upper_bound_95": clopper_pearson_upper(k, n) * 100,
            })
    return rows


def add_external_scores(scores):
    """The study's score dump adds VCTK only through a helper file kept on
    Drive. If it did not, build the external partition here, the same way."""
    voice = scores["modalities"]["voice"]
    if "external" in voice["partitions"] or not VCTK_EMB.is_file():
        return scores
    subjects, enroll, probes = vctk_external_data()
    M, q, o = voice["M"], voice["q"], voice["o"]
    d = len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - o))
    recognition = load_iom_tensor("voice", o, d)[:M, :q].to(DEVICE, torch.float32)
    pair = [unlink_tensor("voice", i, M, q, k).to(DEVICE) for i in (1, 2)]
    h = histograms_for("voice", subjects, enroll, probes, M, q, o,
                       selected_poly_key("voice"), recognition, pair)
    h["dlink_curve"], h["dlink_global"] = unlinkability_curve(h["mated"],
                                                              h["nonmated"])
    h["unprotected"] = cosine_histograms(subjects, enroll, probes)
    h["unprotected"]["EER"] = eer_cosine(h["unprotected"]["genuine"],
                                         h["unprotected"]["impostor"])
    voice["partitions"]["external"] = h
    atomic_json(DIR["runs"] / "scores" / "score_histograms.json", scores)
    print(f"  external: {h['n_subjects']} speakers added to the score dump")
    return scores


# ------------------------------------------------------------ comparison
def _leaves(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from _leaves(v, f"{prefix}/{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _leaves(v, f"{prefix}[{i}]")
    else:
        yield prefix, obj


def compare_with_original(name, regenerated, relative):
    """Every number in a regenerated result against the study's original."""
    path = RUNS_IN / relative
    if not path.is_file():
        print(f"  {name:<22} no original on Drive to compare against")
        return None
    old = dict(_leaves(json.loads(path.read_text())))
    new = dict(_leaves(regenerated))
    numeric = [k for k in old if isinstance(old[k], (int, float))
               and not isinstance(old[k], bool) and k in new]
    worst, where = 0.0, None
    for k in numeric:
        diff = abs(float(new[k]) - float(old[k]))
        if diff > worst:
            worst, where = diff, k
    missing = [k for k in old if k not in new]
    verdict = "identical" if worst == 0.0 and not missing else (
        f"largest difference {worst:.2e} at {where}")
    print(f"  {name:<22} {len(numeric):6d} numbers compared: {verdict}"
          + (f"; {len(missing)} fields absent" if missing else ""))
    return worst


# ---------------------------------------------------------------- figures
def draw_figures(out, sweeps, selected, floors, intervals, baseline,
                 ablation, inversion, revocation, score_json):
    data = {"sweep": sweeps, "floor": floors, "selected": selected,
            "intervals": intervals_for_plots(intervals), "baseline": baseline,
            "ablation": ablation["modalities"], "revocation": revocation,
            "inversion": inversion["modalities"]}
    stems = results_plots.draw_all(data, out)

    # Figures A-D: the study's own figure code, given the intervals this
    # notebook computed instead of the ones typed into it.
    figures_only.CARRIED_CI = {
        tuple(part.split("/")): data["intervals"][part]["EER"]
        for part in data["intervals"]}
    figures_only.make_figures(score_json, out)
    return stems + ["figA_det_preservation", "figB_score_separability",
                    "figC_unlinkability", "figD_revocability"]


print("All-figures runtime ready.")


In [ ]:
%%writefile {RUNTIME}/vctk_prepare_cells.py
# 9. The study's VCTK preparation
"""
VCTK external corpus preparation - works from the archive already in Drive.

VCTK-Corpus-0.92.zip (11,747,302,977 bytes) finished downloading on Sep 17
into datasets/VCTK-Corpus-0.92/. Nothing else has happened to it: the
folder holds only the zip, so there is no manifest and no 16 kHz audio for
extract_vctk_embeddings() to read. vctk_download_retry.py has done its job
and is no longer needed.

Load after the v1.1.4 notebook cells and external_eval_cells.py:

    exec(open("/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4/"
              "vctk_prepare_cells.py").read())
    prepare_vctk()

Then the existing external path runs unchanged:

    ensure_runtime_deps()        # ECAPA needs speechbrain present
    evaluate_external("voice")

WHAT IT DOES, AND WHY THIS WAY
------------------------------
* Streams the members it needs straight out of the zip. Extracting the
  whole archive would write 11.7 GB to a Colab disk that does not reliably
  have it, to use a few hundred MB of it. Only the utterances that end up
  in the manifest are ever decoded or written.
* mic1 only, as config_v1_1_4.json records
  ("vctk_validity": "mic1_post_resample_samples>=24000"). mic2 is a
  different microphone and p280's mic2 is missing from the release.
* Resamples 48 kHz -> 16 kHz with an exact 1/3 polyphase decimation
  (scipy.signal.resample_poly), which is anti-aliased and deterministic,
  then applies the >= 24000 sample gate AFTER resampling, in that order,
  because that is what the sealed validity rule says.
* Takes the first 15 valid utterances per speaker in sorted filename
  order, and drops any speaker that cannot supply 15. Fifteen is what the
  voice protocol requires: vctk_external_data() skips any speaker without
  exactly 15, enrolling ranks 0-4 and probing with ranks 5-14. Sorted
  order rather than a random sample keeps this reproducible without
  spending a seed, and VCTK filename order is not correlated with
  anything the evaluation measures.
* Resumable and idempotent: a verified stage returns immediately, and an
  already-written wav of the right length is not decoded again.

VCTK is external. It is read only after the internal method is frozen, and
it is never used for key selection, thresholds, or configuration choice.
"""

import io
import json
import os
import re
import zipfile
from fractions import Fraction

import numpy as np
import pandas as pd

VCTK_TARGET_SR = 16000
VCTK_MIN_SAMPLES = 24000          # sealed validity gate, post-resample
VCTK_UTTS_PER_SPEAKER = 15        # ranks 0-4 enroll, 5-14 probe
VCTK_ZIP_BYTES = 11747302977      # the copy in Drive, as downloaded Sep 17

_MIC1 = re.compile(r"(?:^|/)(p\d{3}|s5)_(\d+)_mic1\.flac$")

# Cell 17 defines these when it runs the download itself. It has not, so
# fall back to the conventional locations and register them, because
# extract_vctk_embeddings() reads both.
if "VCTK_MANIFEST" not in globals():
    VCTK_MANIFEST = DIR["protocol"] / "vctk_manifest.tsv"
if "audio_vctk" not in DIR:
    DIR["audio_vctk"] = DIR["preprocessed"] / "vctk_16k"


def locate_vctk_archive():
    """Find the downloaded archive. Returns a Path."""
    candidates = []
    if "VCTK_ARCHIVE" in globals():
        candidates.append(VCTK_ARCHIVE)
    candidates.append(
        DIR["datasets"] / "VCTK-Corpus-0.92" / "VCTK-Corpus-0.92.zip"
    )
    candidates.extend(sorted(DIR["datasets"].glob("**/VCTK-Corpus-0.92.zip")))

    for path in candidates:
        if path and path.exists() and path.stat().st_size > 0:
            size = path.stat().st_size
            if size != VCTK_ZIP_BYTES:
                print(f"NOTE: {path.name} is {size} bytes, expected "
                      f"{VCTK_ZIP_BYTES}. Continuing; the zip directory "
                      f"check below will catch a truncated file.")
            return path

    raise RuntimeError(
        "VCTK-Corpus-0.92.zip not found under "
        f"{DIR['datasets']}. Expected the copy downloaded on Sep 17."
    )


def verify_vctk_archive_md5(path=None):
    """Optional integrity check. Reads all 11.7 GB, so it is slow over
    Drive; the zip central directory check in prepare_vctk() is usually
    enough."""
    path = path or locate_vctk_archive()
    if "VCTK_MD5" not in globals():
        print("VCTK_MD5 is not defined; nothing to compare against.")
        return None
    import hashlib

    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 24), b""):
            h.update(block)
    got = h.hexdigest()
    print(f"md5 {got} {'OK' if got == VCTK_MD5 else 'MISMATCH'}")
    return got


# ------------------------------------------------------------- audio I/O

def _decode_flac(raw):
    """FLAC bytes -> (float32 mono ndarray, sample_rate)."""
    try:
        import soundfile as sf

        x, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        return x.mean(axis=1).astype(np.float32), int(sr)
    except ImportError:
        pass

    try:
        import torchaudio

        wav, sr = torchaudio.load(io.BytesIO(raw), format="flac")
        return wav.mean(dim=0).numpy().astype(np.float32), int(sr)
    except Exception as exc:
        raise RuntimeError(
            "No working FLAC decoder. Install one with "
            "`pip install soundfile` (or run ensure_runtime_deps(), which "
            f"installs av). Last error: {exc}"
        )


def _to_16k(x, sr):
    """Anti-aliased resample to 16 kHz. Exact 1/3 decimation for VCTK."""
    if sr == VCTK_TARGET_SR:
        return x
    from scipy.signal import resample_poly

    ratio = Fraction(VCTK_TARGET_SR, sr).limit_denominator(1000)
    return resample_poly(x, ratio.numerator, ratio.denominator).astype(
        np.float32
    )


def _wav_frames(path):
    """Frame count of an existing wav, or None if it cannot be read.

    Read from the header rather than inferred from the file size: the
    44-byte canonical header is not guaranteed, and a partially written
    file must not be mistaken for a valid one.
    """
    import wave

    try:
        with wave.open(str(path), "rb") as wf:
            return wf.getnframes()
    except Exception:
        return None


def _write_wav_16k(path, x):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    clipped = np.clip(x, -1.0, 1.0)
    pcm = (clipped * 32767.0).astype(np.int16)
    try:
        import soundfile as sf

        # format given explicitly: the ".tmp" suffix hides it from soundfile.
        # Either branch writes the same 16-bit samples.
        sf.write(tmp, pcm, VCTK_TARGET_SR, subtype="PCM_16", format="WAV")
    except ImportError:
        import wave

        with wave.open(str(tmp), "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(VCTK_TARGET_SR)
            wf.writeframes(pcm.tobytes())
    os.replace(tmp, path)
    return len(pcm)


# ------------------------------------------------------------- preparation

def prepare_vctk(force=False, progress_every=10):
    """Build DIR["audio_vctk"] and VCTK_MANIFEST from the archive in Drive.

    Returns the manifest DataFrame.
    """
    if not force and verified_stage(
        "vctk_manifest", [(VCTK_MANIFEST, "manifest_sha256")]
    ):
        print("RESUME: VCTK manifest verified.")
        return pd.read_csv(
            VCTK_MANIFEST, sep="\t", dtype={"speaker_id": str}
        )

    archive = locate_vctk_archive()
    print(f"archive {archive}")
    print(f"        {archive.stat().st_size} bytes")

    with zipfile.ZipFile(archive) as zf:
        members = [n for n in zf.namelist() if _MIC1.search(n)]
        if not members:
            raise RuntimeError(
                "No *_mic1.flac members found. The archive is not the "
                "VCTK-Corpus-0.92 release, or it is truncated."
            )

        by_speaker = {}
        for name in members:
            speaker = _MIC1.search(name).group(1)
            by_speaker.setdefault(speaker, []).append(name)
        for names in by_speaker.values():
            names.sort()

        print(f"mic1 members {len(members)} across "
              f"{len(by_speaker)} speakers")
        print(f"gate: post-resample samples >= {VCTK_MIN_SAMPLES} "
              f"({VCTK_MIN_SAMPLES / VCTK_TARGET_SR:.2f} s), keeping the "
              f"first {VCTK_UTTS_PER_SPEAKER} valid per speaker")

        rows, skipped, done = [], {}, 0
        for speaker in sorted(by_speaker):
            kept, tried = [], 0
            for name in by_speaker[speaker]:
                if len(kept) >= VCTK_UTTS_PER_SPEAKER:
                    break
                tried += 1
                stem = name.rsplit("/", 1)[-1][:-len(".flac")]
                rel = f"{speaker}/{stem}.wav"
                out_path = DIR["audio_vctk"] / rel

                if out_path.exists():
                    n = _wav_frames(out_path)
                    if n is not None and n >= VCTK_MIN_SAMPLES:
                        kept.append((rel, int(n), name, None))
                        continue
                    # Truncated, unreadable, or below the gate: rewrite it
                    # rather than carry a bad file into the manifest.
                    out_path.unlink()

                x, sr = _decode_flac(zf.read(name))
                if x.size == 0 or not np.isfinite(x).all():
                    continue
                y = _to_16k(x, sr)
                if y.size < VCTK_MIN_SAMPLES:
                    continue
                n = _write_wav_16k(out_path, y)
                kept.append((rel, int(n), name, int(sr)))

            if len(kept) < VCTK_UTTS_PER_SPEAKER:
                skipped[speaker] = len(kept)
                # Leave the partial wavs on disk; they cost little and make
                # a re-run cheap if the gate is ever revisited. They are not
                # in the manifest, so nothing reads them.
                continue

            for rank, (rel, n, member, src_sr) in enumerate(kept):
                rows.append({
                    "speaker_id": speaker,
                    "rank": rank,
                    "relative_path": rel,
                    "n_samples": n,
                    "sample_rate": VCTK_TARGET_SR,
                    "source_sample_rate": src_sr,
                    "source_member": member,
                })

            done += 1
            if progress_every and done % progress_every == 0:
                print(f"  {done} speakers prepared "
                      f"({len(rows)} utterances)", flush=True)

    if not rows:
        raise RuntimeError(
            "No speaker supplied 15 valid utterances. Check the gate and "
            "the decoder before changing the protocol."
        )

    manifest = pd.DataFrame(rows).sort_values(
        ["speaker_id", "rank"]
    ).reset_index(drop=True)

    VCTK_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    tmp = VCTK_MANIFEST.with_name(VCTK_MANIFEST.name + ".tmp")
    manifest.to_csv(tmp, sep="\t", index=False)
    os.replace(tmp, VCTK_MANIFEST)

    mark_stage("vctk_manifest", {
        "corpus": "VCTK-Corpus-0.92",
        "archive_bytes": archive.stat().st_size,
        "mic": "mic1",
        "target_sample_rate": VCTK_TARGET_SR,
        "validity_rule": "mic1_post_resample_samples>=24000",
        "utterances_per_speaker": VCTK_UTTS_PER_SPEAKER,
        "selection": "first N valid in sorted filename order",
        "resampler": "scipy.signal.resample_poly",
        "n_speakers": int(manifest["speaker_id"].nunique()),
        "n_utterances": int(len(manifest)),
        "n_speakers_skipped": len(skipped),
        "manifest_sha256": sha256_file(VCTK_MANIFEST),
    })

    print(f"\nspeakers kept    {manifest['speaker_id'].nunique()}")
    print(f"utterances       {len(manifest)}")
    print(f"speakers skipped {len(skipped)} (fewer than "
          f"{VCTK_UTTS_PER_SPEAKER} valid utterances)")
    if skipped:
        shown = sorted(skipped.items())[:10]
        print("  " + ", ".join(f"{s}:{n}" for s, n in shown)
              + (" ..." if len(skipped) > 10 else ""))
    print(f"\nWrote {VCTK_MANIFEST}")
    print(f"Audio  {DIR['audio_vctk']}")
    return manifest


def vctk_manifest_report():
    if not VCTK_MANIFEST.exists():
        print("No VCTK manifest yet. Run prepare_vctk().")
        return None
    m = pd.read_csv(VCTK_MANIFEST, sep="\t", dtype={"speaker_id": str})
    dur = m["n_samples"] / VCTK_TARGET_SR
    print(f"speakers   {m['speaker_id'].nunique()}")
    print(f"utterances {len(m)}")
    print(f"duration   min {dur.min():.2f}s  median {dur.median():.2f}s  "
          f"max {dur.max():.2f}s  total {dur.sum()/60:.1f} min")
    counts = m.groupby("speaker_id").size()
    bad = counts[counts != VCTK_UTTS_PER_SPEAKER]
    if len(bad):
        print(f"WARNING: {len(bad)} speakers do not have exactly "
              f"{VCTK_UTTS_PER_SPEAKER} utterances; "
              "vctk_external_data() will skip them.")
    missing = [
        r.relative_path for r in m.itertuples(index=False)
        if not (DIR["audio_vctk"] / r.relative_path).exists()
    ]
    if missing:
        print(f"WARNING: {len(missing)} manifest files are missing on "
              f"disk, first: {missing[0]}")
    else:
        print("all manifest files present on disk")
    return m


print("VCTK preparation ready.")
print("  locate_vctk_archive()     - find the zip already in Drive")
print("  prepare_vctk()            - stream mic1, resample, build manifest")
print("  vctk_manifest_report()    - check what was built")
print("  verify_vctk_archive_md5() - optional, reads all 11.7 GB")


In [ ]:
%%writefile {RUNTIME}/fresh_pipeline_only.py
# 10. Acquisition to evaluation, and the comparison with the paper
"""The whole AttackAware PolyIoM v1.1.4 study, rerun from the public data.

Loaded by AttackAware_PolyIoM_KAGGLE_FRESH.ipynb after the study's analysis
runtime (ablation_only, scoredump_only, inversion_only, revocation_only,
allfigures_only, vctk_prepare_cells).

Stages 1 to 5 are ported from the study's own Colab notebook,
AttackAware_PolyIoM_v1_1_4_PublicData_15GB_Resumable_CHECKPOINT_FIX.ipynb
(cells 5 to 22): the downloads, MTCNN and FaceNet, the CFP gate, the LFW
and LibriSpeech splits, ECAPA, the polynomial key generator, the IoM
tensors and the Stage A key search, with the same seed labels and rules.
Only the plumbing changes: Kaggle paths instead of Google Drive, soundfile
for audio input and output (FLAC is lossless, so the samples are the same),
and the ECAPA normalisation checkpoint hash taken from the study's sealed
record (cd70225b...), which the notebook's constant (0eb77a0b...) does not
match.

Every stage compares what it produces with what the study recorded and
prints "identical" or "DIFFERENT". If every stage is identical, this is an
exact reproduction. Where a stage differs (a newer library can move a
float), the run carries on, and the final report sets each result beside
the paper's with the fresh 95% interval.

Every stage writes a marker under seal/, so a rerun skips it and a new
Kaggle session continues where the last one stopped.
"""

import hashlib
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------- the record
PAPER = json.loads((RUNTIME / "paper_reference.json").read_text())

# The study's frozen constants (its notebook, cell 2).
SPLIT = {"background": 50, "development": 42, "evaluation": 58}
M_REF, Q_REF, O_REF = 128, 16, 2
TARGET_FMR = 1e-3
N_INIT, N_MAX = 10_000, 20_000
STAGE1_RHO_MAX, STAGE1_TAR_MIN = 0.90, 0.95
STAGE2_RHO_MAX, STAGE2_TAR_MIN = 0.85, 0.90
ATTACK_RESTARTS, ATTACK_ITERS, ATTACK_LR = 5, 2000, 0.01
ATTACK_BETAS, ATTACK_EPS, ATTACK_INIT_STD = (0.9, 0.999), 1e-8, 0.1
KEYSEARCH_CANDIDATE_BATCH = 32
FACE_VALID_PROB = 0.95
SELECTION_RULE_VOICE = {"eer_max": 0.01, "tar_min": 0.95}
FACE_FLOOR = {"eer_max": 0.03, "tar_min": 0.85}
NONZERO_COEFFS = np.array(list(range(-50, 0)) + list(range(1, 51)), dtype=np.int64)

URLS = {
    "lfw": ["https://ndownloader.figshare.com/files/5976018",
            "http://vis-www.cs.umass.edu/lfw/lfw.tgz"],
    "libri": ["https://www.openslr.org/resources/12/train-clean-100.tar.gz",
              "https://us.openslr.org/resources/12/train-clean-100.tar.gz",
              "https://openslr.elda.org/resources/12/train-clean-100.tar.gz"],
    "vctk": ["https://datashare.ed.ac.uk/bitstreams/"
             "535f4286-e54c-4038-838c-a02285e32cb2/download",
             "https://datashare.ed.ac.uk/bitstream/handle/10283/3443/"
             "VCTK-Corpus-0.92.zip"],
}
REF = {
    "lfw_sha256": "055f7d9c632d7370e6fb4afc7468d40f970c34a80d4c6f50ffec63f5a8d536c0",
    "libri_md5": "2a93770f6d5c6c964bc36631d331a522",
    "vctk_md5": "8a6ba2946b36fcbef0212cad601f4bfa",
    "lfw_raw_images": PAPER["stages"]["lfw_raw"]["raw_images"],
    "lfw_valid": PAPER["stages"]["lfw_embeddings"]["valid_embeddings"],
    "lfw_eligible": PAPER["stages"]["lfw_embeddings"]["identities_with_at_least_3_valid"],
    "lfw_embeddings_sha256": PAPER["stages"]["lfw_embeddings"]["embeddings_sha256"],
    "cfp_source_sha256": PAPER["cfp"]["source_sha256"],
    "cfp_metadata_sha256": PAPER["cfp"]["metadata_sha256"],
    "cfp_overlap": PAPER["lfw_split"]["excluded_exact_overlap_count"],
    "lfw_split_sha256": PAPER["lfw_split"]["split_sha256"],
    "lfw_excluded_sha256": PAPER["lfw_split"]["excluded_sha256"],
    "libri_manifest_sha256": PAPER["stages"]["libri_internal"]["manifest_sha256"],
    "libri_embeddings": PAPER["stages"]["libri_embeddings"]["embeddings"],
    "libri_embeddings_sha256": PAPER["stages"]["libri_embeddings"]["sha256"],
    "vctk_manifest_sha256": PAPER["stages"]["vctk_manifest"]["manifest_sha256"],
    "vctk_speakers": PAPER["stages"]["vctk_manifest"]["n_speakers"],
    "vctk_utterances": PAPER["stages"]["vctk_manifest"]["n_utterances"],
    "vctk_embeddings_sha256": PAPER["stages"]["vctk_embeddings"]["sha256"],
    "ecapa_repo": PAPER["ecapa"]["repo_id"],
    "ecapa_revision": PAPER["ecapa"]["revision"],
    "ecapa_embedding_sha256": PAPER["ecapa"]["embedding_model_ckpt"]["sha256"],
    "ecapa_norm_sha256": PAPER["ecapa"]["mean_var_norm_emb_ckpt"]["sha256"],
    "voice_key_sha256": PAPER["external"].get("voice_key_sha256"),
}
CFP_KAGGLE = "chinafax/cfpw-dataset"

EXTRACT_DEVICE = torch.device(os.environ.get("POLYIOM_EXTRACT_DEVICE", "cpu"))
ATTACK_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SESSION_START = time.time()
SESSION_BUDGET_H = float(os.environ.get("POLYIOM_SESSION_HOURS", "11.0"))

MODELS = PROJECT / "models"
LFW_LOCAL_ROOT = RAW / "lfw_extract"
LIBRI_LOCAL_ROOT = RAW / "LibriSpeech_extract"
LIBRI_SELECTED_ROOT = PROJECT / "datasets" / "LibriSpeech_selected"
VCTK_ARCHIVE = RAW / "VCTK-Corpus-0.92.zip"
LFW_VALIDITY = DIR["protocol"] / "lfw_validity.tsv"
LFW_SPLIT = DIR["protocol"] / "lfw_internal_split.tsv"
LFW_EXCLUDED = DIR["protocol"] / "lfw_excluded_cfp_overlap.txt"
CFP_METADATA_DIR = DIR["protocol"] / "cfp_authoritative_metadata"
CFP_METADATA = CFP_METADATA_DIR / "cfp_subject_names.tsv"


# --------------------------------------------------------------- markers
def stage_file(name):
    return DIR["seal"] / f"stage_{name}.done.json"


def stage_done(name):
    return stage_file(name).exists()


def mark_stage(name, payload=None):
    payload = dict(payload or {})
    payload.update({"stage": name, "protocol_version": PROTOCOL_VERSION,
                    "completed_unix": time.time()})
    atomic_json(stage_file(name), payload)


def verified_stage(stage, artifacts):
    if not stage_done(stage):
        return False
    record = json.loads(stage_file(stage).read_text())
    return all(Path(p).is_file() and record.get(k) == sha256_file(p)
               for p, k in artifacts)


def time_left_h():
    return SESSION_BUDGET_H - (time.time() - SESSION_START) / 3600


def exact(label, got, expected, tol=None):
    """One line of the reproduction record: does the fresh run give the
    study's value?"""
    if tol is not None and isinstance(got, (int, float)) and isinstance(expected, (int, float)):
        same = abs(float(got) - float(expected)) <= tol
    else:
        same = got == expected
    path = DIR["seal"] / "exact_checks.json"
    rows = json.loads(path.read_text()) if path.is_file() else []
    rows = [r for r in rows if r["check"] != label] + [{
        "check": label, "fresh": got, "study": expected,
        "verdict": "identical" if same else "DIFFERENT"}]
    atomic_json(path, rows)
    print(f"  {'identical' if same else 'DIFFERENT':<10} {label}"
          + ("" if same else f": fresh {got}, study {expected}"))
    return same


# ------------------------------------------------------------- downloads
def md5_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def download(name, dest, sha256=None, md5=None):
    """Fetch from the first working source, resuming a partial file, and
    verify it. A file that fails its hash is deleted and the error raised."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = Path(str(dest) + ".part")
    if not dest.is_file():
        last = None
        for url in URLS[name]:
            try:
                offset = part.stat().st_size if part.is_file() else 0
                request = urllib.request.Request(url)
                if offset and url.startswith("http"):
                    request.add_header("Range", f"bytes={offset}-")
                with urllib.request.urlopen(request, timeout=120) as response:
                    resumed = getattr(response, "status", 200) == 206
                    print(f"  {name}: downloading from {url}"
                          + (" (resuming)" if resumed else ""), flush=True)
                    with open(part, "ab" if resumed else "wb") as out:
                        shutil.copyfileobj(response, out, length=1 << 24)
                os.replace(part, dest)
                break
            except Exception as exc:
                last = exc
                print(f"  {name}: {url} failed - {type(exc).__name__}: {exc}")
        else:
            raise RuntimeError(f"{name}: every source failed (last error: {last}). "
                               "Is Internet switched on in the notebook settings?")
    for label, expected, fn in (("sha256", sha256, sha256_file), ("md5", md5, md5_file)):
        if expected:
            got = fn(dest)
            if got != expected:
                dest.unlink()
                raise RuntimeError(f"{name}: {label} {got} is not the study's "
                                   f"{expected}; the file was deleted.")
            exact(f"{name} archive {label}", got, expected)
    return dest


# ---------------------------------------------- stage 0: libraries, models
def install_dependencies():
    """As the study's cell 1, except facenet-pytorch goes in without its
    dependency pins, which would downgrade torch (runtime_deps_cells.py)."""
    before = torch.__version__
    for spec, nodeps in (("facenet-pytorch==2.6.0", True), ("speechbrain==1.1.1", False)):
        try:
            __import__(spec.split("==")[0].replace("-", "_"))
            continue
        except ImportError:
            pass
        cmd = [sys.executable, "-m", "pip", "install", "-q", spec]
        print("  pip install", spec, "(no dependencies)" if nodeps else "")
        subprocess.check_call(cmd + (["--no-deps"] if nodeps else []))
    print(f"  torch {before} before, {torch.__version__} after")


def environment_record():
    from importlib import metadata

    def version(name):
        try:
            return metadata.version(name)
        except Exception:
            return None
    env = {"python": sys.version, "platform": platform.platform(),
           "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
           "cuda_version": torch.version.cuda,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "extract_device": str(EXTRACT_DEVICE), "attack_device": str(ATTACK_DEVICE)}
    for name in ("numpy", "pandas", "scipy", "torchaudio", "torchvision",
                 "facenet-pytorch", "speechbrain", "soundfile", "huggingface-hub"):
        env[name] = version(name)
    atomic_json(DIR["seal"] / "environment.json", env)
    return env


def load_face_models():
    """The study's MTCNN and FaceNet (its notebook, cell 5)."""
    from facenet_pytorch import MTCNN, InceptionResnetV1
    os.environ["TORCH_HOME"] = str(MODELS / "torch")
    mtcnn = MTCNN(image_size=160, margin=0, min_face_size=20,
                  thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
                  select_largest=True, keep_all=False, device=EXTRACT_DEVICE)
    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(EXTRACT_DEVICE)
    return mtcnn, resnet


def load_ecapa():
    """ECAPA at the study's pinned revision, both checkpoints hash-checked."""
    from huggingface_hub import hf_hub_download, snapshot_download
    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except Exception:
        from speechbrain.inference.classifiers import EncoderClassifier
    ecapa_dir = MODELS / f"ecapa_{REF['ecapa_revision']}"
    ecapa_dir.mkdir(parents=True, exist_ok=True)
    if not (ecapa_dir / "hyperparams.yaml").exists():
        snapshot_download(repo_id=REF["ecapa_repo"], revision=REF["ecapa_revision"],
                          local_dir=str(ecapa_dir))
    for filename, expected in (("embedding_model.ckpt", REF["ecapa_embedding_sha256"]),
                               ("mean_var_norm_emb.ckpt", REF["ecapa_norm_sha256"])):
        path = ecapa_dir / filename
        if not path.is_file() or sha256_file(path) != expected:
            fresh = hf_hub_download(repo_id=REF["ecapa_repo"], filename=filename,
                                    revision=REF["ecapa_revision"], force_download=True)
            shutil.copy2(fresh, path)
        exact(f"ECAPA {filename} sha256", sha256_file(path), expected)
    return EncoderClassifier.from_hparams(source=str(ecapa_dir),
                                          savedir=str(MODELS / "ecapa_loaded"),
                                          run_opts={"device": str(EXTRACT_DEVICE)})


def ecapa_embed_waveform(model, wav_16k_mono):
    """The study's embedding call (its notebook, cell 5)."""
    wav = torch.as_tensor(wav_16k_mono, dtype=torch.float32,
                          device=EXTRACT_DEVICE).reshape(1, -1)
    with torch.no_grad():
        e = model.encode_batch(wav, wav_lens=torch.ones(1, device=EXTRACT_DEVICE),
                               normalize=False)
    e = l2_np(e.reshape(-1).detach().cpu().numpy())
    if e.shape != (192,):
        raise RuntimeError(f"Unexpected ECAPA embedding shape: {e.shape}")
    return e


def load_audio(path):
    """Decode, mono, 16 kHz; and whether it is non-empty and finite."""
    import soundfile as sf
    x, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wav = torch.as_tensor(x.T.copy())
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        import torchaudio
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.squeeze(0).contiguous()
    return wav, bool(wav.numel() > 0 and torch.isfinite(wav).all())


def save_flac_16k(path, wav):
    import soundfile as sf
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), wav.numpy(), 16000, format="FLAC", subtype="PCM_16")


# ------------------------------------------------------------ stage 1: LFW
def prepare_lfw_raw():
    root = LFW_LOCAL_ROOT / "lfw"
    if (LFW_LOCAL_ROOT / ".extracted").is_file():
        return root
    archive = download("lfw", RAW / "lfw.tgz", sha256=REF["lfw_sha256"])
    LFW_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        tf.extractall(LFW_LOCAL_ROOT)
    if not root.exists():
        root = [p for p in LFW_LOCAL_ROOT.iterdir() if p.is_dir()][0]
    n_img = len(list(root.rglob("*.jpg")))
    exact("LFW raw images", n_img, REF["lfw_raw_images"])
    mark_stage("lfw_raw", {"archive_sha256": sha256_file(archive), "raw_images": n_img})
    (LFW_LOCAL_ROOT / ".extracted").write_text("ok")
    return root


def stage_face():
    """MTCNN validity and 512-D FaceNet embeddings (cell 7)."""
    if stage_done("lfw_embeddings") and LFW_EMB.exists() and LFW_VALIDITY.exists():
        print("RESUME: LFW embeddings present.")
        return
    from PIL import Image
    root = prepare_lfw_raw()
    mtcnn, resnet = load_face_models()
    identity_files = {}
    for subject_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        files = sorted(subject_dir.glob("*.jpg"))
        if len(files) >= 3:
            identity_files[subject_dir.name] = files
    rows, emb_subject, emb_relpath, embs = [], [], [], []
    for n, subject in enumerate(sorted(identity_files), 1):
        for img_path in identity_files[subject]:
            rec = {"subject_id": subject, "relative_path": str(img_path.relative_to(root)),
                   "sha256": sha256_file(img_path), "valid": False,
                   "mtcnn_prob": np.nan, "error": ""}
            try:
                face, prob = mtcnn(Image.open(img_path).convert("RGB"), return_prob=True)
                prob = float(prob) if prob is not None else float("nan")
                rec["mtcnn_prob"] = prob
                if face is None or not np.isfinite(prob) or prob < FACE_VALID_PROB:
                    rec["error"] = "mtcnn_validity_gate"
                    rows.append(rec)
                    continue
                with torch.no_grad():
                    e = resnet(face.unsqueeze(0).to(EXTRACT_DEVICE)).squeeze(0).cpu().numpy()
                e = l2_np(e)
                rec["valid"] = True
                rec["embedding_norm"] = float(np.linalg.norm(e))
                emb_subject.append(subject)
                emb_relpath.append(rec["relative_path"])
                embs.append(e)
            except Exception as exc:
                rec["error"] = repr(exc)
            rows.append(rec)
        if n % 100 == 0:
            print(f"    {n}/{len(identity_files)} identities, {len(embs):,} valid faces",
                  flush=True)
    validity = pd.DataFrame(rows).sort_values(["subject_id", "relative_path"])
    validity.to_csv(LFW_VALIDITY, sep="\t", index=False)
    np.savez_compressed(LFW_EMB, subject_id=np.array(emb_subject, dtype=object),
                        relative_path=np.array(emb_relpath, dtype=object),
                        embedding=np.stack(embs).astype(np.float32))
    counts = validity[validity["valid"] == True].groupby("subject_id").size()  # noqa: E712
    eligible = int((counts >= 3).sum())
    exact("LFW valid embeddings", len(embs), REF["lfw_valid"])
    exact("LFW identities with at least 3 valid images", eligible, REF["lfw_eligible"])
    exact("LFW embeddings file sha256", sha256_file(LFW_EMB), REF["lfw_embeddings_sha256"])
    mark_stage("lfw_embeddings", {"valid_embeddings": len(embs),
                                  "identities_with_at_least_3_valid": eligible,
                                  "validity_sha256": sha256_file(LFW_VALIDITY),
                                  "embeddings_sha256": sha256_file(LFW_EMB)})


# --------------------------------------------------- stage 2: CFP and split
def find_cfp_list():
    for root in (INPUT_ROOT, RAW):
        if Path(root).exists():
            hits = sorted(Path(root).glob("**/list_name.txt"))
            if hits:
                return hits[0]
    try:
        import kagglehub
        hits = sorted(Path(kagglehub.dataset_download(CFP_KAGGLE)).glob("**/list_name.txt"))
        if hits:
            return hits[0]
    except Exception as exc:
        print(f"  kagglehub could not fetch {CFP_KAGGLE}: {exc}")
    raise FileNotFoundError(f"CFP name list not found. Use 'Add Input' and add the Kaggle "
                            f"dataset {CFP_KAGGLE}, which is the source the study used.")


def cfp_subject_names():
    """The sealed CFP metadata: the release's 500 names in order, with the
    rule case_sensitive_space_to_underscore_v1."""
    source = find_cfp_list()
    exact("CFP list_name.txt sha256", sha256_file(source), REF["cfp_source_sha256"])
    names = [line.strip() for line in source.read_text(errors="replace").splitlines()
             if line.strip()]
    table = pd.DataFrame({"cfp_subject_id": [f"{i:03d}" for i in range(1, len(names) + 1)],
                          "subject_name": [n.replace(" ", "_") for n in names]})
    CFP_METADATA_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(CFP_METADATA, sep="\t", index=False)
    exact("CFP metadata table sha256", sha256_file(CFP_METADATA), REF["cfp_metadata_sha256"])
    return sorted(set(table["subject_name"]))


def lfw_embedding_records():
    z = np.load(LFW_EMB, allow_pickle=True)
    return pd.DataFrame({"subject_id": z["subject_id"].astype(str),
                         "relative_path": z["relative_path"].astype(str),
                         "embedding_index": np.arange(len(z["embedding"]), dtype=np.int64)})


def stage_face_split():
    """Exact CFP overlap removal, the 50/42/58 split and the trials (cells 9-10)."""
    if stage_done("lfw_face_trials") and LFW_FACE_TRIALS.exists():
        print("RESUME: face split and trials present.")
        return
    n_bg, n_dev = SPLIT["background"], SPLIT["development"]
    cfp_set = set(cfp_subject_names())
    rec = lfw_embedding_records()
    counts = rec.groupby("subject_id").size()
    eligible = sorted(counts[counts >= 3].index.astype(str))
    excluded = sorted(u for u in eligible if u in cfp_set)
    remaining = [u for u in eligible if u not in cfp_set]
    ranked = sorted(remaining, key=lambda u: (H(S0, "LFW-SPLIT", str(u)), u))
    if len(ranked) < sum(SPLIT.values()):
        raise RuntimeError(f"only {len(ranked)} eligible LFW identities remain")
    rows = []
    for idx, u in enumerate(ranked[:sum(SPLIT.values())]):
        part = ("background" if idx < n_bg else
                "development" if idx < n_bg + n_dev else "evaluation")
        rows.append({"subject_id": u, "split_rank_hash": H(S0, "LFW-SPLIT", str(u)),
                     "partition": part})
    pd.DataFrame(rows).to_csv(LFW_SPLIT, sep="\t", index=False)
    LFW_EXCLUDED.write_text("\n".join(excluded) + ("\n" if excluded else ""))
    exact("LFW identities excluded by the CFP overlap", len(excluded), REF["cfp_overlap"])
    exact("LFW excluded list sha256", sha256_file(LFW_EXCLUDED), REF["lfw_excluded_sha256"])
    exact("LFW 50/42/58 split sha256", sha256_file(LFW_SPLIT), REF["lfw_split_sha256"])
    atomic_json(DIR["seal"] / "lfw_split.json", {
        "eligible_before_overlap_removal": len(eligible),
        "excluded_exact_overlap_count": len(excluded),
        "split_sha256": sha256_file(LFW_SPLIT),
        "excluded_sha256": sha256_file(LFW_EXCLUDED)})
    mark_stage("lfw_split")

    trials = []
    for s in pd.read_csv(LFW_SPLIT, sep="\t").itertuples(index=False):
        g = rec[rec["subject_id"] == s.subject_id].sort_values("relative_path") \
            .reset_index(drop=True)
        if s.partition == "background":
            for rank, role in enumerate(["attack", "enroll", "probe"]):
                r = g.iloc[rank]
                trials.append({"subject_id": s.subject_id, "partition": s.partition,
                               "role": role, "rank": rank,
                               "relative_path": r.relative_path,
                               "embedding_index": int(r.embedding_index)})
        else:
            for rank in range(len(g)):
                r = g.iloc[rank]
                trials.append({"subject_id": s.subject_id, "partition": s.partition,
                               "role": "enroll" if rank == 0 else "probe", "rank": rank,
                               "relative_path": r.relative_path,
                               "embedding_index": int(r.embedding_index)})
    pd.DataFrame(trials).sort_values(["partition", "subject_id", "role", "rank"]) \
        .to_csv(LFW_FACE_TRIALS, sep="\t", index=False)
    mark_stage("lfw_face_trials", {"manifest_sha256": sha256_file(LFW_FACE_TRIALS),
                                   "rows": len(trials)})


# ------------------------------------------------------ stage 3: LibriSpeech
def stage_voice():
    """Speaker selection, 15 valid utterances each, ECAPA (cells 11-13)."""
    if stage_done("libri_embeddings") and LIBRI_EMB.exists():
        print("RESUME: LibriSpeech embeddings present.")
        return
    n_bg, n_dev, n_total = SPLIT["background"], SPLIT["development"], sum(SPLIT.values())
    if not (stage_done("libri_internal") and LIBRI_MANIFEST.exists()):
        archive = download("libri", RAW / "train-clean-100.tar.gz", md5=REF["libri_md5"])
        LIBRI_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive, "r:gz") as tf:
            tf.extractall(LIBRI_LOCAL_ROOT)
        root = next(LIBRI_LOCAL_ROOT.rglob("train-clean-100"))
        mark_stage("libri_local", {"archive_md5": md5_file(archive)})
        candidates = []
        for spk in sorted((p for p in root.iterdir() if p.is_dir()), key=lambda p: p.name):
            files = sorted(spk.rglob("*.flac"), key=lambda p: p.relative_to(root).as_posix())
            if len(files) >= 15:
                candidates.append((spk.name, files))
        candidates.sort(key=lambda it: (H(S0, "LIBRISPEECH-SPLIT", str(it[0])), it[0]))
        rows, selected = [], []
        for speaker_id, files in candidates:
            valid_items = []
            for src in files:
                try:
                    wav, valid = load_audio(src)
                    if valid:
                        valid_items.append((src, wav))
                    if len(valid_items) == 15:
                        break
                except Exception:
                    continue
            if len(valid_items) < 15:
                continue
            index = len(selected)
            if index >= n_total:
                break
            partition = ("background" if index < n_bg else
                         "development" if index < n_bg + n_dev else "evaluation")
            selected.append(speaker_id)
            for rank, (src, wav) in enumerate(valid_items):
                dest = LIBRI_SELECTED_ROOT / speaker_id / f"{rank:02d}_{src.stem}.flac"
                if not dest.exists():
                    save_flac_16k(dest, wav)
                if partition == "background":
                    role = ["attack", "enroll", "probe"][rank] if rank < 3 else "reserve"
                else:
                    role = "enroll_member" if rank < 5 else "probe"
                rows.append({"speaker_id": speaker_id,
                             "split_rank_hash": H(S0, "LIBRISPEECH-SPLIT", str(speaker_id)),
                             "partition": partition, "rank": rank, "role": role,
                             "source_relative_path": src.relative_to(root).as_posix(),
                             "stored_relative_path": dest.relative_to(PROJECT).as_posix(),
                             "n_samples_16k": int(wav.numel()), "sha256": sha256_file(dest)})
        if len(selected) != n_total:
            raise RuntimeError(f"only {len(selected)} LibriSpeech speakers qualify")
        pd.DataFrame(rows).sort_values(["partition", "speaker_id", "rank"]) \
            .to_csv(LIBRI_MANIFEST, sep="\t", index=False)
        exact("LibriSpeech manifest sha256", sha256_file(LIBRI_MANIFEST),
              REF["libri_manifest_sha256"])
        mark_stage("libri_internal", {"selected_speakers": len(selected),
                                      "manifest_sha256": sha256_file(LIBRI_MANIFEST)})
        shutil.rmtree(LIBRI_LOCAL_ROOT, ignore_errors=True)
        archive.unlink(missing_ok=True)

    model = load_ecapa()
    manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
    ids, ranks, paths, embs = [], [], [], []
    for n, r in enumerate(manifest.itertuples(index=False), 1):
        wav, valid = load_audio(PROJECT / r.stored_relative_path)
        if not valid:
            raise RuntimeError(f"Persisted invalid waveform: {r.stored_relative_path}")
        embs.append(ecapa_embed_waveform(model, wav.numpy()))
        ids.append(str(r.speaker_id))
        ranks.append(int(r.rank))
        paths.append(str(r.stored_relative_path))
        if n % 250 == 0:
            print(f"    {n:,}/{len(manifest):,} utterances embedded", flush=True)
    np.savez_compressed(LIBRI_EMB, speaker_id=np.array(ids, dtype=object),
                        rank=np.array(ranks, dtype=np.int64),
                        relative_path=np.array(paths, dtype=object),
                        embedding=np.stack(embs).astype(np.float32))
    exact("LibriSpeech embeddings", len(embs), REF["libri_embeddings"])
    exact("LibriSpeech embeddings file sha256", sha256_file(LIBRI_EMB),
          REF["libri_embeddings_sha256"])
    mark_stage("libri_embeddings", {"embeddings": len(embs), "sha256": sha256_file(LIBRI_EMB)})


# ------------------------------------------------------- stage 4: tensors
def stage_tensors():
    """The frozen IoM tensors, seeded H(2026, 'IOM', modality, overlap) (cell 15)."""
    for modality, d in DIMENSION.items():
        for o in CONFIG["overlaps"]:
            k = 1 + math.ceil((d - G) / (G - o))
            path = RUNS_IN / "iom_tensors" / f"{modality}_o{o}_k{k}.pt"
            if path.is_file():
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            gen = torch.Generator(device="cpu")
            gen.manual_seed(H(S0, "IOM", modality, o))
            torch.save(torch.randn((256, 32, k), generator=gen, dtype=torch.float32), path)
    print("  frozen IoM tensors present for both modalities and every overlap")


# ------------------------------------------------------ stage 5: Stage A
def generate_candidate(modality, candidate_id):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(H(S0, "POLY", modality, candidate_id))
    perm = torch.randperm(len(NONZERO_COEFFS), generator=gen)[:G].numpy()
    Cc = NONZERO_COEFFS[perm].astype(np.float32)
    Ee = (torch.randperm(G, generator=gen) + 1).numpy().astype(np.int64)
    return Cc, Ee


def poly_transform_candidates(x, Cs, Es, o):
    """P(x) for a batch of candidate keys; vals is [B, N, k, G] or
    [B, N, R, k, G]. The study's notebook shaped the keys as [B, 1, G], which
    lines B up against N and fails unless the batch has one candidate (its
    sealed search ran on 7 Sep with earlier, sharded code). Here the keys are
    [B, 1, ..., 1, G], so candidate b uses its own key, as intended. Each
    candidate's attack is independent of the others in the batch."""
    if x.ndim == 2:
        x = x.unsqueeze(0).expand(Cs.shape[0], -1, -1)
    idx, mask = poly_indices(x.shape[-1], o, x.device)
    vals = x[..., idx]
    coeff_shape = [Cs.shape[0]] + [1] * (vals.ndim - 2) + [G]
    coeff = Cs.to(x.device, x.dtype).view(*coeff_shape)
    exp = Es.to(x.device, torch.int64).view(*coeff_shape)
    vals = vals * mask.to(vals.dtype).view(*([1] * (vals.ndim - 2) + list(mask.shape)))
    return (torch.pow(vals, exp) * coeff).sum(dim=-1)


def iom_hash(P, R, M, q):
    scores = torch.einsum("...k,mqk->...mq", P, R[:M, :q, :].to(P.device, P.dtype))
    return torch.argmax(scores, dim=-1).to(torch.int16)


def integer_threshold(impostor_counts, M, alpha=TARGET_FMR):
    imp = np.asarray(impostor_counts, dtype=np.int64)
    for c in range(0, M + 2):
        if np.mean(imp >= c) <= alpha:
            return int(c)
    raise AssertionError("Threshold sentinel logic failed.")


def background_tensors(modality):
    """Attack, enrolment and probe embeddings of the background people (cell 20)."""
    if modality == "face":
        trials = pd.read_csv(LFW_FACE_TRIALS, sep="\t")
        E = np.load(LFW_EMB, allow_pickle=True)["embedding"].astype(np.float32)
        bg = trials[trials["partition"] == "background"]
        ids, A, B, C = [], [], [], []
        for uid, g in bg.groupby("subject_id"):
            byrole = {r.role: r for r in g.itertuples(index=False)}
            ids.append(str(uid))
            A.append(E[int(byrole["attack"].embedding_index)])
            B.append(E[int(byrole["enroll"].embedding_index)])
            C.append(E[int(byrole["probe"].embedding_index)])
    else:
        manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
        z = np.load(LIBRI_EMB, allow_pickle=True)
        E = z["embedding"].astype(np.float32)
        table = pd.DataFrame({"speaker_id": z["speaker_id"].astype(str),
                              "rank": z["rank"].astype(int),
                              "embedding_index": np.arange(len(E))})
        merged = manifest.merge(table, on=["speaker_id", "rank"], how="inner")
        bg = merged[merged["partition"] == "background"]
        ids, A, B, C = [], [], [], []
        for uid, g in bg.groupby("speaker_id"):
            byrank = {int(r.rank): r for r in g.itertuples(index=False)}
            ids.append(str(uid))
            A.append(E[int(byrank[0].embedding_index)])
            B.append(E[int(byrank[1].embedding_index)])
            C.append(E[int(byrank[2].embedding_index)])
    t = lambda xs: torch.tensor(np.stack(xs), dtype=torch.float32)
    return ids, t(A), t(B), t(C)


def attack_initial_batch(modality, candidate_ids, subject_ids, d):
    out = []
    for cid in candidate_ids:
        per_subject = []
        for uid in subject_ids:
            restarts = []
            for restart in range(ATTACK_RESTARTS):
                gen = torch.Generator(device="cpu")
                gen.manual_seed(H(S0, "ATTACK", modality, int(cid), uid, restart))
                v = torch.randn((d,), generator=gen, dtype=torch.float32) * ATTACK_INIT_STD
                restarts.append(v / torch.linalg.vector_norm(v))
            per_subject.append(torch.stack(restarts))
        out.append(torch.stack(per_subject))
    return torch.stack(out)


def evaluate_candidate_batch(modality, candidate_ids, background):
    """The study's white-box candidate evaluation (cell 18), unchanged."""
    dev = ATTACK_DEVICE
    subject_ids, A_cpu, B_cpu, C_cpu = background
    A, B, Cprobe = A_cpu.to(dev), B_cpu.to(dev), C_cpu.to(dev)
    d = A.shape[-1]
    generated = [generate_candidate(modality, int(c)) for c in candidate_ids]
    Cs = torch.tensor(np.stack([x[0] for x in generated]), dtype=torch.float32, device=dev)
    Es = torch.tensor(np.stack([x[1] for x in generated]), dtype=torch.int64, device=dev)
    with torch.no_grad():
        targetP = poly_transform_candidates(A, Cs, Es, O_REF)
    xprime = torch.nn.Parameter(attack_initial_batch(modality, candidate_ids,
                                                     subject_ids, d).to(dev))
    opt = torch.optim.Adam([xprime], lr=ATTACK_LR, betas=ATTACK_BETAS, eps=ATTACK_EPS,
                           weight_decay=0.0, amsgrad=False)
    for _ in range(ATTACK_ITERS):
        opt.zero_grad(set_to_none=True)
        diff = poly_transform_candidates(xprime, Cs, Es, O_REF) - targetP.unsqueeze(2)
        (diff * diff).sum(dim=-1).sum().backward()
        opt.step()
        with torch.no_grad():
            xprime.data /= torch.linalg.vector_norm(xprime.data, dim=-1,
                                                    keepdim=True).clamp_min(1e-12)
    with torch.no_grad():
        rho = (xprime * A.unsqueeze(0).unsqueeze(2)).sum(dim=-1)
        rho_max = rho.max(dim=-1).values.max(dim=-1).values.cpu().numpy()
        PB = poly_transform_candidates(B, Cs, Es, O_REF)
        PC = poly_transform_candidates(Cprobe, Cs, Es, O_REF)
        R = load_iom_tensor(modality, O_REF, d).to(dev)
        zB, zC = iom_hash(PB, R, M_REF, Q_REF), iom_hash(PC, R, M_REF, Q_REF)
        results, N = [], zB.shape[1]
        for bi, cid in enumerate(candidate_ids):
            cc = (zB[bi, :, None, :] == zC[bi, None, :, :]).sum(dim=-1).cpu().numpy()
            genuine, impostor = np.diag(cc), cc[~np.eye(N, dtype=bool)]
            tau_bg = integer_threshold(impostor, M_REF, TARGET_FMR)
            Cc, Ee = generated[bi]
            results.append({"candidate_id": int(cid), "C": Cc.astype(int).tolist(),
                            "E": Ee.astype(int).tolist(), "rho_max": float(rho_max[bi]),
                            "TAR_background": float(np.mean(genuine >= tau_bg)),
                            "FMR_background": float(np.mean(impostor >= tau_bg)),
                            "c_tau_BG": int(tau_bg)})
    del xprime, opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


def stage_a(modality):
    """The resumable 10k -> 20k key search (cell 19). Returns True when the
    key is selected, False when this session ran out of time."""
    out_dir = RUNS_IN / "key_search" / modality
    selected = out_dir / "selected_key.json"
    if selected.exists():
        print(f"  {modality}: key already selected")
        return True
    out_dir.mkdir(parents=True, exist_ok=True)
    log = out_dir / "candidates.jsonl"
    existing = {}
    if log.exists():
        for line in log.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                existing[int(r["candidate_id"])] = r
    background = background_tensors(modality)
    if len(background[0]) != SPLIT["background"]:
        raise RuntimeError(f"{modality}: {len(background[0])} background identities")

    def run(first, last, stage):
        pending = [c for c in range(first, last + 1) if c not in existing]
        started = time.time()
        for start in range(0, len(pending), KEYSEARCH_CANDIDATE_BATCH):
            if time_left_h() < 0.3:
                print(f"  {modality}: session time nearly used; {len(existing):,} "
                      f"candidates saved. Save this version and run it again.")
                return False
            batch = pending[start:start + KEYSEARCH_CANDIDATE_BATCH]
            rows = evaluate_candidate_batch(modality, batch, background)
            with open(log, "a") as f:
                for r in rows:
                    r["stage_generated"] = stage
                    f.write(json.dumps(r, sort_keys=True) + "\n")
                f.flush()
                os.fsync(f.fileno())
            existing.update({r["candidate_id"]: r for r in rows})
            done = start + len(batch)
            rate = done / max(time.time() - started, 1e-9)
            print(f"    {modality} stage {stage}: {len(existing):,} candidates "
                  f"({rate * 60:.0f}/min, about "
                  f"{(len(pending) - done) / max(rate, 1e-9) / 3600:.1f} h left)", flush=True)
        return True

    def passing(rows, rho_max, tar_min):
        return [r for r in rows if r["rho_max"] <= rho_max and r["TAR_background"] >= tar_min]

    def rank(rows):
        return min(rows, key=lambda r: (r["rho_max"], -r["TAR_background"], r["candidate_id"]))

    torch.use_deterministic_algorithms(True, warn_only=False)    # as the study
    try:
        if not run(1, N_INIT, 1):
            return False
        p1 = passing([existing[c] for c in range(1, N_INIT + 1)],
                     STAGE1_RHO_MAX, STAGE1_TAR_MIN)
        if p1:
            winner = dict(rank(p1), selected_stage=1)
        else:
            if not run(N_INIT + 1, N_MAX, 2):
                return False
            p2 = passing([existing[c] for c in range(1, N_MAX + 1)],
                         STAGE2_RHO_MAX, STAGE2_TAR_MIN)
            if not p2:
                atomic_json(out_dir / "FAILED.json", {"modality": modality, "status": "FAIL"})
                raise RuntimeError(f"{modality}: no candidate passed the Stage-2 gates")
            winner = dict(rank(p2), selected_stage=2)
        atomic_json(selected, winner)
    finally:
        torch.use_deterministic_algorithms(True, warn_only=True)
    print(f"  {modality}: candidate {winner['candidate_id']} selected "
          f"(rho_max {winner['rho_max']:.4f}, TAR {winner['TAR_background']:.4f})")
    if modality == "voice" and REF["voice_key_sha256"]:
        exact("voice selected_key.json sha256", sha256_file(selected), REF["voice_key_sha256"])
    return True


# ------------------------------------------------------- stage 6: Stage B
def stage_b(modality):
    """The 80-setting development sweep, then the sealed selection rule."""
    seal_path = DIR["seal"] / f"operating_point_{modality}.json"
    if seal_path.is_file():
        print(f"  {modality}: operating point already sealed")
        return json.loads(seal_path.read_text())
    sweep_path = RUNS_IN / "sweep" / modality / "sweep_80.csv"
    if not sweep_path.is_file():
        subjects, enroll, probes = partition_data(modality, "development")
        d = len(enroll[subjects[0]])
        rows = []
        for o in GRID["o"]:
            frozen = load_iom_tensor(modality, o, d)
            for M in GRID["M"]:
                for q in GRID["q"]:
                    H_ = identity_histograms(modality, subjects, enroll, probes,
                                             M, q, o, None, frozen=frozen)
                    rows.append(sweep_row(modality, H_))
            print(f"    {modality} o={o}: {len(rows)} settings", flush=True)
        sweep_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).sort_values(["o", "M", "q"]).to_csv(sweep_path, index=False)
    rows = pd.read_csv(sweep_path).to_dict("records")
    sealed = {(r["M"], r["q"], r["o"]): r for r in PAPER["sweeps"][modality]}
    fields = ("EER", "c_EER", "c_tau_DEV", "TAR_DEV", "FMR_DEV", "Dsys_DEV")
    same = sum(1 for r in rows if (r["M"], r["q"], r["o"]) in sealed
               and all(abs(float(r[f]) - float(sealed[(r["M"], r["q"], r["o"])][f])) <= 1e-9
                       for f in fields))
    exact(f"{modality} sweep rows equal to the sealed sweep", same, len(sealed))

    def eligible(rule):
        return [r for r in rows if r["EER"] <= rule["eer_max"] and r["TAR_DEV"] >= rule["tar_min"]]

    rule = dict(SELECTION_RULE_VOICE, rule_id="min_dsys_subject_to_eer_floor_v1")
    ok = eligible(rule)
    if not ok and modality == "face":
        rule = dict(FACE_FLOOR, rule_id="min_dsys_subject_to_eer_floor_v1_face",
                    deviation_reason="face cannot meet the voice floor; the study's "
                                     "registered face floor is applied")
        ok = eligible(rule)
    if not ok:
        raise RuntimeError(f"{modality}: no setting meets the floor {rule}")
    best = min(ok, key=lambda r: (r["Dsys_DEV"], r["EER"], r["M"], r["q"], r["o"]))
    chosen = {"modality": modality, "rule": rule, "M": int(best["M"]),
              "q": int(best["q"]), "o": int(best["o"]),
              "development": {"EER": float(best["EER"]), "TAR_DEV": float(best["TAR_DEV"]),
                              "Dsys_DEV": float(best["Dsys_DEV"]),
                              "c_tau_DEV": int(best["c_tau_DEV"])},
              "n_eligible": len(ok), "sweep_sha256": sha256_file(sweep_path)}
    atomic_json(seal_path, chosen)
    paper = PAPER["heldout"][modality]
    exact(f"{modality} sealed operating point (M, q, o, tau)",
          [chosen["M"], chosen["q"], chosen["o"], chosen["development"]["c_tau_DEV"]],
          [paper["M"], paper["q"], paper["o"], paper["c_tau_carried_from_DEV"]])
    return chosen


# --------------------------------------------------- stage 7: evaluation
def evaluate_partition(modality, subjects, enroll, probes, chosen, tag):
    M, q, o = chosen["M"], chosen["q"], chosen["o"]
    tau = chosen["development"]["c_tau_DEV"]
    H_ = identity_histograms(modality, subjects, enroll, probes, M, q, o, tau)
    gen, imp, _, _ = _apply_weights(H_, np.ones(H_["n"]))
    point = _metrics(H_, np.ones(H_["n"]))
    eer, c_eer = eer_from_histograms(gen, imp)      # the ablation checks against this
    return {"modality": modality, "M": M, "q": q, "o": o, "protocol": PROTOCOL_VERSION,
            "c_tau_carried_from_DEV": tau, "n_subjects": H_["n"],
            "n_genuine": int(gen.sum()), "n_impostor": int(imp.sum()),
            f"EER_{tag}": eer, f"c_EER_{tag}": int(c_eer),
            f"TAR_{tag}_at_dev_threshold": point["TAR"],
            f"FMR_{tag}_at_dev_threshold": point["FMR"],
            f"FNMR_{tag}_at_dev_threshold": 1.0 - point["TAR"],
            f"Dsys_{tag}": point["Dsys"],
            "development_reference": {**chosen["development"]},
            "selection_rule_id": chosen["rule"]["rule_id"]}


def _exact_metrics(label, result, paper, tag):
    for key in (f"EER_{tag}", f"TAR_{tag}_at_dev_threshold",
                f"FMR_{tag}_at_dev_threshold", f"Dsys_{tag}"):
        exact(f"{label} {key}", round(result[key], 12), round(paper[key], 12), tol=1e-9)


def stage_heldout(modality):
    path = HELDOUT_RESULTS[modality]
    if path.is_file():
        print(f"  {modality}: held-out result already written")
        return
    chosen = json.loads((DIR["seal"] / f"operating_point_{modality}.json").read_text())
    data = face_heldout_data() if modality == "face" else voice_heldout_data()
    result = dict(evaluate_partition(modality, *data, chosen, "HOLDOUT"), partition="evaluation")
    path.parent.mkdir(parents=True, exist_ok=True)
    atomic_json(path, result)
    _exact_metrics(f"{modality} held-out", result, PAPER["heldout"][modality], "HOLDOUT")


def extract_vctk_embeddings():
    if verified_stage("vctk_embeddings", [(VCTK_EMB, "sha256")]):
        return
    manifest = pd.read_csv(VCTK_MANIFEST, sep="\t", dtype={"speaker_id": str}) \
        .sort_values(["speaker_id", "rank"]).reset_index(drop=True)
    model = load_ecapa()
    embs = []
    for r in manifest.itertuples(index=False):
        wav, _ = load_audio(DIR["audio_vctk"] / r.relative_path)
        embs.append(ecapa_embed_waveform(model, wav.numpy()))
    np.savez(VCTK_EMB, embedding=np.stack(embs).astype(np.float32),
             speaker_id=manifest.speaker_id.to_numpy().astype(str),
             rank=manifest["rank"].to_numpy(),
             embedding_index=np.arange(len(embs)))
    exact("VCTK embeddings file sha256", sha256_file(VCTK_EMB), REF["vctk_embeddings_sha256"])
    mark_stage("vctk_embeddings", {"n": len(embs), "sha256": sha256_file(VCTK_EMB),
                                   "manifest_sha256": sha256_file(VCTK_MANIFEST)})


def stage_external():
    """VCTK, read only after both internal evaluations are written."""
    path = RUNS_IN / "external" / "voice" / "external_result.json"
    if path.is_file():
        print("  voice: external result already written")
        return
    if not all(HELDOUT_RESULTS[m].is_file() for m in ("voice", "face")):
        raise RuntimeError("the internal evaluation must finish before VCTK is read")
    if not stage_done("vctk_manifest"):
        download("vctk", VCTK_ARCHIVE, md5=REF["vctk_md5"])
    manifest = prepare_vctk()
    exact("VCTK speakers kept", int(manifest.speaker_id.nunique()), REF["vctk_speakers"])
    exact("VCTK utterances kept", int(len(manifest)), REF["vctk_utterances"])
    exact("VCTK manifest sha256", sha256_file(VCTK_MANIFEST), REF["vctk_manifest_sha256"])
    extract_vctk_embeddings()
    VCTK_ARCHIVE.unlink(missing_ok=True)
    chosen = json.loads((DIR["seal"] / "operating_point_voice.json").read_text())
    result = dict(evaluate_partition("voice", *vctk_external_data(), chosen, "EXT"),
                  corpus="VCTK-Corpus-0.92")
    path.parent.mkdir(parents=True, exist_ok=True)
    atomic_json(path, result)
    _exact_metrics("voice external", result, PAPER["external"], "EXT")


# ----------------------------------------- stage 9: against the paper
def _ci_row(label, paper_point, fresh, scale=100.0, unit="%"):
    p, lo, hi = fresh
    return {"quantity": label, "paper": round(paper_point * scale, 4),
            "fresh": round(p * scale, 4),
            "fresh_95ci": [round(lo * scale, 4), round(hi * scale, 4)], "unit": unit,
            "verdict": ("paper inside fresh interval" if lo <= paper_point <= hi
                        else "paper OUTSIDE fresh interval")}


def compare_with_paper(intervals, ablation, inversion, revocation):
    rows, claims = [], []
    names = {"EER": "EER_{}", "TAR": "TAR_{}_at_dev_threshold",
             "FMR": "FMR_{}_at_dev_threshold", "Dsys": "Dsys_{}"}
    for part, paper_res in (("voice/evaluation", PAPER["heldout"]["voice"]),
                            ("face/evaluation", PAPER["heldout"]["face"]),
                            ("voice/external", PAPER["external"])):
        if part not in intervals["partitions"]:
            continue
        tag = "HOLDOUT" if part.endswith("evaluation") else "EXT"
        for k, name in names.items():
            m = intervals["partitions"][part]["metrics"][k]
            scale, unit = (1.0, "") if k == "Dsys" else (100.0, "%")
            rows.append(_ci_row(f"{part} {k}", paper_res[name.format(tag)],
                                (m["estimate"], m["lower"], m["upper"]), scale, unit))
    for mod in ("voice", "face"):
        for arm in ARMS:
            a = ablation["modalities"][mod]["arms"][arm]
            pa = PAPER["ablation"]["modalities"][mod]["arms"][arm]
            r = revocation["modalities"][mod]["arms"][arm]
            pr = PAPER["revocation"]["modalities"][mod]["arms"][arm]
            rows += [_ci_row(f"ablation {mod} {arm} EER", pa["EER"], (a["EER"], *a["EER_CI"])),
                     _ci_row(f"ablation {mod} {arm} Dsys", pa["Dsys"],
                             (a["Dsys"], *a["Dsys_CI"]), 1.0, ""),
                     _ci_row(f"revocation {mod} {arm} PRAR", pr["PRAR"],
                             (r["PRAR"], *r["PRAR_CI"])),
                     _ci_row(f"inversion {mod} {arm} cosine", pr["cos_to_true"],
                             (r["cos_to_true"], *r["cos_CI"]), 1.0, "")]
        rv, iv = revocation["modalities"][mod]["arms"], inversion["modalities"][mod]["arms"]
        lo, hi = ablation["modalities"][mod]["contrasts_vs_polyiom"]["randproj_iom"]["delta_EER_CI"]
        claims += [
            (f"{mod}: hashing alone never revokes (PRAR 100%)", rv["iom_only"]["PRAR"] == 1.0),
            (f"{mod}: keyed compression revokes for the median key",
             rv["polyiom"]["per_key_median"] < 0.5 and rv["randproj_iom"]["per_key_median"] < 0.5),
            (f"{mod}: cosine polynomial < linear map < hashing alone",
             iv["polyiom"]["cos_to_true"] < iv["randproj_iom"]["cos_to_true"]
             < iv["iom_only"]["cos_to_true"]),
            (f"{mod}: the attack reaches the threshold in every arm (SAR 100%)",
             all(iv[a]["SAR"] == 1.0 for a in ARMS)),
            (f"{mod}: the polynomial costs EER against the linear map (firm)", hi < 0)]
    return rows, claims


def write_report(rows, claims):
    exact_rows = json.loads((DIR["seal"] / "exact_checks.json").read_text()) \
        if (DIR["seal"] / "exact_checks.json").is_file() else []
    n_same = sum(r["verdict"] == "identical" for r in exact_rows)
    lines = ["# AttackAware PolyIoM v1.1.4 - fresh run on Kaggle", "",
             "## 1. Stage by stage: does the fresh run give the study's output?", "",
             f"{n_same} of {len(exact_rows)} checks are identical.", "",
             "| Check | Fresh | Study | Verdict |", "|---|---|---|---|"]
    lines += [f"| {r['check']} | {r['fresh']} | {r['study']} | {r['verdict']} |"
              for r in exact_rows]
    inside = sum(r["verdict"].startswith("paper inside") for r in rows)
    lines += ["", "## 2. Results against the paper", "",
              f"{inside} of {len(rows)} paper values lie inside the fresh 95% interval.", "",
              "| Quantity | Paper | Fresh | Fresh 95% interval | Verdict |",
              "|---|---|---|---|---|"]
    lines += [f"| {r['quantity']} | {r['paper']}{r['unit']} | {r['fresh']}{r['unit']} | "
              f"[{r['fresh_95ci'][0]}, {r['fresh_95ci'][1]}] | {r['verdict']} |" for r in rows]
    lines += ["", "## 3. The paper's findings", "", "| Finding | Holds in the fresh run |",
              "|---|---|"] + [f"| {c} | {'yes' if ok else 'NO'} |" for c, ok in claims]
    text = "\n".join(lines) + "\n"
    (PROJECT / "REPLICATION_REPORT.md").write_text(text)
    atomic_json(PROJECT / "replication_report.json",
                {"exact": exact_rows, "results": rows,
                 "findings": [{"finding": c, "holds": bool(ok)} for c, ok in claims]})
    print(text)
    return text


# ---------------------------------------------------------------- status
def status():
    names = [("environment", DIR["seal"] / "environment.json"),
             ("face embeddings", stage_file("lfw_embeddings")),
             ("face split and trials", stage_file("lfw_face_trials")),
             ("voice embeddings", stage_file("libri_embeddings")),
             ("Stage A voice key", RUNS_IN / "key_search/voice/selected_key.json"),
             ("Stage A face key", RUNS_IN / "key_search/face/selected_key.json"),
             ("Stage B voice", DIR["seal"] / "operating_point_voice.json"),
             ("Stage B face", DIR["seal"] / "operating_point_face.json"),
             ("held-out", HELDOUT_RESULTS["face"]),
             ("external", RUNS_IN / "external/voice/external_result.json"),
             ("revocation", RUNS_IN / "revocation/revocation_result.json"),
             ("report", PROJECT / "REPLICATION_REPORT.md")]
    for label, path in names:
        print(f"  {'done' if Path(path).is_file() else '....'}  {label}")


print("Fresh pipeline ready.")


In [ ]:
%%writefile {RUNTIME}/paper_reference.json
{
 "heldout": {
  "face": {
   "Dsys_HOLDOUT": 0.15080275075742733,
   "EER_HOLDOUT": 0.04799467957362694,
   "FMR_HOLDOUT_at_dev_threshold": 0.000406421459053038,
   "FNMR_HOLDOUT_at_dev_threshold": 0.17760617760617758,
   "M": 256,
   "TAR_HOLDOUT_at_dev_threshold": 0.8223938223938224,
   "c_EER_HOLDOUT": 31,
   "c_tau_carried_from_DEV": 68,
   "development_reference": {
    "Dsys_DEV": 0.1331002909486603,
    "EER": 0.0283906723831433,
    "TAR_DEV": 0.8625954198473282,
    "c_tau_DEV": 68
   },
   "modality": "face",
   "n_genuine": 259,
   "n_impostor": 14763,
   "n_subjects": 58,
   "o": 1,
   "partition": "evaluation",
   "protocol": "1.1.1",
   "q": 32,
   "selection_rule_id": "min_dsys_subject_to_eer_floor_v1_face_eer0.03_tar0.85"
  },
  "voice": {
   "Dsys_HOLDOUT": 0.08843067398431459,
   "EER_HOLDOUT": 0.0192823016202742,
   "FMR_HOLDOUT_at_dev_threshold": 0.0018451300665456745,
   "FNMR_HOLDOUT_at_dev_threshold": 0.06034482758620685,
   "M": 128,
   "TAR_HOLDOUT_at_dev_threshold": 0.9396551724137931,
   "c_EER_HOLDOUT": 26,
   "c_tau_carried_from_DEV": 37,
   "development_reference": {
    "Dsys_DEV": 0.0525185719007491,
    "EER": 0.0095238095238095,
    "TAR_DEV": 0.9523809523809524,
    "c_tau_DEV": 37
   },
   "modality": "voice",
   "n_genuine": 580,
   "n_impostor": 33060,
   "n_subjects": 58,
   "o": 1,
   "partition": "evaluation",
   "protocol": "1.1.1",
   "q": 32,
   "selection_rule_id": "min_dsys_subject_to_eer_floor_v1"
  }
 },
 "external": {
  "Dsys_EXT": 0.08030046844714253,
  "EER_EXT": 0.027977167019027485,
  "FMR_EXT_at_dev_threshold": 0.003769808173477898,
  "FNMR_EXT_at_dev_threshold": 0.10999999999999999,
  "M": 128,
  "TAR_EXT_at_dev_threshold": 0.89,
  "c_EER_EXT": 28,
  "c_tau_carried_from_DEV": 37,
  "corpus": "VCTK-Corpus-0.92",
  "development_reference": {
   "Dsys_DEV": 0.0525185719007491,
   "EER": 0.0095238095238095,
   "TAR_DEV": 0.9523809523809524,
   "c_tau_DEV": 37
  },
  "modality": "voice",
  "n_genuine": 1100,
  "n_impostor": 119900,
  "n_subjects": 110,
  "o": 1,
  "operating_point_sha256": "027e18fa7fa5de062c4a3a538b1a2a36391f1626bbb72e577ee510c94bffab58",
  "protocol": "1.1.1",
  "q": 32,
  "selection_rule_id": "min_dsys_subject_to_eer_floor_v1",
  "vctk_embeddings_sha256": "5fd9c3b6e71282bbbe890e2c7dfb01b4de00c7e4acac5ec156995709e495007a",
  "vctk_manifest_sha256": "3025b1d448e318ccde93c3becab392c15f075b128c260bcd3dbc4c61742a9705",
  "voice_key_sha256": "e5b716fa56c5f7c800b1efb3fdfdda4be36ff8f3d1ba95329de4dfc359337678"
 },
 "ablation": {
  "analysis": "hardening_ablation",
  "arms": {
   "iom_only": "z ----------------> IoM-GRP   (the practical baseline)",
   "polyiom": "z -> P_K*(z; o*) -> IoM-GRP   (the proposed method)",
   "randproj_iom": "z -> A z ----------> IoM-GRP   (isolates the polynomial)"
  },
  "changes_any_seal": false,
  "confidence": 0.95,
  "master_seed": 2026,
  "method_id": "identity_cluster_percentile_v1_paired",
  "modalities": {
   "face": {
    "M": 256,
    "arms": {
     "iom_only": {
      "Dsys": 0.11696539072288004,
      "Dsys_CI": [
       0.10122273256555415,
       0.2575621193668769
      ],
      "EER": 0.04247104247104247,
      "EER_CI": [
       0.013369041102825092,
       0.08592738876244022
      ],
      "TAR_at_matched_FMR_descriptive": 0.8764478764478765
     },
     "polyiom": {
      "Dsys": 0.15080275075742733,
      "Dsys_CI": [
       0.12497592639109305,
       0.28061138210717557
      ],
      "EER": 0.04799467957362694,
      "EER_CI": [
       0.017368262392620758,
       0.09737455861197072
      ],
      "TAR_at_matched_FMR_descriptive": 0.8223938223938224
     },
     "randproj_iom": {
      "Dsys": 0.09614022644879111,
      "Dsys_CI": [
       0.09229079332453327,
       0.2327258253452317
      ],
      "EER": 0.03861003861003861,
      "EER_CI": [
       0.01689047001547002,
       0.08375024630541868
      ],
      "TAR_at_matched_FMR_descriptive": 0.8841698841698842
     }
    },
    "c_tau_carried_from_DEV": 68,
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_Dsys": -0.03383736003454729,
      "delta_Dsys_CI": [
       -0.1318095991898863,
       0.09417188135517908
      ],
      "delta_EER": -0.0055236371025844735,
      "delta_EER_CI": [
       -0.021176719840478565,
       0.00416097968719376
      ]
     },
     "randproj_iom": {
      "delta_Dsys": -0.05466252430863622,
      "delta_Dsys_CI": [
       -0.13927409682400674,
       0.06969930050428631
      ],
      "delta_EER": -0.009384640963588334,
      "delta_EER_CI": [
       -0.025751072961373397,
       0.004537195270363405
      ]
     }
    },
    "d": 512,
    "frozen_tensor_is_gaussian": true,
    "k": 128,
    "n_subjects": 58,
    "o": 1,
    "polyiom_reproduces_sealed_heldout": true,
    "q": 32,
    "reference_FMR_for_matched_TAR": 0.000406421459053038
   },
   "voice": {
    "M": 128,
    "arms": {
     "iom_only": {
      "Dsys": 0.07282230188335274,
      "Dsys_CI": [
       0.05808619133849283,
       0.19156131357397968
      ],
      "EER": 0.0027298850574712643,
      "EER_CI": [
       0.00034276508115965455,
       0.00567936822790284
      ],
      "TAR_at_matched_FMR_descriptive": 0.996551724137931
     },
     "polyiom": {
      "Dsys": 0.08843067398431459,
      "Dsys_CI": [
       0.06940323384881929,
       0.22061115253113128
      ],
      "EER": 0.0192823016202742,
      "EER_CI": [
       0.011097485553269934,
       0.02950966879593137
      ],
      "TAR_at_matched_FMR_descriptive": 0.9396551724137931
     },
     "randproj_iom": {
      "Dsys": 0.11316196400960966,
      "Dsys_CI": [
       0.08321051702917112,
       0.22090514142909964
      ],
      "EER": 0.007002419842710224,
      "EER_CI": [
       0.0030539788016163347,
       0.013793103448275862
      ],
      "TAR_at_matched_FMR_descriptive": 0.9896551724137931
     }
    },
    "c_tau_carried_from_DEV": 37,
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_Dsys": -0.015608372100961845,
      "delta_Dsys_CI": [
       -0.12474995222638989,
       0.08654315783644746
      ],
      "delta_EER": -0.016552416562802933,
      "delta_EER_CI": [
       -0.025631959284318967,
       -0.009031486540190231
      ]
     },
     "randproj_iom": {
      "delta_Dsys": 0.024731290025295075,
      "delta_Dsys_CI": [
       -0.09236389690750378,
       0.10568412545913647
      ],
      "delta_EER": -0.012279881777563974,
      "delta_EER_CI": [
       -0.02187452937884844,
       -0.0028518109672935545
      ]
     }
    },
    "d": 192,
    "frozen_tensor_is_gaussian": true,
    "k": 48,
    "n_subjects": 58,
    "o": 1,
    "polyiom_reproduces_sealed_heldout": true,
    "q": 32,
    "reference_FMR_for_matched_TAR": 0.0018451300665456745
   }
  },
  "n_bootstrap": 2000,
  "schema": 1,
  "source_heldout_sha256": {
   "face": "90b00186c204fc9f112a4d111717a22181ca0cefea027fdaf32db03782a88108",
   "voice": "33dc476b920dabf5b9ee47951cb806b69fde87ee790875d7b95cca699e99ee15"
  },
  "threshold_policy": "none fitted; EER and Dsys are threshold-free"
 },
 "inversion": {
  "adversary": "level 3: holds K*, R, the algorithm and the template",
  "analysis": "full_knowledge_inversion",
  "attack": "annealed-softmax gradient descent on the unit sphere, identical budget for every arm",
  "budget": {
   "lr": 0.05,
   "restarts": 5,
   "steps": 800,
   "temperature": [
    1.0,
    0.05
   ]
  },
  "changes_any_seal": false,
  "confidence": 0.95,
  "master_seed": 2026,
  "measures": "practical inversion resistance under this budget; NOT information-theoretic irreversibility",
  "method_id": "identity_cluster_percentile_v1_paired",
  "modalities": {
   "face": {
    "M": 256,
    "arms": {
     "iom_only": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 8.0,
      "cos_chance": 0.028728477656841278,
      "cos_to_true": 0.8758925795555115,
      "cross_key_collisions": 129.93103448275863,
      "cross_key_impostor_mean": 10.34543228149414,
      "mean_collisions": 256.0,
      "random_guess_collisions": 8.431034482758621
     },
     "polyiom": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 8.0,
      "cos_chance": 0.028728477656841278,
      "cos_to_true": 0.41843363642692566,
      "cross_key_collisions": 208.01724137931035,
      "cross_key_impostor_mean": 12.934059143066406,
      "mean_collisions": 256.0,
      "random_guess_collisions": 11.10344827586207
     },
     "randproj_iom": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 8.0,
      "cos_chance": 0.028728477656841278,
      "cos_to_true": 0.49227818846702576,
      "cross_key_collisions": 198.20689655172413,
      "cross_key_impostor_mean": 10.888082504272461,
      "mean_collisions": 238.82758620689654,
      "random_guess_collisions": 8.517241379310345
     }
    },
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_SAR": 0.0,
      "delta_SAR_CI": [
       0.0,
       0.0
      ]
     },
     "randproj_iom": {
      "delta_SAR": 0.0,
      "delta_SAR_CI": [
       0.0,
       0.0
      ]
     }
    },
    "d": 512,
    "k": 128,
    "n_subjects": 58,
    "tau": 68
   },
   "voice": {
    "M": 128,
    "arms": {
     "iom_only": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 4.0,
      "cos_chance": 0.12024644017219543,
      "cos_to_true": 0.9128499031066895,
      "cross_key_collisions": 73.8103448275862,
      "cross_key_impostor_mean": 7.222020626068115,
      "mean_collisions": 128.0,
      "random_guess_collisions": 3.810344827586207
     },
     "polyiom": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 4.0,
      "cos_chance": 0.12024644017219543,
      "cos_to_true": 0.22194789350032806,
      "cross_key_collisions": 109.41379310344827,
      "cross_key_impostor_mean": 10.227465629577637,
      "mean_collisions": 128.0,
      "random_guess_collisions": 9.258620689655173
     },
     "randproj_iom": {
      "SAR": 1.0,
      "SAR_CI": [
       0.9999999999999999,
       1.0000000000000002
      ],
      "chance_collisions": 4.0,
      "cos_chance": 0.12024644017219543,
      "cos_to_true": 0.4896182715892792,
      "cross_key_collisions": 107.13793103448276,
      "cross_key_impostor_mean": 7.620084762573242,
      "mean_collisions": 125.72413793103448,
      "random_guess_collisions": 4.172413793103448
     }
    },
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_SAR": 0.0,
      "delta_SAR_CI": [
       0.0,
       0.0
      ]
     },
     "randproj_iom": {
      "delta_SAR": 0.0,
      "delta_SAR_CI": [
       0.0,
       0.0
      ]
     }
    },
    "d": 192,
    "k": 48,
    "n_subjects": 58,
    "tau": 37
   }
  },
  "n_bootstrap": 2000,
  "schema": 1
 },
 "revocation": {
  "analysis": "post_revocation_acceptance",
  "changes_any_seal": false,
  "confidence": 0.95,
  "fresh_key_construction": "resampled from K*'s own coefficient and exponent values; NOT passed through Stage A's inversion gate",
  "master_seed": 2026,
  "method_id": "two_level_key_and_identity_percentile_v1_paired",
  "modalities": {
   "face": {
    "M": 256,
    "arms": {
     "iom_only": {
      "PRAR": 1.0,
      "PRAR_CI": [
       0.9999999999999998,
       1.0000000000000002
      ],
      "attack_collisions": 256.0,
      "cos_CI": [
       0.8734827009511407,
       0.8782054836636988
      ],
      "cos_to_true": 0.8758925795555115,
      "impostor_floor_collisions": 10.47748031616211,
      "keys_failing_outright": 1.0,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0
      ],
      "per_key_max": 1.0,
      "per_key_median": 1.0,
      "per_key_min": 1.0
     },
     "polyiom": {
      "PRAR": 0.2560344934463501,
      "PRAR_CI": [
       0.13017241379310346,
       0.3858081896551722
      ],
      "attack_collisions": 256.0,
      "cos_CI": [
       0.40754718234569864,
       0.4295518100903979
      ],
      "cos_to_true": 0.41843363642692566,
      "impostor_floor_collisions": 13.807577061653138,
      "keys_failing_outright": 0.25,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       0.017241379246115685,
       0.0,
       1.0,
       0.0,
       0.08620689809322357,
       0.0,
       0.0517241396009922,
       0.0,
       0.0,
       0.2586206793785095,
       0.0,
       0.0,
       0.931034505367279,
       1.0,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0,
       1.0,
       0.0,
       1.0,
       0.0,
       0.0,
       0.03448275849223137,
       0.0,
       0.0,
       0.0,
       1.0,
       0.0,
       0.0,
       0.8620689511299133,
       0.0,
       0.0,
       1.0,
       0.0,
       0.0,
       1.0,
       1.0
      ],
      "per_key_max": 1.0,
      "per_key_median": 0.0,
      "per_key_min": 0.0
     },
     "randproj_iom": {
      "PRAR": 0.028879310935735703,
      "PRAR_CI": [
       0.01767241379310345,
       0.04267241379310345
      ],
      "attack_collisions": 238.82758620689654,
      "cos_CI": [
       0.4852230567228178,
       0.4993468746671389
      ],
      "cos_to_true": 0.49227818846702576,
      "impostor_floor_collisions": 10.923623728752137,
      "keys_failing_outright": 0.0,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       0.0,
       0.03448275849223137,
       0.017241379246115685,
       0.0517241396009922,
       0.0,
       0.017241379246115685,
       0.0517241396009922,
       0.017241379246115685,
       0.08620689809322357,
       0.017241379246115685,
       0.0,
       0.0,
       0.03448275849223137,
       0.0517241396009922,
       0.03448275849223137,
       0.03448275849223137,
       0.017241379246115685,
       0.0517241396009922,
       0.0517241396009922,
       0.017241379246115685,
       0.017241379246115685,
       0.0517241396009922,
       0.06896551698446274,
       0.06896551698446274,
       0.08620689809322357,
       0.0517241396009922,
       0.017241379246115685,
       0.03448275849223137,
       0.0,
       0.017241379246115685,
       0.0,
       0.017241379246115685,
       0.03448275849223137,
       0.0,
       0.017241379246115685,
       0.017241379246115685,
       0.017241379246115685,
       0.0,
       0.03448275849223137,
       0.017241379246115685
      ],
      "per_key_max": 0.08620689809322357,
      "per_key_median": 0.017241379246115685,
      "per_key_min": 0.0
     }
    },
    "bootstrap_axes": "keys and identities",
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_PRAR": 0.7439655065536499,
      "delta_PRAR_CI": [
       0.6141918103448274,
       0.8698275862068965
      ],
      "delta_cos": 0.4574589431285858,
      "delta_cos_CI": [
       0.44661127680848395,
       0.46815001432751796
      ]
     },
     "randproj_iom": {
      "delta_PRAR": -0.2271551787853241,
      "delta_PRAR_CI": [
       -0.3586422413793103,
       -0.09995689655172435
      ],
      "delta_cos": 0.0738445520401001,
      "delta_cos_CI": [
       0.059983370157665375,
       0.08756925154091992
      ]
     }
    },
    "d": 512,
    "k": 128,
    "n_subjects": 58,
    "tau": 68
   },
   "voice": {
    "M": 128,
    "arms": {
     "iom_only": {
      "PRAR": 1.0,
      "PRAR_CI": [
       0.9999999999999998,
       1.0000000000000002
      ],
      "attack_collisions": 128.0,
      "cos_CI": [
       0.9098439594794965,
       0.9158417161939474
      ],
      "cos_to_true": 0.9128499031066895,
      "impostor_floor_collisions": 6.910798561573029,
      "keys_failing_outright": 1.0,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0,
       1.0
      ],
      "per_key_max": 1.0,
      "per_key_median": 1.0,
      "per_key_min": 1.0
     },
     "polyiom": {
      "PRAR": 0.07500000298023224,
      "PRAR_CI": [
       0.021120689655172414,
       0.14440732758620686
      ],
      "attack_collisions": 128.0,
      "cos_CI": [
       0.1960923122788041,
       0.24785558165869961
      ],
      "cos_to_true": 0.22194789350032806,
      "impostor_floor_collisions": 9.421748328208924,
      "keys_failing_outright": 0.05,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       0.0,
       0.017241379246115685,
       0.0,
       0.0,
       0.0,
       0.017241379246115685,
       0.0,
       0.0,
       0.03448275849223137,
       0.8448275923728943,
       0.0,
       0.017241379246115685,
       0.0,
       0.27586206793785095,
       0.03448275849223137,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0517241396009922,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0,
       0.03448275849223137,
       0.0,
       0.0517241396009922,
       0.8965517282485962,
       0.06896551698446274,
       0.43103447556495667,
       0.03448275849223137,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0,
       0.0,
       0.06896551698446274,
       0.12068965286016464
      ],
      "per_key_max": 0.8965517282485962,
      "per_key_median": 0.0,
      "per_key_min": 0.0
     },
     "randproj_iom": {
      "PRAR": 0.05732758715748787,
      "PRAR_CI": [
       0.04008620689655172,
       0.07629310344827586
      ],
      "attack_collisions": 125.72413793103448,
      "cos_CI": [
       0.4815505725171031,
       0.4982657071973743
      ],
      "cos_to_true": 0.4896182715892792,
      "impostor_floor_collisions": 7.5688445448875425,
      "keys_failing_outright": 0.0,
      "n_keys_degenerate": 0,
      "n_keys_usable": 40,
      "per_key_PRAR": [
       0.06896551698446274,
       0.0517241396009922,
       0.03448275849223137,
       0.08620689809322357,
       0.06896551698446274,
       0.0517241396009922,
       0.08620689809322357,
       0.0,
       0.06896551698446274,
       0.0517241396009922,
       0.0517241396009922,
       0.06896551698446274,
       0.06896551698446274,
       0.017241379246115685,
       0.0517241396009922,
       0.06896551698446274,
       0.017241379246115685,
       0.017241379246115685,
       0.06896551698446274,
       0.06896551698446274,
       0.017241379246115685,
       0.13793103396892548,
       0.017241379246115685,
       0.06896551698446274,
       0.08620689809322357,
       0.017241379246115685,
       0.0517241396009922,
       0.0,
       0.08620689809322357,
       0.08620689809322357,
       0.1034482792019844,
       0.06896551698446274,
       0.06896551698446274,
       0.017241379246115685,
       0.1034482792019844,
       0.017241379246115685,
       0.1034482792019844,
       0.06896551698446274,
       0.03448275849223137,
       0.06896551698446274
      ],
      "per_key_max": 0.13793103396892548,
      "per_key_median": 0.06896551698446274,
      "per_key_min": 0.0
     }
    },
    "bootstrap_axes": "keys and identities",
    "contrasts_vs_polyiom": {
     "iom_only": {
      "delta_PRAR": 0.925000011920929,
      "delta_PRAR_CI": [
       0.8555926724137931,
       0.9788793103448273
      ],
      "delta_cos": 0.6909019947052002,
      "delta_cos_CI": [
       0.664920701476713,
       0.7177548396690137
      ]
     },
     "randproj_iom": {
      "delta_PRAR": -0.01767241582274437,
      "delta_PRAR_CI": [
       -0.0879310344827586,
       0.03966594827586202
      ],
      "delta_cos": 0.2676703929901123,
      "delta_cos_CI": [
       0.23907936881376624,
       0.2965192413993245
      ]
     }
    },
    "d": 192,
    "k": 48,
    "n_subjects": 58,
    "tau": 37
   }
  },
  "n_bootstrap": 2000,
  "n_fresh_keys": 40,
  "question": "does an old template's reconstruction still get in after the subject re-enrols under a fully re-keyed system",
  "re_keys": {
   "iom_only": "fresh R",
   "polyiom": "fresh K** and fresh R",
   "randproj_iom": "fresh A and fresh R"
  },
  "schema": 1
 },
 "intervals": {
  "analysis": "heldout_and_external_confidence_intervals",
  "units": "percent for EER, TAR and FMR; fraction for Dsys",
  "method_id": "identity_cluster_percentile_v1",
  "n_bootstrap": 2000,
  "confidence": 0.95,
  "master_seed": 2026,
  "source": "transcribed from the held-out and external confidence-run console output; these intervals are reported in Sections 5.2 and 5.3 of the paper. The raw run output is not part of this package. figures.py checks every point estimate below against the sealed result JSONs before drawing, so a transcription error in a point would stop the build.",
  "partitions": {
   "voice/evaluation": {
    "EER": [
     1.928,
     1.092,
     2.919
    ],
    "TAR": [
     93.966,
     89.655,
     97.414
    ],
    "FMR": [
     0.185,
     0.028,
     0.48
    ],
    "Dsys": [
     0.088431,
     0.070803,
     0.218493
    ]
   },
   "face/evaluation": {
    "EER": [
     4.799,
     1.945,
     10.345
    ],
    "TAR": [
     82.239,
     72.767,
     89.933
    ],
    "FMR": [
     0.041,
     0.0,
     0.218
    ],
    "Dsys": [
     0.150803,
     0.122519,
     0.282221
    ]
   },
   "voice/external": {
    "EER": [
     2.798,
     1.992,
     3.807
    ],
    "TAR": [
     89.0,
     85.727,
     91.727
    ],
    "FMR": [
     0.377,
     0.202,
     0.622
    ],
    "Dsys": [
     0.0803,
     0.060294,
     0.155097
    ]
   }
  }
 },
 "baseline": {
  "analysis": "unprotected_cosine_baseline",
  "units": "percent",
  "source": "transcribed from the SCOREDUMP notebook console output, 2026-09-22; protected EERs reproduced the sealed held-out values exactly in that run",
  "bound_rule": "one-sided 95% Clopper-Pearson upper bound on the genuine error rate, k = floor(eer * n_genuine) errors; reported wherever eer < 1/n_genuine",
  "partitions": [
   {
    "modality": "voice",
    "partition": "development",
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "protected_eer": 0.9524,
    "unprotected_eer": 0.0,
    "cost_pp": 0.9524,
    "one_genuine_error_pct": 0.2381,
    "baseline_resolved": false,
    "unprotected_upper_bound_95": 0.7107
   },
   {
    "modality": "voice",
    "partition": "evaluation",
    "n_subjects": 58,
    "n_genuine": 580,
    "n_impostor": 33060,
    "protected_eer": 1.9282,
    "unprotected_eer": 0.0848,
    "cost_pp": 1.8434,
    "one_genuine_error_pct": 0.1724,
    "baseline_resolved": false,
    "unprotected_upper_bound_95": 0.5152
   },
   {
    "modality": "voice",
    "partition": "external",
    "n_subjects": 110,
    "n_genuine": 1100,
    "n_impostor": 119900,
    "protected_eer": 2.7977,
    "unprotected_eer": 0.0033,
    "cost_pp": 2.7944,
    "one_genuine_error_pct": 0.0909,
    "baseline_resolved": false,
    "unprotected_upper_bound_95": 0.272
   },
   {
    "modality": "face",
    "partition": "development",
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "protected_eer": 2.8391,
    "unprotected_eer": 2.6718,
    "cost_pp": 0.1673,
    "one_genuine_error_pct": 0.3817,
    "baseline_resolved": true,
    "unprotected_upper_bound_95": 4.9596
   },
   {
    "modality": "face",
    "partition": "evaluation",
    "n_subjects": 58,
    "n_genuine": 259,
    "n_impostor": 14763,
    "protected_eer": 4.7995,
    "unprotected_eer": 3.4749,
    "cost_pp": 1.3246,
    "one_genuine_error_pct": 0.3861,
    "baseline_resolved": true,
    "unprotected_upper_bound_95": 5.5041
   }
  ]
 },
 "config": {
  "protocol_version": "1.1.1",
  "master_seed": 2026,
  "internal_split_counts": {
   "background": 50,
   "development": 42,
   "evaluation": 58
  },
  "g": 5,
  "M_grid": [
   32,
   64,
   128,
   256
  ],
  "q_grid": [
   4,
   8,
   16,
   32
  ],
  "overlaps": [
   0,
   1,
   2,
   3,
   4
  ],
  "reference": {
   "M": 128,
   "q": 16,
   "o": 2
  },
  "target_fmr": 0.001,
  "candidate_budget": {
   "stage1": 10000,
   "maximum": 20000
  },
  "stage1": {
   "rho_max": 0.9,
   "tar_min": 0.95
  },
  "stage2": {
   "rho_max": 0.85,
   "tar_min": 0.9
  },
  "attack": {
   "restarts": 5,
   "iterations": 2000,
   "lr": 0.01,
   "betas": [
    0.9,
    0.999
   ],
   "eps": 1e-08,
   "init_std": 0.1
  },
  "lfw_cfp_overlap": "exact_case_sensitive_subject_identifier",
  "librispeech_validity": "decode_mono_16k_nonempty_finite_no_duration_gate",
  "vctk_validity": "mic1_post_resample_samples>=24000"
 },
 "ecapa": {
  "embedding_model_ckpt": {
   "relative_path": "models/ecapa_3d2520d6a433ff8e42cb7e80f0d5f23b912ca189/embedding_model.ckpt",
   "sha256": "0575cb64845e6b9a10db9bcb74d5ac32b326b8dc90352671d345e2ee3d0126a2"
  },
  "mean_var_norm_emb_ckpt": {
   "relative_path": "models/ecapa_3d2520d6a433ff8e42cb7e80f0d5f23b912ca189/mean_var_norm_emb.ckpt",
   "sha256": "cd70225b05b37be64fc5a95e24395d804231d43f74b2e1e5a513db7b69b34c33"
  },
  "repo_id": "speechbrain/spkrec-ecapa-voxceleb",
  "revision": "3d2520d6a433ff8e42cb7e80f0d5f23b912ca189"
 },
 "cfp": {
  "id_order_evidence": "https://github.com/Fizza-Rubab/unmasking-face-embeddings/blob/main/experiments/eval_naming.py",
  "match_field": "lfw_identifier",
  "match_rule": "case_sensitive_space_to_underscore_v1",
  "metadata_filename": "cfp_subject_names.tsv",
  "metadata_format": "tsv",
  "metadata_relative_path": "protocol/cfp_authoritative_metadata/cfp_subject_names.tsv",
  "metadata_sha256": "dcbcd60c4b60ad7498d5ef2f7d5889b41700feb3ef3a9a93b392b7a38cdc2a9e",
  "provenance_status": "release_metadata_from_mirror_original_host_not_verified",
  "rows": 500,
  "scope": "exact_name_overlap_exclusion_not_alias_resolution",
  "source_distribution": "Kaggle mirror chinafax/cfpw-dataset version 1",
  "source_readme": "Release README describes 500 names in order; calls the file list_names.txt",
  "source_release_path": "cfp-dataset/Data/list_name.txt",
  "source_sha256": "e2ada6d692e05380ba9c643a61672f1d9b56f01aa098edaf45aae082e00f2f5c",
  "source_url": "https://www.kaggle.com/datasets/chinafax/cfpw-dataset/versions/1",
  "subject_id_field": "cfp_subject_id",
  "subject_name_field": "subject_name",
  "unique_subject_names": 500
 },
 "lfw_split": {
  "cfp_metadata_sha256": "dcbcd60c4b60ad7498d5ef2f7d5889b41700feb3ef3a9a93b392b7a38cdc2a9e",
  "eligible_before_overlap_removal": 900,
  "excluded_exact_overlap_count": 100,
  "excluded_sha256": "c978dce9c13d2e49822b5d3d1b0767057395a0e50e029d2c5346793a0c99b44d",
  "match_rule": "case_sensitive_space_to_underscore_v1",
  "selected_background": 50,
  "selected_development": 42,
  "selected_evaluation": 58,
  "split_sha256": "79c5aac6d3c19a01db3388f19342a4cef318443693387b347ad958b35f62e8b6"
 },
 "sweeps": {
  "face": [
   {
    "modality": "face",
    "M": 32,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 22,
    "EER": 0.06835433551384859,
    "c_EER": 15,
    "TAR_DEV": 0.37404580152671757,
    "FMR_DEV": 0.0003723701359150996,
    "FNMR_DEV": 0.6259541984732824,
    "Dsys_DEV": 0.18932000699359236
   },
   {
    "modality": "face",
    "M": 32,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 18,
    "EER": 0.06296420466267795,
    "c_EER": 10,
    "TAR_DEV": 0.5190839694656488,
    "FMR_DEV": 0.0003723701359150996,
    "FNMR_DEV": 0.48091603053435117,
    "Dsys_DEV": 0.09955848742919231
   },
   {
    "modality": "face",
    "M": 32,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 13,
    "EER": 0.04742761642350603,
    "c_EER": 7,
    "TAR_DEV": 0.6526717557251909,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.34732824427480913,
    "Dsys_DEV": 0.0328349499193155
   },
   {
    "modality": "face",
    "M": 32,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 11,
    "EER": 0.05059698571148953,
    "c_EER": 5,
    "TAR_DEV": 0.6908396946564885,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.30916030534351147,
    "Dsys_DEV": 0.05490473973748766
   },
   {
    "modality": "face",
    "M": 64,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 39,
    "EER": 0.04924595047477192,
    "c_EER": 28,
    "TAR_DEV": 0.7061068702290076,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.29389312977099236,
    "Dsys_DEV": 0.13877585031344952
   },
   {
    "modality": "face",
    "M": 64,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 32,
    "EER": 0.04172020255460661,
    "c_EER": 20,
    "TAR_DEV": 0.7290076335877863,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.2709923664122137,
    "Dsys_DEV": 0.1508322142702081
   },
   {
    "modality": "face",
    "M": 64,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 27,
    "EER": 0.0314470627281779,
    "c_EER": 14,
    "TAR_DEV": 0.6030534351145038,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.3969465648854962,
    "Dsys_DEV": 0.13787110061217783
   },
   {
    "modality": "face",
    "M": 64,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 21,
    "EER": 0.030891060854676466,
    "c_EER": 11,
    "TAR_DEV": 0.7251908396946565,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.2748091603053435,
    "Dsys_DEV": 0.07653922441850908
   },
   {
    "modality": "face",
    "M": 128,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 74,
    "EER": 0.03816793893129771,
    "c_EER": 56,
    "TAR_DEV": 0.7862595419847328,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.2137404580152672,
    "Dsys_DEV": 0.19272098806613272
   },
   {
    "modality": "face",
    "M": 128,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 59,
    "EER": 0.030426328676364682,
    "c_EER": 38,
    "TAR_DEV": 0.7709923664122137,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.2290076335877863,
    "Dsys_DEV": 0.16035725949526555
   },
   {
    "modality": "face",
    "M": 128,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 48,
    "EER": 0.030534351145038167,
    "c_EER": 27,
    "TAR_DEV": 0.7748091603053435,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.22519083969465647,
    "Dsys_DEV": 0.13175130978952954
   },
   {
    "modality": "face",
    "M": 128,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 37,
    "EER": 0.030063142022429556,
    "c_EER": 19,
    "TAR_DEV": 0.8091603053435115,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.19083969465648853,
    "Dsys_DEV": 0.06233247337094944
   },
   {
    "modality": "face",
    "M": 256,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 145,
    "EER": 0.031050134103569216,
    "c_EER": 110,
    "TAR_DEV": 0.8358778625954199,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.16412213740458015,
    "Dsys_DEV": 0.20257002841275012
   },
   {
    "modality": "face",
    "M": 256,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 118,
    "EER": 0.029096138953424052,
    "c_EER": 75,
    "TAR_DEV": 0.7900763358778626,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.20992366412213737,
    "Dsys_DEV": 0.1901859922697613
   },
   {
    "modality": "face",
    "M": 256,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 94,
    "EER": 0.03435114503816794,
    "c_EER": 50,
    "TAR_DEV": 0.7938931297709924,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.20610687022900764,
    "Dsys_DEV": 0.16964514114939327
   },
   {
    "modality": "face",
    "M": 256,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 83,
    "EER": 0.030341095758044254,
    "c_EER": 37,
    "TAR_DEV": 0.7404580152671756,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.2595419847328244,
    "Dsys_DEV": 0.20464674072489336
   },
   {
    "modality": "face",
    "M": 32,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 21,
    "EER": 0.06211493005773627,
    "c_EER": 15,
    "TAR_DEV": 0.4847328244274809,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.5152671755725191,
    "Dsys_DEV": 0.15969309388504266
   },
   {
    "modality": "face",
    "M": 32,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 17,
    "EER": 0.054510267713149124,
    "c_EER": 10,
    "TAR_DEV": 0.5916030534351145,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.40839694656488545,
    "Dsys_DEV": 0.16888935660651555
   },
   {
    "modality": "face",
    "M": 32,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 14,
    "EER": 0.05280063448002379,
    "c_EER": 7,
    "TAR_DEV": 0.5419847328244275,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.4580152671755725,
    "Dsys_DEV": 0.04344869096168857
   },
   {
    "modality": "face",
    "M": 32,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 12,
    "EER": 0.05605675071323926,
    "c_EER": 5,
    "TAR_DEV": 0.5114503816793893,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.4885496183206107,
    "Dsys_DEV": 0.036944574638872664
   },
   {
    "modality": "face",
    "M": 64,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 39,
    "EER": 0.037567544386311005,
    "c_EER": 29,
    "TAR_DEV": 0.6603053435114504,
    "FMR_DEV": 0.0005585552038726494,
    "FNMR_DEV": 0.3396946564885496,
    "Dsys_DEV": 0.10411382892085878
   },
   {
    "modality": "face",
    "M": 64,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 30,
    "EER": 0.03816793893129771,
    "c_EER": 19,
    "TAR_DEV": 0.7595419847328244,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.24045801526717558,
    "Dsys_DEV": 0.10493302496680654
   },
   {
    "modality": "face",
    "M": 64,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 26,
    "EER": 0.039440203562340966,
    "c_EER": 13,
    "TAR_DEV": 0.6717557251908397,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.3282442748091603,
    "Dsys_DEV": 0.07338610911217683
   },
   {
    "modality": "face",
    "M": 64,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 20,
    "EER": 0.03993200333568542,
    "c_EER": 9,
    "TAR_DEV": 0.7099236641221374,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.29007633587786263,
    "Dsys_DEV": 0.09173563869582423
   },
   {
    "modality": "face",
    "M": 128,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 75,
    "EER": 0.03296321998612075,
    "c_EER": 56,
    "TAR_DEV": 0.7290076335877863,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.2709923664122137,
    "Dsys_DEV": 0.17816393931648444
   },
   {
    "modality": "face",
    "M": 128,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 54,
    "EER": 0.03545714781401804,
    "c_EER": 36,
    "TAR_DEV": 0.8740458015267175,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.12595419847328249,
    "Dsys_DEV": 0.18812137651440347
   },
   {
    "modality": "face",
    "M": 128,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 45,
    "EER": 0.02548633341541492,
    "c_EER": 27,
    "TAR_DEV": 0.7900763358778626,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.20992366412213737,
    "Dsys_DEV": 0.09292717824618564
   },
   {
    "modality": "face",
    "M": 128,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 37,
    "EER": 0.0281273640052266,
    "c_EER": 18,
    "TAR_DEV": 0.7862595419847328,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.2137404580152672,
    "Dsys_DEV": 0.13203237812938495
   },
   {
    "modality": "face",
    "M": 256,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 140,
    "EER": 0.030534351145038167,
    "c_EER": 108,
    "TAR_DEV": 0.8625954198473282,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.13740458015267176,
    "Dsys_DEV": 0.21193079358415445
   },
   {
    "modality": "face",
    "M": 256,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 104,
    "EER": 0.030534351145038167,
    "c_EER": 72,
    "TAR_DEV": 0.8931297709923665,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.10687022900763354,
    "Dsys_DEV": 0.19669177850474584
   },
   {
    "modality": "face",
    "M": 256,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 86,
    "EER": 0.026111716951411606,
    "c_EER": 52,
    "TAR_DEV": 0.851145038167939,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.14885496183206104,
    "Dsys_DEV": 0.13923543436635827
   },
   {
    "modality": "face",
    "M": 256,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 68,
    "EER": 0.028390672383143363,
    "c_EER": 35,
    "TAR_DEV": 0.8625954198473282,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.13740458015267176,
    "Dsys_DEV": 0.13310029094866038
   },
   {
    "modality": "face",
    "M": 32,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 21,
    "EER": 0.057019918030070366,
    "c_EER": 15,
    "TAR_DEV": 0.6030534351145038,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.3969465648854962,
    "Dsys_DEV": 0.09115334104050016
   },
   {
    "modality": "face",
    "M": 32,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 16,
    "EER": 0.048423286101016996,
    "c_EER": 10,
    "TAR_DEV": 0.7061068702290076,
    "FMR_DEV": 0.0003723701359150996,
    "FNMR_DEV": 0.29389312977099236,
    "Dsys_DEV": 0.10142847633580007
   },
   {
    "modality": "face",
    "M": 32,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 13,
    "EER": 0.051891762009500395,
    "c_EER": 7,
    "TAR_DEV": 0.7061068702290076,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.29389312977099236,
    "Dsys_DEV": 0.1199243434548809
   },
   {
    "modality": "face",
    "M": 32,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 11,
    "EER": 0.04539514573937433,
    "c_EER": 5,
    "TAR_DEV": 0.7519083969465649,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.24809160305343514,
    "Dsys_DEV": 0.06960635188451729
   },
   {
    "modality": "face",
    "M": 64,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 37,
    "EER": 0.03396733336176383,
    "c_EER": 29,
    "TAR_DEV": 0.7557251908396947,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.2442748091603053,
    "Dsys_DEV": 0.17774895779904343
   },
   {
    "modality": "face",
    "M": 64,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 28,
    "EER": 0.03603211644187794,
    "c_EER": 19,
    "TAR_DEV": 0.8206106870229007,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.17938931297709926,
    "Dsys_DEV": 0.12346617694382983
   },
   {
    "modality": "face",
    "M": 64,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 23,
    "EER": 0.03741709423100988,
    "c_EER": 13,
    "TAR_DEV": 0.8091603053435115,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.19083969465648853,
    "Dsys_DEV": 0.08025066031412502
   },
   {
    "modality": "face",
    "M": 64,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 19,
    "EER": 0.03517746124183521,
    "c_EER": 9,
    "TAR_DEV": 0.8053435114503816,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.19465648854961837,
    "Dsys_DEV": 0.06038410976245591
   },
   {
    "modality": "face",
    "M": 128,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 72,
    "EER": 0.026717557251908396,
    "c_EER": 57,
    "TAR_DEV": 0.8282442748091603,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.1717557251908397,
    "Dsys_DEV": 0.28280418656365836
   },
   {
    "modality": "face",
    "M": 128,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 55,
    "EER": 0.03065437611023093,
    "c_EER": 39,
    "TAR_DEV": 0.8702290076335878,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.1297709923664122,
    "Dsys_DEV": 0.19493797168647964
   },
   {
    "modality": "face",
    "M": 128,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 41,
    "EER": 0.026717557251908396,
    "c_EER": 27,
    "TAR_DEV": 0.8969465648854962,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.10305343511450382,
    "Dsys_DEV": 0.136898772167983
   },
   {
    "modality": "face",
    "M": 128,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 36,
    "EER": 0.022900763358778626,
    "c_EER": 19,
    "TAR_DEV": 0.8778625954198473,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.12213740458015265,
    "Dsys_DEV": 0.21001707454875304
   },
   {
    "modality": "face",
    "M": 256,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 140,
    "EER": 0.026717557251908396,
    "c_EER": 110,
    "TAR_DEV": 0.8778625954198473,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.12213740458015265,
    "Dsys_DEV": 0.24167604337227935
   },
   {
    "modality": "face",
    "M": 256,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 114,
    "EER": 0.030534351145038167,
    "c_EER": 74,
    "TAR_DEV": 0.8206106870229007,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.17938931297709926,
    "Dsys_DEV": 0.20806702038931474
   },
   {
    "modality": "face",
    "M": 256,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 83,
    "EER": 0.026717557251908396,
    "c_EER": 50,
    "TAR_DEV": 0.8854961832061069,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.1145038167938931,
    "Dsys_DEV": 0.14811459252582995
   },
   {
    "modality": "face",
    "M": 256,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 74,
    "EER": 0.022900763358778626,
    "c_EER": 36,
    "TAR_DEV": 0.851145038167939,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.14885496183206104,
    "Dsys_DEV": 0.20015901835864353
   },
   {
    "modality": "face",
    "M": 32,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 20,
    "EER": 0.04920937840785169,
    "c_EER": 15,
    "TAR_DEV": 0.6564885496183206,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.3435114503816794,
    "Dsys_DEV": 0.09780117657602731
   },
   {
    "modality": "face",
    "M": 32,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 16,
    "EER": 0.04187587595760025,
    "c_EER": 10,
    "TAR_DEV": 0.6259541984732825,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.3740458015267175,
    "Dsys_DEV": 0.10221185837547689
   },
   {
    "modality": "face",
    "M": 32,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 13,
    "EER": 0.045059372349448686,
    "c_EER": 7,
    "TAR_DEV": 0.683206106870229,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.316793893129771,
    "Dsys_DEV": 0.05974609641130809
   },
   {
    "modality": "face",
    "M": 32,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 11,
    "EER": 0.044321405566900104,
    "c_EER": 6,
    "TAR_DEV": 0.6908396946564885,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.30916030534351147,
    "Dsys_DEV": 0.06611963231470917
   },
   {
    "modality": "face",
    "M": 64,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 37,
    "EER": 0.03730852838582478,
    "c_EER": 28,
    "TAR_DEV": 0.7748091603053435,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.22519083969465647,
    "Dsys_DEV": 0.17901307902545452
   },
   {
    "modality": "face",
    "M": 64,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 29,
    "EER": 0.036910131852879946,
    "c_EER": 19,
    "TAR_DEV": 0.8053435114503816,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.19465648854961837,
    "Dsys_DEV": 0.11764352973204015
   },
   {
    "modality": "face",
    "M": 64,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 28,
    "EER": 0.03769288733086525,
    "c_EER": 13,
    "TAR_DEV": 0.5725190839694656,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.4274809160305344,
    "Dsys_DEV": 0.07171903852746761
   },
   {
    "modality": "face",
    "M": 64,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 22,
    "EER": 0.03285052521693743,
    "c_EER": 10,
    "TAR_DEV": 0.683206106870229,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.316793893129771,
    "Dsys_DEV": 0.08554435849953275
   },
   {
    "modality": "face",
    "M": 128,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 70,
    "EER": 0.030534351145038167,
    "c_EER": 55,
    "TAR_DEV": 0.8549618320610687,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.14503816793893132,
    "Dsys_DEV": 0.16282784161162364
   },
   {
    "modality": "face",
    "M": 128,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 57,
    "EER": 0.03134958867560957,
    "c_EER": 36,
    "TAR_DEV": 0.8129770992366412,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.1870229007633588,
    "Dsys_DEV": 0.23406035377757584
   },
   {
    "modality": "face",
    "M": 128,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 47,
    "EER": 0.026717557251908396,
    "c_EER": 26,
    "TAR_DEV": 0.7824427480916031,
    "FMR_DEV": 0.0005585552038726494,
    "FNMR_DEV": 0.21755725190839692,
    "Dsys_DEV": 0.13617423308164708
   },
   {
    "modality": "face",
    "M": 128,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 39,
    "EER": 0.022900763358778626,
    "c_EER": 19,
    "TAR_DEV": 0.7480916030534351,
    "FMR_DEV": 0.0005585552038726494,
    "FNMR_DEV": 0.25190839694656486,
    "Dsys_DEV": 0.13837527261210308
   },
   {
    "modality": "face",
    "M": 256,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 137,
    "EER": 0.026717557251908396,
    "c_EER": 108,
    "TAR_DEV": 0.8854961832061069,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.1145038167938931,
    "Dsys_DEV": 0.2373319943442713
   },
   {
    "modality": "face",
    "M": 256,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 102,
    "EER": 0.026717557251908396,
    "c_EER": 72,
    "TAR_DEV": 0.9007633587786259,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.0992366412213741,
    "Dsys_DEV": 0.16595589691614307
   },
   {
    "modality": "face",
    "M": 256,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 81,
    "EER": 0.023685036076545016,
    "c_EER": 50,
    "TAR_DEV": 0.8740458015267175,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.12595419847328249,
    "Dsys_DEV": 0.26656572288160546
   },
   {
    "modality": "face",
    "M": 256,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 71,
    "EER": 0.026717557251908396,
    "c_EER": 34,
    "TAR_DEV": 0.8587786259541985,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.14122137404580148,
    "Dsys_DEV": 0.17163419940309982
   },
   {
    "modality": "face",
    "M": 32,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 22,
    "EER": 0.04911349416390032,
    "c_EER": 15,
    "TAR_DEV": 0.5,
    "FMR_DEV": 0.0004654626698938745,
    "FNMR_DEV": 0.5,
    "Dsys_DEV": 0.16863716564243514
   },
   {
    "modality": "face",
    "M": 32,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 18,
    "EER": 0.04507856854281401,
    "c_EER": 10,
    "TAR_DEV": 0.5419847328244275,
    "FMR_DEV": 0.0004654626698938745,
    "FNMR_DEV": 0.4580152671755725,
    "Dsys_DEV": 0.19520244093994354
   },
   {
    "modality": "face",
    "M": 32,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 14,
    "EER": 0.04643436301071866,
    "c_EER": 7,
    "TAR_DEV": 0.6946564885496184,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.30534351145038163,
    "Dsys_DEV": 0.10288224819244313
   },
   {
    "modality": "face",
    "M": 32,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 11,
    "EER": 0.05052934941825154,
    "c_EER": 5,
    "TAR_DEV": 0.7175572519083969,
    "FMR_DEV": 0.0004654626698938745,
    "FNMR_DEV": 0.2824427480916031,
    "Dsys_DEV": 0.06182875586945358
   },
   {
    "modality": "face",
    "M": 64,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 40,
    "EER": 0.032597482979162366,
    "c_EER": 29,
    "TAR_DEV": 0.6297709923664122,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.3702290076335878,
    "Dsys_DEV": 0.1594444333005757
   },
   {
    "modality": "face",
    "M": 64,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 28,
    "EER": 0.0380385560874628,
    "c_EER": 19,
    "TAR_DEV": 0.8244274809160306,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.17557251908396942,
    "Dsys_DEV": 0.15841787071340263
   },
   {
    "modality": "face",
    "M": 64,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 24,
    "EER": 0.03835453774385072,
    "c_EER": 14,
    "TAR_DEV": 0.8129770992366412,
    "FMR_DEV": 0.0006516477378514243,
    "FNMR_DEV": 0.1870229007633588,
    "Dsys_DEV": 0.18428144421378856
   },
   {
    "modality": "face",
    "M": 64,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 19,
    "EER": 0.04014053271704368,
    "c_EER": 9,
    "TAR_DEV": 0.816793893129771,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.18320610687022898,
    "Dsys_DEV": 0.04379217311647991
   },
   {
    "modality": "face",
    "M": 128,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 73,
    "EER": 0.03279262086513995,
    "c_EER": 54,
    "TAR_DEV": 0.8053435114503816,
    "FMR_DEV": 0.0005585552038726494,
    "FNMR_DEV": 0.19465648854961837,
    "Dsys_DEV": 0.21203692976407515
   },
   {
    "modality": "face",
    "M": 128,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 58,
    "EER": 0.030534351145038167,
    "c_EER": 36,
    "TAR_DEV": 0.7900763358778626,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.20992366412213737,
    "Dsys_DEV": 0.10726632255459154
   },
   {
    "modality": "face",
    "M": 128,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 45,
    "EER": 0.030534351145038167,
    "c_EER": 25,
    "TAR_DEV": 0.8702290076335878,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.1297709923664122,
    "Dsys_DEV": 0.12646932112243758
   },
   {
    "modality": "face",
    "M": 128,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 35,
    "EER": 0.030423182390869337,
    "c_EER": 18,
    "TAR_DEV": 0.8473282442748091,
    "FMR_DEV": 0.0007447402718301992,
    "FNMR_DEV": 0.15267175572519087,
    "Dsys_DEV": 0.1174773052161679
   },
   {
    "modality": "face",
    "M": 256,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 136,
    "EER": 0.022900763358778626,
    "c_EER": 106,
    "TAR_DEV": 0.8893129770992366,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.11068702290076338,
    "Dsys_DEV": 0.2304838882997572
   },
   {
    "modality": "face",
    "M": 256,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 109,
    "EER": 0.024928435114503815,
    "c_EER": 72,
    "TAR_DEV": 0.8396946564885496,
    "FMR_DEV": 0.0008378328058089741,
    "FNMR_DEV": 0.16030534351145043,
    "Dsys_DEV": 0.22523200246533187
   },
   {
    "modality": "face",
    "M": 256,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 86,
    "EER": 0.030534351145038167,
    "c_EER": 47,
    "TAR_DEV": 0.851145038167939,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.14885496183206104,
    "Dsys_DEV": 0.10090390670877723
   },
   {
    "modality": "face",
    "M": 256,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 262,
    "n_impostor": 10742,
    "c_tau_DEV": 66,
    "EER": 0.026717557251908396,
    "c_EER": 34,
    "TAR_DEV": 0.8778625954198473,
    "FMR_DEV": 0.000930925339787749,
    "FNMR_DEV": 0.12213740458015265,
    "Dsys_DEV": 0.1567503264677415
   }
  ],
  "voice": [
   {
    "modality": "voice",
    "M": 32,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 24,
    "EER": 0.05448197773164748,
    "c_EER": 18,
    "TAR_DEV": 0.5119047619047619,
    "FMR_DEV": 0.0004645760743321719,
    "FNMR_DEV": 0.48809523809523814,
    "Dsys_DEV": 0.1373904284117684
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 19,
    "EER": 0.05591956080677885,
    "c_EER": 13,
    "TAR_DEV": 0.6857142857142857,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.3142857142857143,
    "Dsys_DEV": 0.0640405166033787
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 17,
    "EER": 0.049215179813232666,
    "c_EER": 11,
    "TAR_DEV": 0.6571428571428571,
    "FMR_DEV": 0.0005807200929152149,
    "FNMR_DEV": 0.34285714285714286,
    "Dsys_DEV": 0.08997233301922923
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 14,
    "EER": 0.0373015873015873,
    "c_EER": 8,
    "TAR_DEV": 0.7309523809523809,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.2690476190476191,
    "Dsys_DEV": 0.13945570610220737
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 43,
    "EER": 0.031040950296967802,
    "c_EER": 34,
    "TAR_DEV": 0.7738095238095238,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.22619047619047616,
    "Dsys_DEV": 0.1963994301746319
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 36,
    "EER": 0.03661646994980328,
    "c_EER": 25,
    "TAR_DEV": 0.7785714285714286,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.22142857142857142,
    "Dsys_DEV": 0.15524712934158585
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 30,
    "EER": 0.028350970017636683,
    "c_EER": 20,
    "TAR_DEV": 0.7904761904761904,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.20952380952380956,
    "Dsys_DEV": 0.12625343045813123
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 25,
    "EER": 0.023496744542320952,
    "c_EER": 15,
    "TAR_DEV": 0.8238095238095238,
    "FMR_DEV": 0.0006968641114982578,
    "FNMR_DEV": 0.17619047619047623,
    "Dsys_DEV": 0.06843931602491743
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 83,
    "EER": 0.026077097505668934,
    "c_EER": 66,
    "TAR_DEV": 0.8119047619047619,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.1880952380952381,
    "Dsys_DEV": 0.21376048229644495
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 68,
    "EER": 0.02464576074332172,
    "c_EER": 50,
    "TAR_DEV": 0.8238095238095238,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.17619047619047623,
    "Dsys_DEV": 0.21943236775585742
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 58,
    "EER": 0.01904761904761905,
    "c_EER": 40,
    "TAR_DEV": 0.830952380952381,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.169047619047619,
    "Dsys_DEV": 0.10973871526287383
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 47,
    "EER": 0.01780855199222546,
    "c_EER": 30,
    "TAR_DEV": 0.861904761904762,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.13809523809523805,
    "Dsys_DEV": 0.11665916907471875
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 4,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 161,
    "EER": 0.019581280788177342,
    "c_EER": 133,
    "TAR_DEV": 0.8761904761904762,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.12380952380952381,
    "Dsys_DEV": 0.2514963186506918
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 8,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 131,
    "EER": 0.02142857142857143,
    "c_EER": 98,
    "TAR_DEV": 0.8714285714285714,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.12857142857142856,
    "Dsys_DEV": 0.23204931835616294
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 16,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 104,
    "EER": 0.018444444444444444,
    "c_EER": 75,
    "TAR_DEV": 0.9071428571428571,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.09285714285714286,
    "Dsys_DEV": 0.14136672977311954
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 32,
    "o": 0,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 87,
    "EER": 0.01828087167070218,
    "c_EER": 58,
    "TAR_DEV": 0.9,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.09999999999999998,
    "Dsys_DEV": 0.1427005860270003
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 23,
    "EER": 0.045738295318127245,
    "c_EER": 17,
    "TAR_DEV": 0.5833333333333334,
    "FMR_DEV": 0.0004645760743321719,
    "FNMR_DEV": 0.41666666666666663,
    "Dsys_DEV": 0.07025304258243507
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 18,
    "EER": 0.0388748419721871,
    "c_EER": 13,
    "TAR_DEV": 0.7023809523809523,
    "FMR_DEV": 0.0005226480836236934,
    "FNMR_DEV": 0.29761904761904767,
    "Dsys_DEV": 0.15418264662745504
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 14,
    "EER": 0.024836309523809524,
    "c_EER": 10,
    "TAR_DEV": 0.8047619047619048,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.1952380952380952,
    "Dsys_DEV": 0.05169940801170041
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 12,
    "EER": 0.02653249144961852,
    "c_EER": 7,
    "TAR_DEV": 0.8380952380952381,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.16190476190476188,
    "Dsys_DEV": 0.12871981653851353
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 39,
    "EER": 0.024660685530250748,
    "c_EER": 32,
    "TAR_DEV": 0.8666666666666667,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.1333333333333333,
    "Dsys_DEV": 0.14050909780498969
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 31,
    "EER": 0.01816101816101816,
    "c_EER": 25,
    "TAR_DEV": 0.861904761904762,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.13809523809523805,
    "Dsys_DEV": 0.10187554236068369
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 25,
    "EER": 0.0166256157635468,
    "c_EER": 18,
    "TAR_DEV": 0.8904761904761904,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.10952380952380958,
    "Dsys_DEV": 0.13369635540689295
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 20,
    "EER": 0.017233560090702947,
    "c_EER": 14,
    "TAR_DEV": 0.9214285714285714,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.07857142857142863,
    "Dsys_DEV": 0.06965322304609416
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 73,
    "EER": 0.012306046013911182,
    "c_EER": 64,
    "TAR_DEV": 0.9238095238095239,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.07619047619047614,
    "Dsys_DEV": 0.12066641374873054
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 57,
    "EER": 0.010793650793650793,
    "c_EER": 48,
    "TAR_DEV": 0.9333333333333333,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.06666666666666665,
    "Dsys_DEV": 0.1552325666741784
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 45,
    "EER": 0.00923141186299081,
    "c_EER": 36,
    "TAR_DEV": 0.9452380952380952,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.05476190476190479,
    "Dsys_DEV": 0.13260755303971636
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 37,
    "EER": 0.009523809523809525,
    "c_EER": 29,
    "TAR_DEV": 0.9523809523809523,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.04761904761904767,
    "Dsys_DEV": 0.05251857190074913
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 4,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 141,
    "EER": 0.009523809523809525,
    "c_EER": 126,
    "TAR_DEV": 0.9595238095238096,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.04047619047619044,
    "Dsys_DEV": 0.20916199784989814
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 8,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 109,
    "EER": 0.005900621118012423,
    "c_EER": 95,
    "TAR_DEV": 0.9523809523809523,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.04761904761904767,
    "Dsys_DEV": 0.2815447577175441
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 16,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 84,
    "EER": 0.008771929824561403,
    "c_EER": 70,
    "TAR_DEV": 0.9666666666666667,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.033333333333333326,
    "Dsys_DEV": 0.19540367623761273
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 32,
    "o": 1,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 67,
    "EER": 0.007142857142857143,
    "c_EER": 55,
    "TAR_DEV": 0.9761904761904762,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.023809523809523836,
    "Dsys_DEV": 0.1072645528611956
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 22,
    "EER": 0.03935333710763037,
    "c_EER": 17,
    "TAR_DEV": 0.6833333333333333,
    "FMR_DEV": 0.0005226480836236934,
    "FNMR_DEV": 0.31666666666666665,
    "Dsys_DEV": 0.19137662750657985
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 18,
    "EER": 0.02142857142857143,
    "c_EER": 13,
    "TAR_DEV": 0.7047619047619048,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.2952380952380952,
    "Dsys_DEV": 0.11697129320289643
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 14,
    "EER": 0.024180581323438467,
    "c_EER": 9,
    "TAR_DEV": 0.7952380952380952,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.2047619047619048,
    "Dsys_DEV": 0.08589554325033043
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 12,
    "EER": 0.02550220088035214,
    "c_EER": 7,
    "TAR_DEV": 0.8023809523809524,
    "FMR_DEV": 0.0003484320557491289,
    "FNMR_DEV": 0.19761904761904758,
    "Dsys_DEV": 0.04291588079710068
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 39,
    "EER": 0.02073689811119979,
    "c_EER": 33,
    "TAR_DEV": 0.9119047619047619,
    "FMR_DEV": 0.0005807200929152149,
    "FNMR_DEV": 0.08809523809523812,
    "Dsys_DEV": 0.16788376267374933
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 33,
    "EER": 0.016666666666666666,
    "c_EER": 24,
    "TAR_DEV": 0.861904761904762,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.13809523809523805,
    "Dsys_DEV": 0.15877368917529974
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 26,
    "EER": 0.013978127136021872,
    "c_EER": 19,
    "TAR_DEV": 0.9119047619047619,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.08809523809523812,
    "Dsys_DEV": 0.07903051132063302
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 21,
    "EER": 0.01659495123350545,
    "c_EER": 14,
    "TAR_DEV": 0.9095238095238095,
    "FMR_DEV": 0.0006968641114982578,
    "FNMR_DEV": 0.09047619047619049,
    "Dsys_DEV": 0.08860046456747067
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 74,
    "EER": 0.009767529058867642,
    "c_EER": 66,
    "TAR_DEV": 0.9476190476190476,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.05238095238095242,
    "Dsys_DEV": 0.23935471361013594
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 58,
    "EER": 0.009523809523809525,
    "c_EER": 48,
    "TAR_DEV": 0.9547619047619048,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.04523809523809519,
    "Dsys_DEV": 0.16724965406845438
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 45,
    "EER": 0.007738095238095238,
    "c_EER": 37,
    "TAR_DEV": 0.9619047619047619,
    "FMR_DEV": 0.0006968641114982578,
    "FNMR_DEV": 0.03809523809523807,
    "Dsys_DEV": 0.10363864637019643
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 38,
    "EER": 0.013338453661034306,
    "c_EER": 27,
    "TAR_DEV": 0.9452380952380952,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.05476190476190479,
    "Dsys_DEV": 0.16782691581905987
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 4,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 141,
    "EER": 0.007142857142857143,
    "c_EER": 130,
    "TAR_DEV": 0.9714285714285714,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.02857142857142858,
    "Dsys_DEV": 0.2486553491235278
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 8,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 109,
    "EER": 0.006152758132956153,
    "c_EER": 96,
    "TAR_DEV": 0.9761904761904762,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.023809523809523836,
    "Dsys_DEV": 0.19674794809393453
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 16,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 82,
    "EER": 0.006596409055425449,
    "c_EER": 72,
    "TAR_DEV": 0.9761904761904762,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.023809523809523836,
    "Dsys_DEV": 0.17342733864136076
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 32,
    "o": 2,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 69,
    "EER": 0.009221466364323508,
    "c_EER": 52,
    "TAR_DEV": 0.9714285714285714,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.02857142857142858,
    "Dsys_DEV": 0.14967565675300937
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 21,
    "EER": 0.04314477468839885,
    "c_EER": 17,
    "TAR_DEV": 0.7476190476190476,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.2523809523809524,
    "Dsys_DEV": 0.11435547974855352
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 16,
    "EER": 0.030549913488302112,
    "c_EER": 12,
    "TAR_DEV": 0.8666666666666667,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.1333333333333333,
    "Dsys_DEV": 0.11760400514589812
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 13,
    "EER": 0.02579854987262395,
    "c_EER": 9,
    "TAR_DEV": 0.8714285714285714,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.12857142857142856,
    "Dsys_DEV": 0.044617301675914836
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 10,
    "EER": 0.020441788882853522,
    "c_EER": 7,
    "TAR_DEV": 0.9119047619047619,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.08809523809523812,
    "Dsys_DEV": 0.023292435719620194
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 39,
    "EER": 0.022778202189966894,
    "c_EER": 33,
    "TAR_DEV": 0.9,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.09999999999999998,
    "Dsys_DEV": 0.15701156241432127
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 29,
    "EER": 0.015319548872180452,
    "c_EER": 24,
    "TAR_DEV": 0.9380952380952381,
    "FMR_DEV": 0.0005226480836236934,
    "FNMR_DEV": 0.06190476190476191,
    "Dsys_DEV": 0.12183605948605142
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 22,
    "EER": 0.01698497580850522,
    "c_EER": 17,
    "TAR_DEV": 0.9452380952380952,
    "FMR_DEV": 0.0005807200929152149,
    "FNMR_DEV": 0.05476190476190479,
    "Dsys_DEV": 0.12803635159332355
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 18,
    "EER": 0.010099424385138672,
    "c_EER": 14,
    "TAR_DEV": 0.95,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.050000000000000044,
    "Dsys_DEV": 0.1235808174018865
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 71,
    "EER": 0.009523809523809525,
    "c_EER": 64,
    "TAR_DEV": 0.9595238095238096,
    "FMR_DEV": 0.0005807200929152149,
    "FNMR_DEV": 0.04047619047619044,
    "Dsys_DEV": 0.2101571019452547
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 52,
    "EER": 0.007581453634085213,
    "c_EER": 46,
    "TAR_DEV": 0.9857142857142858,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.014285714285714235,
    "Dsys_DEV": 0.14002248765724484
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 39,
    "EER": 0.004761904761904762,
    "c_EER": 34,
    "TAR_DEV": 0.9833333333333333,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.01666666666666672,
    "Dsys_DEV": 0.16002306695181068
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 32,
    "EER": 0.005334538878842676,
    "c_EER": 27,
    "TAR_DEV": 0.9833333333333333,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.01666666666666672,
    "Dsys_DEV": 0.1405909600814617
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 4,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 136,
    "EER": 0.004761904761904762,
    "c_EER": 128,
    "TAR_DEV": 0.9833333333333333,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.01666666666666672,
    "Dsys_DEV": 0.22997716336987084
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 8,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 98,
    "EER": 0.004617604617604618,
    "c_EER": 90,
    "TAR_DEV": 0.9880952380952381,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.011904761904761862,
    "Dsys_DEV": 0.17174619349214937
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 16,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 74,
    "EER": 0.004761904761904762,
    "c_EER": 66,
    "TAR_DEV": 0.9952380952380953,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.004761904761904745,
    "Dsys_DEV": 0.2054338444741759
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 32,
    "o": 3,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 58,
    "EER": 0.004679802955665025,
    "c_EER": 51,
    "TAR_DEV": 0.9904761904761905,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.00952380952380949,
    "Dsys_DEV": 0.13335782438269744
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 21,
    "EER": 0.04151305683563748,
    "c_EER": 17,
    "TAR_DEV": 0.7428571428571429,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.2571428571428571,
    "Dsys_DEV": 0.09016376940316503
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 17,
    "EER": 0.0210985384252711,
    "c_EER": 13,
    "TAR_DEV": 0.861904761904762,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.13809523809523805,
    "Dsys_DEV": 0.09160691442398511
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 13,
    "EER": 0.016627793974732748,
    "c_EER": 10,
    "TAR_DEV": 0.9238095238095239,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.07619047619047614,
    "Dsys_DEV": 0.10852962662132434
   },
   {
    "modality": "voice",
    "M": 32,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 11,
    "EER": 0.01755031909671085,
    "c_EER": 7,
    "TAR_DEV": 0.9,
    "FMR_DEV": 0.0005807200929152149,
    "FNMR_DEV": 0.09999999999999998,
    "Dsys_DEV": 0.03943338343948356
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 37,
    "EER": 0.021440566150893604,
    "c_EER": 32,
    "TAR_DEV": 0.930952380952381,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.06904761904761902,
    "Dsys_DEV": 0.13666333598261166
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 28,
    "EER": 0.008351934523809525,
    "c_EER": 25,
    "TAR_DEV": 0.9761904761904762,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.023809523809523836,
    "Dsys_DEV": 0.09753930383121248
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 24,
    "EER": 0.007200229489386116,
    "c_EER": 19,
    "TAR_DEV": 0.9595238095238096,
    "FMR_DEV": 0.0006387921022067364,
    "FNMR_DEV": 0.04047619047619044,
    "Dsys_DEV": 0.12240691793043788
   },
   {
    "modality": "voice",
    "M": 64,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 18,
    "EER": 0.00796423372842587,
    "c_EER": 14,
    "TAR_DEV": 0.9642857142857143,
    "FMR_DEV": 0.0004065040650406504,
    "FNMR_DEV": 0.0357142857142857,
    "Dsys_DEV": 0.12168129558811466
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 70,
    "EER": 0.007142857142857143,
    "c_EER": 63,
    "TAR_DEV": 0.9833333333333333,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.01666666666666672,
    "Dsys_DEV": 0.23277459828779373
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 51,
    "EER": 0.004071773636991028,
    "c_EER": 47,
    "TAR_DEV": 0.9904761904761905,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.00952380952380949,
    "Dsys_DEV": 0.2026180225528297
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 41,
    "EER": 0.004761904761904762,
    "c_EER": 35,
    "TAR_DEV": 0.9857142857142858,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.014285714285714235,
    "Dsys_DEV": 0.16259028682562998
   },
   {
    "modality": "voice",
    "M": 128,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 32,
    "EER": 0.004761904761904762,
    "c_EER": 26,
    "TAR_DEV": 0.9904761904761905,
    "FMR_DEV": 0.0007549361207897793,
    "FNMR_DEV": 0.00952380952380949,
    "Dsys_DEV": 0.07247059844423989
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 4,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 132,
    "EER": 0.004761904761904762,
    "c_EER": 123,
    "TAR_DEV": 0.9928571428571429,
    "FMR_DEV": 0.0009872241579558653,
    "FNMR_DEV": 0.0071428571428571175,
    "Dsys_DEV": 0.35392523120682995
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 8,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 95,
    "EER": 0.0033041788143828963,
    "c_EER": 87,
    "TAR_DEV": 0.9952380952380953,
    "FMR_DEV": 0.0008710801393728223,
    "FNMR_DEV": 0.004761904761904745,
    "Dsys_DEV": 0.18980481758145684
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 16,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 71,
    "EER": 0.0031593406593406594,
    "c_EER": 66,
    "TAR_DEV": 0.9952380952380953,
    "FMR_DEV": 0.0008130081300813008,
    "FNMR_DEV": 0.004761904761904745,
    "Dsys_DEV": 0.17719968410273246
   },
   {
    "modality": "voice",
    "M": 256,
    "q": 32,
    "o": 4,
    "n_subjects": 42,
    "n_genuine": 420,
    "n_impostor": 17220,
    "c_tau_DEV": 55,
    "EER": 0.002380952380952381,
    "c_EER": 49,
    "TAR_DEV": 0.9952380952380953,
    "FMR_DEV": 0.0009291521486643438,
    "FNMR_DEV": 0.004761904761904745,
    "Dsys_DEV": 0.1347769554259671
   }
  ]
 },
 "stages": {
  "lfw_raw": {
   "archive_sha256": "055f7d9c632d7370e6fb4afc7468d40f970c34a80d4c6f50ffec63f5a8d536c0",
   "completed_unix": 1789359031.2497926,
   "protocol_version": "1.1.1",
   "raw_images": 13233,
   "stage": "lfw_raw"
  },
  "lfw_embeddings": {
   "completed_unix": 1788770241.5083568,
   "embeddings_sha256": "6da5f90d0513173c7ce447c5415f96526175a032b982ef9d03b35d275083deb0",
   "identities_with_at_least_3_valid": 900,
   "protocol_version": "1.1.1",
   "stage": "lfw_embeddings",
   "valid_embeddings": 7581,
   "validity_sha256": "b424e215f3a2b928873fb50f1fed621d1964fa580106523b492e5315a3c1cea4"
  },
  "libri_local": {
   "archive_md5": "2a93770f6d5c6c964bc36631d331a522",
   "completed_unix": 1789359573.272247,
   "protocol_version": "1.1.1",
   "stage": "libri_local"
  },
  "libri_internal": {
   "background": 50,
   "completed_unix": 1788771438.5137756,
   "development": 42,
   "evaluation": 58,
   "manifest_sha256": "e44365f70ac69aa583a71e25e67dd8b3446e7abba03d14df63da33e52ad742a4",
   "protocol_version": "1.1.1",
   "selected_speakers": 150,
   "stage": "libri_internal"
  },
  "libri_embeddings": {
   "completed_unix": 1788781600.0447152,
   "embeddings": 2250,
   "protocol_version": "1.1.1",
   "sha256": "65d96727c00c8fac0a487cd8e2a7ffa59b20942baebc1a7ada5b6d1f4d9d4751",
   "stage": "libri_embeddings"
  },
  "vctk_manifest": {
   "archive_bytes": 11747302977,
   "completed_unix": 1789654999.6629865,
   "corpus": "VCTK-Corpus-0.92",
   "manifest_sha256": "3025b1d448e318ccde93c3becab392c15f075b128c260bcd3dbc4c61742a9705",
   "mic": "mic1",
   "n_speakers": 110,
   "n_speakers_skipped": 0,
   "n_utterances": 1650,
   "protocol_version": "1.1.1",
   "resampler": "scipy.signal.resample_poly",
   "selection": "first N valid in sorted filename order",
   "stage": "vctk_manifest",
   "target_sample_rate": 16000,
   "utterances_per_speaker": 15,
   "validity_rule": "mic1_post_resample_samples>=24000"
  },
  "vctk_embeddings": {
   "checkpoint_interval": 32,
   "completed_unix": 1789657559.2200065,
   "manifest_sha256": "3025b1d448e318ccde93c3becab392c15f075b128c260bcd3dbc4c61742a9705",
   "n": 1650,
   "protocol_version": "1.1.1",
   "sha256": "5fd9c3b6e71282bbbe890e2c7dfb01b4de00c7e4acac5ec156995709e495007a",
   "stage": "vctk_embeddings"
  }
 }
}


In [ ]:
#@title 12. Load the runtime
config_path = DIR["seal"] / "config_v1_1_4.json"
if not config_path.is_file():
    config_path.write_text(json.dumps(
        json.loads((RUNTIME / "paper_reference.json").read_text())["config"], indent=2))
for name in ("ablation_only.py", "scoredump_only.py", "inversion_only.py",
             "revocation_only.py", "allfigures_only.py", "vctk_prepare_cells.py",
             "fresh_pipeline_only.py"):
    exec(compile((RUNTIME / name).read_text(), str(RUNTIME / name), "exec"), globals())


def _module(name):
    spec = importlib.util.spec_from_file_location(name, RUNTIME / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


results_plots = _module("results_plots")
figures_only = _module("figures_only")
print("\nWhere this run stands:")
status()


In [ ]:
#@title 13. Stage 0 - libraries and environment record
install_dependencies()
display(environment_record())


In [ ]:
#@title 14. Stage 1 - LFW: download, MTCNN, FaceNet embeddings
stage_face()


In [ ]:
#@title 15. Stage 2 - CFP exclusion, the 50/42/58 face split and trials
stage_face_split()


In [ ]:
#@title 16. Stage 3 - LibriSpeech: download, 150 speakers, ECAPA embeddings
stage_voice()


In [ ]:
#@title 17. Stage 4 - frozen IoM tensors
stage_tensors()


In [ ]:
#@title 18. Stage 5 - Stage A key search (GPU; may span sessions)
KEYS_READY = stage_a("voice") and stage_a("face")    # face waits for voice
print("keys ready" if KEYS_READY else "key search continues in the next session")


In [ ]:
#@title 19. Stage 6 - Stage B sweep, selection, held-out evaluation
INTERNAL_READY = False
if KEYS_READY:
    for m in ("voice", "face"):
        stage_b(m)
    for m in ("voice", "face"):
        stage_heldout(m)
    INTERNAL_READY = True
else:
    print("waiting for the key search")


In [ ]:
#@title 20. Stage 7 - external corpus VCTK (read only now)
EXTERNAL_READY = False
if INTERNAL_READY:
    stage_external()
    EXTERNAL_READY = True
else:
    print("waiting for the internal evaluation")


In [ ]:
#@title 21. Stage 8 - score dump, intervals, ablation, attack, revocation
ANALYSES_READY = False
if EXTERNAL_READY:
    SCORES = add_external_scores(run_score_dump())
    INTERVALS = run_intervals(N_BOOTSTRAP, CONFIDENCE)
    BASELINE = baseline_from_scores(SCORES)
    ABLATION = run_ablation(N_BOOTSTRAP, CONFIDENCE)
    INVERSION = run_inversion(N_BOOTSTRAP, CONFIDENCE)
    REVOCATION = run_revocation(N_BOOTSTRAP, CONFIDENCE)
    ANALYSES_READY = True
else:
    print("waiting for the external evaluation")


In [ ]:
#@title 22. Stage 9 - every figure, the comparison with the paper, and a zip
if ANALYSES_READY:
    SWEEPS, SELECTED, FLOORS = {}, {}, {}
    for m in ("voice", "face"):
        SWEEPS[m] = pd.read_csv(RUNS_IN / "sweep" / m / "sweep_80.csv").to_dict("records")
        op = json.loads((DIR["seal"] / f"operating_point_{m}.json").read_text())
        SELECTED[m] = next(r for r in SWEEPS[m]
                           if (r["M"], r["q"], r["o"]) == (op["M"], op["q"], op["o"]))
        FLOORS[m] = (op["rule"]["eer_max"], op["rule"]["tar_min"])
    figures = PROJECT / "figures"
    draw_figures(figures, SWEEPS, SELECTED, FLOORS, INTERVALS, BASELINE,
                 ABLATION, INVERSION, REVOCATION, RUNS_IN / "scores" / "score_histograms.json")
    rows, claims = compare_with_paper(INTERVALS, ABLATION, INVERSION, REVOCATION)
    write_report(rows, claims)
    bundle = RAW / "bundle"
    shutil.rmtree(bundle, ignore_errors=True)
    shutil.copytree(figures, bundle / "figures")
    shutil.copytree(DIR["seal"], bundle / "seal")
    for f in RUNS_IN.glob("**/*.json"):
        (bundle / "runs" / f.relative_to(RUNS_IN)).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, bundle / "runs" / f.relative_to(RUNS_IN))
    for name in ("REPLICATION_REPORT.md", "replication_report.json"):
        shutil.copy(PROJECT / name, bundle / name)
    print("zip:", shutil.make_archive(str(PROJECT / "AttackAware_PolyIoM_fresh_results"),
                                      "zip", bundle))
else:
    print("not finished yet; Save & Run All again in the next session")
    status()
